# PhotoGraphiQ manuscript experiment suite (R1-R42)

One executable notebook combining every script in `paper_experiments/` (`common.py`, `metadata.py`, and experiments R1-R42, in order). Each experiment has a markdown section explaining the scientific/algorithmic claim it tests, the equations it uses, and what success looks like, followed by one mathematically-commented code cell that runs the experiment, validates it against an independent reference, and produces one publication-style figure.

**Runtime**: running the whole notebook top-to-bottom executes every experiment, including the large-Fock and long-benchmark ones excluded from `run_all_safe.py` (see `README.md`). Wall-clock time is dominated by the cutoff=24-96 non-Gaussian/GKP scripts (R18, R20-R23, R26-R28, R33-R34) and the multi-mode Gaussian/compiler benchmarks (R10, R36-R38); it varies substantially by machine (the Fock backend in particular is CPU-bound), so treat any specific minute estimate as this suite's own observation on the machine it was built on, not a portable claim.

**Isolation note**: `run_all_safe.py` runs each script in its own subprocess so failures/warnings can't leak between experiments; this notebook instead runs everything in one kernel for a single linear narrative. Shared plotting/style helpers (`set_reference_style`, `style_axis`) and utilities (`common`, `metadata`) are defined once in the setup cell below and reused by every later experiment; each experiment cell still defines its own scientific constants and any experiment-specific plotting logic, so redefining a same-named helper in a later cell cannot affect an earlier cell's already-completed run.

**Optional dependencies**: R17 (Graphix) and R41-R42 (JAX) check `common.HAS_GRAPHIX`/`common.HAS_JAX` and print `SKIPPED` (doing nothing further) if their optional package is missing, rather than stopping the notebook. The install cell below installs both, so on a fresh environment they run for real.


## Install dependencies

In [1]:
# %cd /content

# !rm -rf PhotoGraphiQ

!git clone \
    --branch manuscript-experiments \
    --single-branch \
    https://github.com/chinmoybiswasdeep/PhotoGraphiQ.git

%cd /content/PhotoGraphiQ

%pip install -e ".[dev,autodiff]"

Cloning into 'PhotoGraphiQ'...
remote: Enumerating objects: 1193, done.
remote: Counting objects: 100% (1193/1193), done.
remote: Compressing objects: 100% (793/793), done.
remote: Total 1193 (delta 449), reused 1108 (delta 368), pack-reused 0 (from 0)
Receiving objects: 100% (1193/1193), 12.34 MiB | 22.40 MiB/s, done.
Resolving deltas: 100% (449/449), done.
/content/PhotoGraphiQ
Obtaining file:///content/PhotoGraphiQ
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 kB 1.5 MB/s eta 0:00:00a 0:00:01
  Preparing metadata (setup.py) ... done
  Using cached pathspec-1.1.1-py3-none-any.whl.metadata (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 8.1 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.

In [2]:
!ls /content/PhotoGraphiQ/paper_experiments

01_core_mbqc		10_convergence	     MANUSCRIPT_MAP.md
02_compiler		11_performance	     metadata.py
03_adaptive_flow	12_serialization     README.md
04_piquasso_validation	13_autodiff	     requirements_notes.md
05_graphix_validation	common.py	     results
06_backend_crosschecks	EXPERIMENT_INDEX.md  run_all_safe.py
07_non_gaussian		figures		     STATUS.md
08_gkp			generate_tables.py   tables
09_noise		ISSUES_FOUND.md


## Setup: make `common` and `metadata` importable

In [ ]:
import sys
import importlib
from pathlib import Path

# Force the exact directory
PAPER_EXPERIMENTS_DIR = Path(
    "/content/PhotoGraphiQ/paper_experiments"
).resolve()

print("Directory:", PAPER_EXPERIMENTS_DIR)
print("Directory exists:", PAPER_EXPERIMENTS_DIR.exists())
print("common.py exists:", (PAPER_EXPERIMENTS_DIR / "common.py").exists())
print("metadata.py exists:", (PAPER_EXPERIMENTS_DIR / "metadata.py").exists())

# Force Python to search this directory FIRST
sys.path.insert(0, str(PAPER_EXPERIMENTS_DIR))

# Clear Python's import cache
importlib.invalidate_caches()

print("\nFirst sys.path entry:")
print(sys.path[0])

# Now import
import common
import metadata

print("\nSUCCESS")
print("common loaded from:", common.__file__)
print("metadata loaded from:", metadata.__file__)
print("HAS_GRAPHIX:", common.HAS_GRAPHIX)
print("HAS_JAX:", common.HAS_JAX)

# ---------------------------------------------------------------------
# Shared reference figure style (R1/R2 style, reused by every later
# experiment so it is not redefined 40 times). Individual experiments may
# still add their own small plotting helpers where the figure layout is
# genuinely experiment-specific.
# ---------------------------------------------------------------------

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np


def set_reference_style():
    mpl.rcParams.update({
        # White paper background
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",

        # Times-like serif font available in Colab
        "font.family": "serif",
        "font.serif": ["Liberation Serif", "DejaVu Serif"],

        "font.size": 9,
        "axes.labelsize": 9,
        "axes.titlesize": 10,
        "legend.fontsize": 7.5,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,

        # Axes
        "axes.edgecolor": "black",
        "axes.linewidth": 0.8,

        # Lines
        "lines.linewidth": 1.5,
        "lines.markersize": 5,

        # Ticks
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.top": True,
        "ytick.right": True,
        "xtick.major.size": 4,
        "ytick.major.size": 4,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,

        # Export
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "figure.dpi": 150,
        "savefig.dpi": 300,
    })


def style_axis(ax, xlabel, ylabel, title, panel_label):
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title, pad=6)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)

    ax.tick_params(direction="in", top=True, right=True)
    ax.grid(False)

    ax.text(
        -0.12, 1.05,
        panel_label,
        transform=ax.transAxes,
        fontsize=11,
        fontweight="bold",
        va="top",
    )


def add_vline_reference(ax, x, **kwargs):
    """Thin black dashed reference line (e.g. a special parameter value)."""
    style = {"color": "black", "linestyle": "--", "linewidth": 0.8}
    style.update(kwargs)
    ax.axvline(x, **style)


print("\nShared reference style helpers ready: "
      "set_reference_style(), style_axis(), add_vline_reference()")


## R1 — One-Step CV Wire / Teleportation

R1 validates the elementary continuous-variable MBQC teleportation step implemented by a one-step wire.

For a shear parameter \(k\), the ideal one-step symplectic map is

$$
S(k)=
\begin{pmatrix}
-k & -1 \\
1 & 0
\end{pmatrix}.
$$

Therefore, for input quadratures

$$
\mu_{\text{in}}
=
\begin{pmatrix}
q_{\text{in}} \\
p_{\text{in}}
\end{pmatrix},
$$

the output mean is

$$
\mu_{\text{out}}
=
S(k)\mu_{\text{in}}+d.
$$

With \(d=0\),

$$
q_{\text{out}}
=
-kq_{\text{in}}-p_{\text{in}},
$$

$$
p_{\text{out}}
=
q_{\text{in}}.
$$

For the special case \(k=0\),

$$
S(0)=
\begin{pmatrix}
0 & -1 \\
1 & 0
\end{pmatrix},
$$

so

$$
q_{\text{out}}=-p_{\text{in}},
\qquad
p_{\text{out}}=q_{\text{in}},
$$

which is the Fourier-transform teleportation step.

For a coherent input state \(\alpha\), using the \(\hbar=2\) convention,

$$
\mu_{\text{in}}
=
\begin{pmatrix}
2\operatorname{Re}(\alpha) \\
2\operatorname{Im}(\alpha)
\end{pmatrix},
$$

and

$$
V_{\text{in}}=I.
$$

The Gaussian channel transforms the state moments as

$$
\mu_{\text{out}}
=
S\mu_{\text{in}}+d,
$$

$$
V_{\text{out}}
=
SV_{\text{in}}S^T+N.
$$

Finite resource squeezing \(r\) adds Gaussian noise

$$
N=
\begin{pmatrix}
0 & 0 \\
0 & e^{-2r}
\end{pmatrix}.
$$

Hence finite squeezing does not change the deterministic symplectic map \(S\), but increases the output covariance.

For each coherent input, R1 compares four descriptions of the output:

### 1. Analytic channel

The exact unconditional Gaussian prediction is

$$
\mu_{\text{analytic}}
=
S\mu_{\text{in}}+d,
$$

$$
V_{\text{analytic}}
=
SV_{\text{in}}S^T+N.
$$

### 2. Gaussian Monte-Carlo ensemble

Many conditional trajectories are simulated using the Gaussian backend.

If trajectory \(j\) has mean \(\mu_j\) and covariance \(V_j\), then the ensemble mean is

$$
\bar{\mu}
=
\frac{1}{M}
\sum_{j=1}^{M}\mu_j.
$$

The total ensemble covariance follows the law of total covariance,

$$
V_{\text{ensemble}}
=
\frac{1}{M}
\sum_{j=1}^{M}V_j
+
\operatorname{Cov}(\mu_j).
$$

As the number of shots \(M\) increases,

$$
\bar{\mu}
\rightarrow
\mu_{\text{analytic}},
$$

and

$$
V_{\text{ensemble}}
\rightarrow
V_{\text{analytic}}.
$$

### 3. Piquasso ensemble

The same MBQC pattern is simulated using the Piquasso backend.

Agreement between

$$
\text{Analytic}
\approx
\text{Gaussian}
\approx
\text{Piquasso}
$$

provides an independent backend-level validation of the implementation.

### 4. Single trajectory

A single simulation gives one conditional output state corresponding to one random homodyne outcome.

Its mean and covariance need not exactly equal the unconditional analytic values.

The quadrature statistics are

$$
x_\theta
=
q\cos\theta+p\sin\theta,
$$

$$
\langle x_\theta\rangle
=
h^T\mu,
$$

$$
\operatorname{Var}(x_\theta)
=
h^TVh,
$$

where

$$
h=
\begin{pmatrix}
\cos\theta \\
\sin\theta
\end{pmatrix}.
$$

For the standard quadratures,

$$
\theta=0
\Rightarrow q,
$$

and

$$
\theta=\frac{\pi}{2}
\Rightarrow p.
$$

The numerical agreement is quantified using Frobenius errors,

$$
\epsilon_\mu
=
\|\mu_{\text{sim}}-\mu_{\text{analytic}}\|_F,
$$

$$
\epsilon_V
=
\|V_{\text{sim}}-V_{\text{analytic}}\|_F.
$$

### Physical interpretation

R1 verifies that a one-step CV-MBQC wire performs the expected teleportation transformation while correctly reproducing the finite-squeezing Gaussian noise.

The main validation target is

$$
\boxed{
\text{Analytic channel}
\approx
\text{Gaussian ensemble}
\approx
\text{Piquasso ensemble}
}
$$

while a single trajectory represents one conditional realization and is therefore expected to fluctuate around the ensemble result.

In [23]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

# ---------------------------------------------------------------------
# Setup
# ---------------------------------------------------------------------

ROOT = Path("/content/PhotoGraphiQ")
sys.path[:0] = [
    str(ROOT / "src"),
    str(ROOT / "paper_experiments"),
]

import common
import photographiq as pg

SHOTS = 500
R = 1.0
K = 0.5
SEED = 20260913
TOL = 0.35

ALPHAS = [
    0.0 + 0.0j,

    # Small amplitudes
    0.2 + 0.2j,
    -0.2 + 0.2j,
    0.2 - 0.2j,
    -0.2 - 0.2j,

    # Medium amplitudes
    0.5 + 0.0j,
    -0.5 + 0.0j,
    0.0 + 0.5j,
    0.0 - 0.5j,

    0.5 + 0.5j,
    -0.5 + 0.5j,
    0.5 - 0.5j,
    -0.5 - 0.5j,

    # Larger amplitudes
    1.0 + 0.0j,
    -1.0 + 0.0j,
    0.0 + 1.0j,
    0.0 - 1.0j,

    1.0 + 0.5j,
    -1.0 + 0.5j,
    1.0 - 0.5j,
    -1.0 - 0.5j,

    # A few asymmetric test points
    0.3 + 0.7j,
    -0.8 + 0.3j,
    0.6 - 0.9j,
    -0.7 - 0.4j,
]


# ---------------------------------------------------------------------
# Plot style
# ---------------------------------------------------------------------

def set_reference_style():
    mpl.rcParams.update({
        # White paper background
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",

        # Times-like serif font available in Colab
        "font.family": "serif",
        "font.serif": [
            "Liberation Serif",
            "DejaVu Serif",
        ],

        "font.size": 9,
        "axes.labelsize": 9,
        "axes.titlesize": 10,
        "legend.fontsize": 7.5,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,

        # Axes
        "axes.edgecolor": "black",
        "axes.linewidth": 0.8,

        # Lines
        "lines.linewidth": 1.5,
        "lines.markersize": 5,

        # Ticks
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.top": True,
        "ytick.right": True,
        "xtick.major.size": 4,
        "ytick.major.size": 4,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,

        # Export
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "figure.dpi": 150,
        "savefig.dpi": 300,
    })


def style_axis(
    ax,
    xlabel,
    ylabel,
    title,
    panel_label,
):
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title, pad=6)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)

    ax.tick_params(
        direction="in",
        top=True,
        right=True,
    )

    ax.grid(False)

    ax.text(
        -0.12,
        1.05,
        panel_label,
        transform=ax.transAxes,
        fontsize=11,
        fontweight="bold",
        va="top",
    )


# ---------------------------------------------------------------------
# Mathematics
# ---------------------------------------------------------------------

def coherent_moments(alpha):
    # hbar = 2:
    #
    # mu = [2 Re(alpha), 2 Im(alpha)]^T
    # V  = I
    return (
        np.array([
            2 * alpha.real,
            2 * alpha.imag,
        ]),
        np.eye(2),
    )


def wire_channel(k, r):
    # One-step CV teleportation:
    #
    #       [-k  -1]
    # S  =  [      ]
    #       [ 1   0]
    #
    # Finite-squeezing noise:
    #
    # N = diag(0, e^(-2r))
    #
    # d = 0
    S = np.array([
        [-k, -1.0],
        [1.0, 0.0],
    ])

    N = np.diag([
        0.0,
        np.exp(-2 * r),
    ])

    d = np.zeros(2)

    return S, N, d


def apply_channel(mu, V, S, N, d):
    # Gaussian affine channel:
    #
    # mu_out = S mu + d
    #
    # V_out = S V S^T + N
    return (
        S @ mu + d,
        S @ V @ S.T + N,
    )


def quadrature(mu, V, theta):
    # x_theta = q cos(theta) + p sin(theta)
    #
    # <x_theta> = h^T mu
    #
    # Var(x_theta) = h^T V h
    h = np.array([
        np.cos(theta),
        np.sin(theta),
    ])

    return (
        float(h @ mu),
        float(h @ V @ h),
    )


def frob(a, b):
    # Frobenius / Euclidean error:
    #
    # ||A-B||_F
    return float(
        np.linalg.norm(
            np.asarray(a) - np.asarray(b)
        )
    )


def ensemble_moments(shots):
    # For conditional trajectories j:
    #
    # mu_bar = mean(mu_j)
    #
    # V_ensemble
    # = mean(V_j) + Cov(mu_j)
    #
    # This is the law of total covariance.
    states = [
        trajectory.state
        for trajectory in shots.trajectories
    ]

    means = np.stack([
        state.mean
        for state in states
    ])

    covariances = np.stack([
        state.covariance
        for state in states
    ])

    mu = means.mean(axis=0)

    conditional_cov = (
        covariances.mean(axis=0)
    )

    centered = means - mu

    classical_cov = (
        centered.T @ centered
        / len(states)
    )

    return (
        mu,
        conditional_cov + classical_cov,
    )


# ---------------------------------------------------------------------
# Main experiment
# ---------------------------------------------------------------------

def main():
    # common.setup_style() may change matplotlib defaults,
    # so apply our final style afterwards.
    common.setup_style()
    set_reference_style()

    # -------------------------------------------------------------
    # One-step CV wire
    #
    # General transformation:
    #
    # q_out = -k q_in - p_in
    # p_out =  q_in
    #
    # For k=0:
    #
    # q_out = -p_in
    # p_out =  q_in
    #
    # which is the Fourier teleportation step.
    # -------------------------------------------------------------

    pattern = pg.protocols.wire(
        [K],
        squeezing=R,
    )

    output_node = pattern.outputs[0]

    # Pattern-level analytic channel
    channel = pg.gaussian_channel(pattern)

    # Independent hand-derived channel
    S, N, d = wire_channel(K, R)

    # -------------------------------------------------------------
    # Verify analytic channel
    # -------------------------------------------------------------

    np.testing.assert_allclose(
        channel.matrix,
        S,
        atol=1e-12,
    )

    np.testing.assert_allclose(
        channel.noise,
        N,
        atol=1e-12,
    )

    np.testing.assert_allclose(
        channel.displacement,
        d,
        atol=1e-12,
    )

    print("S =\n", S)
    print("\nN =\n", N)
    print("\nd =", d)
    print("\ne^(-2r) =", np.exp(-2 * R))

    rows = []

    # =============================================================
    # Sweep coherent inputs
    # =============================================================

    for i, alpha in enumerate(ALPHAS):

        # ---------------------------------------------------------
        # Coherent input
        #
        # mu_in =
        # [2 Re(alpha), 2 Im(alpha)]^T
        #
        # V_in = I
        # ---------------------------------------------------------

        inp = pg.GaussianInput.coherent(alpha)

        mu_in, V_in = coherent_moments(
            alpha
        )

        np.testing.assert_allclose(
            inp.mean,
            mu_in,
        )

        np.testing.assert_allclose(
            inp.covariance,
            V_in,
        )

        # ---------------------------------------------------------
        # Analytic output
        #
        # mu_out = S mu_in + d
        #
        # V_out = S V_in S^T + N
        # ---------------------------------------------------------

        mu_analytic, V_analytic = (
            apply_channel(
                mu_in,
                V_in,
                S,
                N,
                d,
            )
        )

        pg_analytic = channel.apply(
            inp.state(0)
        )

        np.testing.assert_allclose(
            pg_analytic.mean,
            mu_analytic,
            atol=1e-12,
        )

        np.testing.assert_allclose(
            pg_analytic.covariance,
            V_analytic,
            atol=1e-12,
        )

        # q: theta = 0
        q_mean, q_var = quadrature(
            mu_analytic,
            V_analytic,
            0.0,
        )

        # p: theta = pi/2
        p_mean, p_var = quadrature(
            mu_analytic,
            V_analytic,
            np.pi / 2,
        )

        # ---------------------------------------------------------
        # Monte-Carlo ensembles
        #
        # Repeat:
        #
        # prepare
        # -> entangle
        # -> homodyne
        # -> feed-forward
        #
        # for M = SHOTS trajectories.
        # ---------------------------------------------------------

        ensembles = {}

        for backend in (
            "gaussian",
            "piquasso",
        ):

            shots = pg.run_shots(
                pattern,
                SHOTS,
                backend=backend,
                inputs={0: inp},
                seed=SEED + 97 * i,
            )

            pkg_state = (
                shots.ensemble_state()
            )

            # Independently reconstruct ensemble.
            mu_mc, V_mc = (
                ensemble_moments(shots)
            )

            np.testing.assert_allclose(
                pkg_state.mean,
                mu_mc,
                atol=1e-12,
            )

            np.testing.assert_allclose(
                pkg_state.covariance,
                V_mc,
                atol=1e-12,
            )

            ensembles[backend] = pkg_state

        # ---------------------------------------------------------
        # One conditional trajectory
        #
        # This is one random homodyne realization,
        # not the unconditional ensemble.
        # ---------------------------------------------------------

        single = pg.simulate(
            pattern,
            backend="gaussian",
            inputs={0: inp},
            seed=SEED + i,
        )

        single_q_mean, single_q_var = (
            quadrature(
                single.state.mean,
                single.state.covariance,
                0.0,
            )
        )

        single_p_mean, single_p_var = (
            quadrature(
                single.state.mean,
                single.state.covariance,
                np.pi / 2,
            )
        )

        np.testing.assert_allclose(
            single.state.quadrature(
                output_node
            ),
            [
                single_q_mean,
                single_q_var,
            ],
        )

        # ---------------------------------------------------------
        # Store results
        # ---------------------------------------------------------

        row = {
            "alpha_real": alpha.real,
            "alpha_imag": alpha.imag,

            "input_q_mean":
                mu_in[0],

            "input_p_mean":
                mu_in[1],

            "analytic_q_mean":
                q_mean,

            "analytic_p_mean":
                p_mean,

            "analytic_q_var":
                q_var,

            "analytic_p_var":
                p_var,

            "single_q_mean":
                single_q_mean,

            "single_p_mean":
                single_p_mean,

            "single_q_var":
                single_q_var,

            "single_p_var":
                single_p_var,
        }

        for backend, state in ensembles.items():

            row.update({
                f"{backend}_q_mean":
                    state.mean[0],

                f"{backend}_p_mean":
                    state.mean[1],

                f"{backend}_q_var":
                    state.covariance[0, 0],

                f"{backend}_p_var":
                    state.covariance[1, 1],

                f"{backend}_mean_error":
                    frob(
                        state.mean,
                        mu_analytic,
                    ),

                f"{backend}_cov_error":
                    frob(
                        state.covariance,
                        V_analytic,
                    ),
            })

        rows.append(row)

    # -----------------------------------------------------------------
    # Validation
    # -----------------------------------------------------------------

    failures = [
        row
        for row in rows
        if (
            row["gaussian_mean_error"] > TOL
            or
            row["gaussian_cov_error"] > TOL
        )
    ]

    if failures:

        common.save_csv(
            failures,
            "R1_wire_teleportation_FAILURES",
        )

        raise AssertionError(
            f"{len(failures)} cases exceeded "
            f"tolerance {TOL}"
        )

    # -----------------------------------------------------------------
    # Save numerical results
    # -----------------------------------------------------------------

    common.save_result(
        rows,
        "R1_wire_teleportation",
        extra={
            "squeezing": R,
            "shear_k": K,
            "shots": SHOTS,
            "matrix": S.tolist(),
            "noise": N.tolist(),
            "displacement": d.tolist(),
        },
    )

    # =================================================================
    # Final 2 x 3 publication figure
    # =================================================================

    x = np.arange(len(rows))

    labels = [
        (
            f"{row['alpha_real']:.1f}"
            f"{row['alpha_imag']:+.1f}i"
        )
        for row in rows
    ]

    fig, axes = plt.subplots(
        2,
        3,
        figsize=(12.0, 6.8),
        facecolor="white",
    )

    (
        axA,
        axB,
        axC,
        axD,
        axE,
        axF,
    ) = axes.ravel()

    # -----------------------------------------------------------------
    # Helper: analytic + ensemble + single trajectory
    # -----------------------------------------------------------------

    def comparison_panel(
        ax,
        analytic_key,
        gaussian_key,
        piquasso_key,
        single_key,
        ylabel,
        title,
        letter,
    ):
        ax.plot(
            x,
            [r[analytic_key] for r in rows],
            "o-",
            color="red",
            label="Analytic",
        )

        ax.plot(
            x,
            [r[gaussian_key] for r in rows],
            "s--",
            color="blue",
            label="Gaussian",
        )

        ax.plot(
            x,
            [r[piquasso_key] for r in rows],
            "^:",
            color="magenta",
            label="Piquasso",
        )

        # Single conditional realization:
        # shown separately/faintly because it is
        # not an ensemble quantity.
        ax.plot(
            x,
            [r[single_key] for r in rows],
            "d-.",
            color="cyan",
            alpha=0.75,
            label="Single trajectory",
        )

        ax.set_xticks(x)

        ax.set_xticklabels(
            labels,
            rotation=35,
            ha="right",
        )

        style_axis(
            ax,
            xlabel=r"Input coherent state $\alpha$",
            ylabel=ylabel,
            title=title,
            panel_label=letter,
        )

    # -----------------------------------------------------------------
    # A. Output q mean
    # -----------------------------------------------------------------

    comparison_panel(
        axA,
        "analytic_q_mean",
        "gaussian_q_mean",
        "piquasso_q_mean",
        "single_q_mean",
        r"$\langle q_{\mathrm{out}}\rangle$",
        "Output q mean",
        "A",
    )

    # -----------------------------------------------------------------
    # B. Output p mean
    # -----------------------------------------------------------------

    comparison_panel(
        axB,
        "analytic_p_mean",
        "gaussian_p_mean",
        "piquasso_p_mean",
        "single_p_mean",
        r"$\langle p_{\mathrm{out}}\rangle$",
        "Output p mean",
        "B",
    )

    # -----------------------------------------------------------------
    # C. Output q variance
    # -----------------------------------------------------------------

    comparison_panel(
        axC,
        "analytic_q_var",
        "gaussian_q_var",
        "piquasso_q_var",
        "single_q_var",
        r"$\mathrm{Var}(q_{\mathrm{out}})$",
        "Output q variance",
        "C",
    )

    # -----------------------------------------------------------------
    # D. Output p variance
    # -----------------------------------------------------------------

    comparison_panel(
        axD,
        "analytic_p_var",
        "gaussian_p_var",
        "piquasso_p_var",
        "single_p_var",
        r"$\mathrm{Var}(p_{\mathrm{out}})$",
        "Output p variance",
        "D",
    )

    # -----------------------------------------------------------------
    # E. Output phase-space means
    # -----------------------------------------------------------------

    analytic_q = np.array([
        r["analytic_q_mean"]
        for r in rows
    ])

    analytic_p = np.array([
        r["analytic_p_mean"]
        for r in rows
    ])

    gaussian_q = np.array([
        r["gaussian_q_mean"]
        for r in rows
    ])

    gaussian_p = np.array([
        r["gaussian_p_mean"]
        for r in rows
    ])

    piquasso_q = np.array([
        r["piquasso_q_mean"]
        for r in rows
    ])

    piquasso_p = np.array([
        r["piquasso_p_mean"]
        for r in rows
    ])

    single_q = np.array([
        r["single_q_mean"]
        for r in rows
    ])

    single_p = np.array([
        r["single_p_mean"]
        for r in rows
    ])

    axE.plot(
        analytic_q,
        analytic_p,
        "o-",
        color="red",
        label="Analytic",
    )

    axE.plot(
        gaussian_q,
        gaussian_p,
        "s--",
        color="blue",
        label="Gaussian",
    )

    axE.plot(
        piquasso_q,
        piquasso_p,
        "^:",
        color="magenta",
        label="Piquasso",
    )

    axE.plot(
        single_q,
        single_p,
        "d-.",
        color="cyan",
        alpha=0.75,
        label="Single trajectory",
    )

    # Mark analytic points by input alpha.
    for i, label in enumerate(labels):
        axE.annotate(
            label,
            (
                analytic_q[i],
                analytic_p[i],
            ),
            xytext=(4, 4),
            textcoords="offset points",
            fontsize=7,
        )

    # q=0 and p=0 references.
    axE.axhline(
        0,
        color="black",
        linewidth=0.6,
        linestyle="--",
    )

    axE.axvline(
        0,
        color="black",
        linewidth=0.6,
        linestyle="--",
    )

    style_axis(
        axE,
        xlabel=r"$\langle q_{\mathrm{out}}\rangle$",
        ylabel=r"$\langle p_{\mathrm{out}}\rangle$",
        title="Output phase-space mean",
        panel_label="E",
    )

    axE.set_aspect(
        "equal",
        adjustable="datalim",
    )

    # -----------------------------------------------------------------
    # F. Ensemble errors
    # -----------------------------------------------------------------

    floor = 1e-16

    gaussian_mean_error = np.maximum(
        [
            r["gaussian_mean_error"]
            for r in rows
        ],
        floor,
    )

    piquasso_mean_error = np.maximum(
        [
            r["piquasso_mean_error"]
            for r in rows
        ],
        floor,
    )

    gaussian_cov_error = np.maximum(
        [
            r["gaussian_cov_error"]
            for r in rows
        ],
        floor,
    )

    piquasso_cov_error = np.maximum(
        [
            r["piquasso_cov_error"]
            for r in rows
        ],
        floor,
    )

    axF.semilogy(
        x,
        gaussian_mean_error,
        "o-",
        color="red",
        label=r"Gaussian $\epsilon_\mu$",
    )

    axF.semilogy(
        x,
        piquasso_mean_error,
        "s--",
        color="blue",
        label=r"Piquasso $\epsilon_\mu$",
    )

    axF.semilogy(
        x,
        gaussian_cov_error,
        "^:",
        color="magenta",
        label=r"Gaussian $\epsilon_V$",
    )

    axF.semilogy(
        x,
        piquasso_cov_error,
        "d-.",
        color="cyan",
        label=r"Piquasso $\epsilon_V$",
    )

    axF.axhline(
        TOL,
        color="black",
        linestyle="--",
        linewidth=0.8,
        label="Tolerance",
    )

    axF.set_xticks(x)

    axF.set_xticklabels(
        labels,
        rotation=35,
        ha="right",
    )

    style_axis(
        axF,
        xlabel=r"Input coherent state $\alpha$",
        ylabel="Frobenius error",
        title="Ensemble validation error",
        panel_label="F",
    )

    # -----------------------------------------------------------------
    # Legends
    # -----------------------------------------------------------------

    # Main comparison legend only once.
    axA.legend(
        frameon=False,
        loc="best",
        ncol=2,
    )

    # Phase-space legend.
    axE.legend(
        frameon=False,
        loc="best",
    )

    # Error legend.
    axF.legend(
        frameon=False,
        loc="best",
    )

    # -----------------------------------------------------------------
    # Figure title
    # -----------------------------------------------------------------

    fig.suptitle(
        "One-step CV wire: analytic and simulator validation",
        fontsize=11,
        fontweight="bold",
        y=0.985,
    )

    fig.text(
        0.5,
        0.95,
        (
            rf"$k={K},\quad r={R},\quad "
            rf"N=\mathrm{{diag}}(0,e^{{-2r}}),\quad "
            rf"M={SHOTS}$ shots"
        ),
        ha="center",
        fontsize=9,
    )

    fig.tight_layout(
        rect=[0, 0, 1, 0.925],
        h_pad=2.0,
        w_pad=1.8,
    )

    # -----------------------------------------------------------------
    # Save publication figure
    # -----------------------------------------------------------------

    common.save_figure(
        fig,
        "R1_wire_teleportation_reference_style",
    )

    fig.savefig(
        "R1_wire_teleportation_reference_style.pdf",
        bbox_inches="tight",
        facecolor="white",
    )

    fig.savefig(
        "R1_wire_teleportation_reference_style.png",
        bbox_inches="tight",
        dpi=300,
        facecolor="white",
    )

    plt.show()
    plt.close(fig)

    # -----------------------------------------------------------------
    # Summary
    # -----------------------------------------------------------------

    common.print_summary(
        "R1 wire teleportation",

        inputs=len(rows),

        max_gaussian_mean_error=max(
            row["gaussian_mean_error"]
            for row in rows
        ),

        max_gaussian_cov_error=max(
            row["gaussian_cov_error"]
            for row in rows
        ),

        max_piquasso_mean_error=max(
            row["piquasso_mean_error"]
            for row in rows
        ),

        max_piquasso_cov_error=max(
            row["piquasso_cov_error"]
            for row in rows
        ),
    )


if __name__ == "__main__":
    main()

S =
 [[-0.5 -1. ]
 [ 1.   0. ]]

N =
 [[0.         0.        ]
 [0.         0.13533528]]

d = [0. 0.]

e^(-2r) = 0.1353352832366127

=== R1 wire teleportation ===
  inputs: 25
  max_gaussian_mean_error: 0.035469723809760295
  max_gaussian_cov_error: 0.02376382609484925
  max_piquasso_mean_error: 0.035469723809759636
  max_piquasso_cov_error: 0.023763826094849328


## R2 — Identity Wire vs Finite Squeezing

R2 studies how a four-step continuous-variable MBQC wire approaches an ideal identity channel as the resource squeezing \(r\) increases.

For one \(k=0\) teleportation step,

$$
T(0)=
\begin{pmatrix}
0 & -1 \\
1 & 0
\end{pmatrix}.
$$

Because

$$
T(0)^4 = I,
$$

four consecutive Fourier-teleportation steps implement the ideal identity transformation,

$$
S = I.
$$

The Gaussian channel transforms the first and second moments as

$$
\mu_{\text{out}} = S\mu_{\text{in}} + d,
$$

$$
V_{\text{out}} = S V_{\text{in}} S^T + N.
$$

For the identity wire,

$$
S = I,
\qquad
d = 0.
$$

Finite squeezing introduces additive Gaussian noise. For each teleportation step,

$$
N_{j+1}
=
T_j N_j T_j^T
+
\begin{pmatrix}
0 & 0 \\
0 & e^{-2r}
\end{pmatrix}.
$$

After four \(k=0\) steps,

$$
\boxed{N = 2e^{-2r} I}
$$

so that

$$
N_{qq} = N_{pp} = 2e^{-2r},
$$

and

$$
\boxed{\operatorname{Tr}(N) = 4e^{-2r}}.
$$

For the coherent input

$$
\alpha = 0.4 + 0.3i,
$$

using the \(\hbar=2\) convention,

$$
\mu_{\text{in}}
=
\begin{pmatrix}
2\operatorname{Re}(\alpha) \\
2\operatorname{Im}(\alpha)
\end{pmatrix}
=
\begin{pmatrix}
0.8 \\
0.6
\end{pmatrix},
$$

with

$$
V_{\text{in}} = I.
$$

Since the logical transformation is identity,

$$
\mu_{\text{out}} = \mu_{\text{in}}.
$$

The output covariance is

$$
V_{\text{out}}
=
I + 2e^{-2r}I,
$$

therefore

$$
\boxed{
\operatorname{Var}(q_{\text{out}})
=
\operatorname{Var}(p_{\text{out}})
=
1 + 2e^{-2r}
}
$$

As \(r\) increases,

$$
e^{-2r} \rightarrow 0,
$$

so

$$
S \rightarrow I,
\qquad
N \rightarrow 0,
\qquad
V_{\text{out}} \rightarrow I.
$$

The numerical experiment checks the following errors:

$$
\|S-I\|_F,
$$

$$
\|S-S_{\text{closed}}\|_F,
$$

$$
\|N-N_{\text{closed}}\|_F,
$$

and

$$
\|\mu_{\text{out}}-\mu_{\text{in}}\|_2.
$$

The finite-squeezing noise follows

$$
\operatorname{Tr}(N)=4e^{-2r}.
$$

Taking the base-10 logarithm,

$$
\log_{10}\operatorname{Tr}(N)
=
\log_{10}(4)
-
\frac{2}{\ln(10)}r.
$$

Hence the theoretical slope is

$$
\boxed{
-\frac{2}{\ln(10)}
\approx -0.8686
}
$$

### Physical interpretation

Increasing \(r\) does not change the ideal logical operation. The four-step wire remains an identity transformation. Increasing \(r\) only suppresses the finite-squeezing Gaussian noise exponentially as

$$
e^{-2r}.
$$

In [20]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

# ---------------------------------------------------------------------
# Setup
# ---------------------------------------------------------------------

ROOT = Path("/content/PhotoGraphiQ")
sys.path[:0] = [
    str(ROOT / "src"),
    str(ROOT / "paper_experiments"),
]

import common
import photographiq as pg
from photographiq.gaussian import wire_channel

SQUEEZING_VALUES = [
    0.2, 0.3, 0.4, 0.5, 0.6, 0.7,
    0.8, 0.9, 1.0, 1.5, 2.0, 3.0, 5.0,
]

INPUT_ALPHA = 0.4 + 0.3j
SEED = 7
TOL = 1e-9


# ---------------------------------------------------------------------
# Plot style
# ---------------------------------------------------------------------

def set_reference_style():
    mpl.rcParams.update({
        # White background
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "savefig.facecolor": "white",

        # Times-like serif font available in Colab/Linux
        "font.family": "serif",
        "font.serif": ["Liberation Serif", "DejaVu Serif"],

        "font.size": 9,
        "axes.labelsize": 9,
        "axes.titlesize": 10,
        "legend.fontsize": 8,
        "xtick.labelsize": 8,
        "ytick.labelsize": 8,

        # Axes
        "axes.edgecolor": "black",
        "axes.linewidth": 0.8,

        # Lines
        "lines.linewidth": 1.5,
        "lines.markersize": 5,

        # Ticks
        "xtick.direction": "in",
        "ytick.direction": "in",
        "xtick.top": True,
        "ytick.right": True,
        "xtick.major.size": 4,
        "ytick.major.size": 4,
        "xtick.major.width": 0.8,
        "ytick.major.width": 0.8,

        # Export
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "figure.dpi": 150,
        "savefig.dpi": 300,
    })


def style_axis(ax, xlabel, ylabel, title, panel_label):
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title, pad=6)

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(0.8)

    ax.tick_params(direction="in", top=True, right=True)
    ax.grid(False)

    ax.text(
        -0.11, 1.05,
        panel_label,
        transform=ax.transAxes,
        fontsize=11,
        fontweight="bold",
        va="top",
    )


def add_r_reference(ax):
    ax.axvline(
        1.0,
        color="black",
        linestyle="--",
        linewidth=0.8,
    )


# ---------------------------------------------------------------------
# Main experiment
# ---------------------------------------------------------------------

def main():
    plt = common.setup_style()
    set_reference_style()

    # Coherent input, hbar = 2:
    #
    # mu_in = [2 Re(alpha), 2 Im(alpha)]^T
    # V_in  = I
    input_state = pg.GaussianInput.coherent(INPUT_ALPHA)

    mu_in = np.array([
        2 * INPUT_ALPHA.real,
        2 * INPUT_ALPHA.imag,
    ])

    rows = []

    for r in SQUEEZING_VALUES:

        # -------------------------------------------------------------
        # Four-step identity wire
        #
        # One k=0 step:
        #
        #       [ 0  -1 ]
        # T  =  [        ]
        #       [ 1   0  ]
        #
        # T^4 = I
        # -------------------------------------------------------------

        pattern = pg.protocols.identity(squeezing=r)
        output_node = pattern.outputs[0]

        # Gaussian channel:
        #
        # mu_out = S mu_in + d
        # V_out  = S V_in S^T + N
        channel = pg.gaussian_channel(pattern)

        # Closed-form oracle:
        #
        # N_(j+1) = T N_j T^T + diag(0, e^(-2r))
        #
        # Four steps:
        #
        # S = I
        # N = 2 e^(-2r) I
        closed_S, closed_N = wire_channel(
            [0.0] * 4,
            r,
        )

        applied = channel.apply(
            input_state.state(0)
        )

        # -------------------------------------------------------------
        # Validation errors
        # -------------------------------------------------------------

        identity_matrix_error = common.frobenius_error(
            channel.matrix,
            np.eye(2),
        )

        oracle_matrix_error = common.frobenius_error(
            channel.matrix,
            closed_S,
        )

        oracle_noise_error = common.frobenius_error(
            channel.noise,
            closed_N,
        )

        # Identity should preserve mean:
        #
        # mu_out = mu_in
        mean_error = common.frobenius_error(
            applied.mean,
            mu_in,
        )

        # -------------------------------------------------------------
        # Finite-squeezing noise
        #
        # Nqq = Npp = 2 e^(-2r)
        #
        # Tr(N) = 4 e^(-2r)
        # -------------------------------------------------------------

        noise_q = float(channel.noise[0, 0])
        noise_p = float(channel.noise[1, 1])
        noise_trace = float(np.trace(channel.noise))

        theory_noise_each = 2 * np.exp(-2 * r)
        theory_noise_trace = 4 * np.exp(-2 * r)

        # -------------------------------------------------------------
        # Output covariance
        #
        # V_in = I
        #
        # V_out = I + 2 e^(-2r) I
        #
        # Var(q) = Var(p) = 1 + 2 e^(-2r)
        # -------------------------------------------------------------

        analytic_q_var = float(
            applied.covariance[0, 0]
        )

        analytic_p_var = float(
            applied.covariance[1, 1]
        )

        theory_output_var = (
            1 + 2 * np.exp(-2 * r)
        )

        # -------------------------------------------------------------
        # One conditional trajectory
        # -------------------------------------------------------------

        single = pg.simulate(
            pattern,
            backend="gaussian",
            inputs={0: input_state},
            seed=SEED,
        )

        _, single_q_var = single.state.quadrature(
            output_node
        )

        _, single_p_var = single.state.quadrature(
            output_node,
            np.pi / 2,
        )

        # -------------------------------------------------------------
        # Store results
        # -------------------------------------------------------------

        rows.append({
            "squeezing_r": r,

            "identity_matrix_error":
                identity_matrix_error,

            "channel_vs_closed_form_matrix_error":
                oracle_matrix_error,

            "channel_vs_closed_form_noise_error":
                oracle_noise_error,

            "output_mean_error":
                mean_error,

            "noise_q":
                noise_q,

            "noise_p":
                noise_p,

            "added_variance_trace":
                noise_trace,

            "predicted_noise_each":
                theory_noise_each,

            "predicted_noise_trace":
                theory_noise_trace,

            "analytic_q_variance":
                analytic_q_var,

            "analytic_p_variance":
                analytic_p_var,

            "predicted_output_variance":
                theory_output_var,

            "single_q_variance":
                single_q_var,

            "single_p_variance":
                single_p_var,

            "log10_added_variance":
                common.safe_log10(noise_trace),
        })

    # -----------------------------------------------------------------
    # Numerical validation
    # -----------------------------------------------------------------

    failures = [
        row
        for row in rows
        if (
            row["identity_matrix_error"] > TOL
            or
            row["channel_vs_closed_form_matrix_error"] > TOL
            or
            row["channel_vs_closed_form_noise_error"] > TOL
        )
    ]

    if failures:
        common.save_csv(
            failures,
            "R2_identity_finite_squeezing_FAILURES",
        )

        raise AssertionError(
            "Identity wire disagreed with closed-form theory."
        )

    # -----------------------------------------------------------------
    # Save numerical results
    # -----------------------------------------------------------------

    common.save_result(
        rows,
        "R2_identity_finite_squeezing",
        extra={
            "input_alpha": [
                INPUT_ALPHA.real,
                INPUT_ALPHA.imag,
            ],
            "protocol":
                "four-step identity wire",

            "theory":
                "S=I, N=2 exp(-2r) I, Tr(N)=4 exp(-2r)",
        },
    )

    # =================================================================
    # 2 x 2 publication figure
    # =================================================================

    rs = np.array([
        row["squeezing_r"]
        for row in rows
    ])

    fig, axes = plt.subplots(
        2,
        2,
        figsize=(9.2, 6.8),
        facecolor="white",
    )

    axA, axB, axC, axD = axes.ravel()

    # -----------------------------------------------------------------
    # A. Added finite-squeezing noise
    # -----------------------------------------------------------------

    axA.semilogy(
        rs,
        [row["noise_q"] for row in rows],
        "o-",
        color="red",
        label=r"$N_{qq}$",
    )

    axA.semilogy(
        rs,
        [row["noise_p"] for row in rows],
        "s--",
        color="blue",
        label=r"$N_{pp}$",
    )

    axA.semilogy(
        rs,
        [row["added_variance_trace"] for row in rows],
        "^:",
        color="magenta",
        label=r"$\mathrm{Tr}(N)$",
    )

    axA.semilogy(
        rs,
        [row["predicted_noise_each"] for row in rows],
        "-.",
        color="cyan",
        label=r"$2e^{-2r}$",
    )

    axA.semilogy(
        rs,
        [row["predicted_noise_trace"] for row in rows],
        "--",
        color="black",
        label=r"$4e^{-2r}$",
    )

    style_axis(
        axA,
        xlabel="Resource squeezing $r$",
        ylabel="Added covariance noise",
        title="Finite-squeezing noise",
        panel_label="A",
    )

    add_r_reference(axA)

    axA.legend(
        frameon=False,
        loc="best",
        ncol=2,
    )

    # -----------------------------------------------------------------
    # B. Output quadrature variances
    # -----------------------------------------------------------------

    axB.plot(
        rs,
        [row["analytic_q_variance"] for row in rows],
        "o-",
        color="red",
        label=r"Analytic $\mathrm{Var}(q)$",
    )

    axB.plot(
        rs,
        [row["analytic_p_variance"] for row in rows],
        "s--",
        color="blue",
        label=r"Analytic $\mathrm{Var}(p)$",
    )

    axB.plot(
        rs,
        [row["single_q_variance"] for row in rows],
        "^:",
        color="magenta",
        label=r"Single-shot $\mathrm{Var}(q)$",
    )

    axB.plot(
        rs,
        [row["single_p_variance"] for row in rows],
        "d-.",
        color="cyan",
        label=r"Single-shot $\mathrm{Var}(p)$",
    )

    axB.plot(
        rs,
        [row["predicted_output_variance"] for row in rows],
        "--",
        color="black",
        label=r"$1+2e^{-2r}$",
    )

    axB.axhline(
        1.0,
        color="black",
        linestyle=":",
        linewidth=0.8,
    )

    style_axis(
        axB,
        xlabel="Resource squeezing $r$",
        ylabel="Output variance",
        title="Output quadrature variance",
        panel_label="B",
    )

    add_r_reference(axB)

    axB.legend(
        frameon=False,
        loc="best",
        ncol=2,
    )

    # -----------------------------------------------------------------
    # C. Numerical validation
    # -----------------------------------------------------------------

    floor = 1e-16

    def positive(key):
        return np.maximum(
            [row[key] for row in rows],
            floor,
        )

    axC.semilogy(
        rs,
        positive("identity_matrix_error"),
        "o-",
        color="red",
        label=r"$\|S-I\|_F$",
    )

    axC.semilogy(
        rs,
        positive("channel_vs_closed_form_matrix_error"),
        "s--",
        color="blue",
        label=r"$\|S-S_{\rm closed}\|_F$",
    )

    axC.semilogy(
        rs,
        positive("channel_vs_closed_form_noise_error"),
        "^:",
        color="magenta",
        label=r"$\|N-N_{\rm closed}\|_F$",
    )

    axC.semilogy(
        rs,
        positive("output_mean_error"),
        "d-.",
        color="cyan",
        label=r"$\|\mu_{\rm out}-\mu_{\rm in}\|_2$",
    )

    axC.axhline(
        TOL,
        color="black",
        linestyle="--",
        linewidth=0.8,
        label="Tolerance",
    )

    style_axis(
        axC,
        xlabel="Resource squeezing $r$",
        ylabel="Numerical error",
        title="Channel validation",
        panel_label="C",
    )

    add_r_reference(axC)

    axC.legend(
        frameon=False,
        loc="best",
    )

    # -----------------------------------------------------------------
    # D. Exponential scaling law
    #
    # Tr(N) = 4e^(-2r)
    #
    # log10 Tr(N)
    # = log10(4) - (2/ln 10) r
    #
    # slope = -2/ln(10)
    # -----------------------------------------------------------------

    log_noise = np.array([
        row["log10_added_variance"]
        for row in rows
    ])

    theory_log = (
        np.log10(4)
        - 2 * rs / np.log(10)
    )

    axD.plot(
        rs,
        log_noise,
        "o-",
        color="red",
        label="Channel",
    )

    axD.plot(
        rs,
        theory_log,
        "--",
        color="blue",
        label=r"$\log_{10}(4e^{-2r})$",
    )

    slope, intercept = np.polyfit(
        rs,
        log_noise,
        1,
    )

    style_axis(
        axD,
        xlabel="Resource squeezing $r$",
        ylabel=r"$\log_{10}[\mathrm{Tr}(N)]$",
        title="Exponential noise suppression",
        panel_label="D",
    )

    add_r_reference(axD)

    axD.legend(
        frameon=False,
        loc="best",
    )

    axD.text(
        0.96,
        0.08,
        (
            rf"fit slope = {slope:.4f}"
            "\n"
            rf"theory = {-2 / np.log(10):.4f}"
        ),
        transform=axD.transAxes,
        ha="right",
        va="bottom",
        fontsize=8,
    )

    # -----------------------------------------------------------------
    # Figure title
    # -----------------------------------------------------------------

    fig.suptitle(
        "Four-step CV identity wire under finite squeezing",
        fontsize=11,
        fontweight="bold",
        y=0.98,
    )

    fig.text(
        0.5,
        0.945,
        (
            r"$S=I,\quad "
            r"N=2e^{-2r}I,\quad "
            r"V_{\mathrm{out}}=V_{\mathrm{in}}+N$"
        ),
        ha="center",
        fontsize=9,
    )

    fig.tight_layout(
        rect=[0, 0, 1, 0.92],
        h_pad=2.0,
        w_pad=1.8,
    )

    # -----------------------------------------------------------------
    # Save figure
    # -----------------------------------------------------------------

    common.save_figure(
        fig,
        "R2_identity_finite_squeezing_reference_style",
    )

    fig.savefig(
        "R2_identity_finite_squeezing_reference_style.pdf",
        bbox_inches="tight",
        facecolor="white",
    )

    fig.savefig(
        "R2_identity_finite_squeezing_reference_style.png",
        bbox_inches="tight",
        dpi=300,
        facecolor="white",
    )

    plt.close(fig)

    # -----------------------------------------------------------------
    # Console summary
    # -----------------------------------------------------------------

    common.print_summary(
        "R2 identity vs finite squeezing",

        squeezing_values=SQUEEZING_VALUES,

        max_identity_matrix_error=max(
            row["identity_matrix_error"]
            for row in rows
        ),

        max_closed_form_matrix_error=max(
            row["channel_vs_closed_form_matrix_error"]
            for row in rows
        ),

        max_closed_form_noise_error=max(
            row["channel_vs_closed_form_noise_error"]
            for row in rows
        ),

        max_output_mean_error=max(
            row["output_mean_error"]
            for row in rows
        ),

        max_noise_trace=max(
            row["added_variance_trace"]
            for row in rows
        ),

        min_noise_trace=min(
            row["added_variance_trace"]
            for row in rows
        ),
    )


if __name__ == "__main__":
    main()


=== R2 identity vs finite squeezing ===
  squeezing_values: [0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0, 1.5, 2.0, 3.0, 5.0]
  max_identity_matrix_error: 1.7319121124709866e-16
  max_closed_form_matrix_error: 1.7319121124709866e-16
  max_closed_form_noise_error: 0.0
  max_output_mean_error: 1.5700924586837752e-16
  max_noise_trace: 2.6812801841425573
  min_noise_trace: 0.00018159971904993942


## R3 — Rotation Gate Compilation

R3 checks that `Circuit(1).rotate(0, theta).compile(squeezing=r)` — a single logical
rotation lowered to a chain of one-step teleportations (R1) — reproduces the ideal
rotation map for **every** angle and resource squeezing, and explains where the
finite-squeezing noise floor actually comes from.

### From a symplectic target to a shear sequence

`Circuit.gaussian`/`.rotate`/`.squeeze` all go through `decompose_symplectic(S)`
(`src/photographiq/compiler.py`), which writes a real single-mode symplectic
\(S=\begin{pmatrix}a&b\\c&d\end{pmatrix}\) (with \(\det S=1\)) as a product of
one-step teleportations \(T(k)=\begin{pmatrix}-k&-1\\1&0\end{pmatrix}\) (R1),

$$
S = T(k_4)\,T(k_3)\,T(k_2)\,T(k_1),
\qquad
k_1=0,\;\;
k_2=\frac{1-d}{c},\;\;
k_3=c,\;\;
k_4=\frac{1-a}{c},
$$

(a leading \(k_0=0\) step is prepended and \(S\) pre-rotated by \(T(0)\) whenever
\(|c|\) is too small to divide by). Each step is one physical teleportation with
resource squeezing \(r\), so finite \(r\) adds Gaussian noise through the **same**
recursion as R1/R2,

$$
N_{j+1} = T(k_j)\,N_j\,T(k_j)^{T} + \operatorname{diag}(0,e^{-2r}),
\qquad N_0 = 0.
$$

For a rotation \(R(\theta)=\begin{pmatrix}\cos\theta&-\sin\theta\\\sin\theta&\cos\theta\end{pmatrix}\)
(\(a=d=\cos\theta\), \(c=\sin\theta\)), the half-angle identity
\((1-\cos\theta)/\sin\theta=\tan(\theta/2)\) gives the closed-form shear sequence

$$
k_1=0,\qquad k_2=k_4=\tan(\theta/2),\qquad k_3=\sin\theta.
$$

**This is the key physical point of R3**: the finite-squeezing noise floor of a
compiled gate is *not* simply \(e^{-2r}\) — it is \(e^{-2r}\) *amplified by the
synthesis shears* \(k_j(\theta)\) through the recursion above. Since
\(\tan(\theta/2)\to\infty\) as \(\theta\to\pm\pi\), gates close to a half-turn need
disproportionately large shears and therefore have a much larger noise floor at
fixed \(r\) than gates close to the identity — a **synthesis/gauge effect**, not a
squeezing limitation, and not a truncation artifact (the ideal map itself stays
exact for any \(r\)).

### What is checked

For a sweep of \(\theta\in[-\pi,\pi]\) at fixed \(r\), and a sweep of \(r\) at fixed
\(\theta\), R3 compares:

1. **Compiled map** \(S_{\text{pattern}}=\) `gaussian_channel(pattern).matrix` against
   the independent target \(R(\theta)\) (Frobenius error) — this should be exact
   (machine precision) for *every* \(r\), since \(r\) only enters the noise, not the
   deterministic map.
2. **Compiled noise** \(N_{\text{pattern}}\) against the closed-form
   \(N_{\text{oracle}}\) obtained by feeding the *actual* decomposition
   \(k_j=\)`decompose_symplectic(R(\theta))` through the independent recursion above
   (`photographiq.gaussian.wire_channel`, the same oracle validated in R1/R2) — an
   independent code path from the compiler's own noise bookkeeping.
3. The noise **spectral norm** \(\|N\|_2\) and **trace** \(\operatorname{Tr}(N)\)
   against \(4e^{-2r}\) (the identity-wire floor, R2) to quantify the synthesis
   amplification directly.

### Success criteria

- Map error \(\lesssim 10^{-9}\) for all \(\theta\), independent of \(r\) (panel A, D).
- Noise-oracle error \(\lesssim 10^{-9}\): the compiler's noise bookkeeping matches
  the closed-form multi-step recursion exactly (panel B).
- \(\|N\|_2\) tracks \(|\tan(\theta/2)|\)-driven amplification of the \(4e^{-2r}\)
  identity-wire floor, growing sharply as \(\theta\to\pm\pi\) (panel B).
- \(\|N\|_2\) decays like \(e^{-2r}\) at fixed \(\theta\) (panel C), confirming finite
  squeezing — not truncation — is the only noise source.


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg
from photographiq.compiler import decompose_symplectic
from photographiq.gaussian import wire_channel

set_reference_style()

THETAS = np.linspace(-np.pi, np.pi, 25)
SQUEEZING_SWEEP = [0.3, 0.5, 0.8, 1.2, 1.6, 2.0, 2.5]
FIXED_THETA_FOR_SQUEEZING_SWEEP = 0.7
FIXED_SQUEEZING_FOR_THETA_SWEEP = 1.2

MAP_TOL = 1e-9          # deterministic algebraic identity -> tight tolerance
NOISE_ORACLE_TOL = 1e-9  # exact recursion, not Monte Carlo -> tight tolerance


def independent_rotation_matrix(theta: float) -> np.ndarray:
    """Independent NumPy rotation matrix; not photographiq.gaussian.rotation."""
    c, s = float(np.cos(theta)), float(np.sin(theta))
    return np.array([[c, -s], [s, c]])


def evaluate(theta: float, r: float) -> dict:
    target = independent_rotation_matrix(theta)

    circuit = pg.Circuit(1).rotate(0, float(theta))
    pattern = circuit.compile(squeezing=r)
    channel = pg.gaussian_channel(pattern)

    # Independent noise oracle: feed the *actual* teleportation shears
    # k_j = decompose_symplectic(R(theta)) through the same multi-step
    # recursion validated in R1/R2 (a separate code path from the
    # compiler's own noise bookkeeping):
    #
    # N_(j+1) = T(k_j) N_j T(k_j)^T + diag(0, e^(-2r))
    ks = decompose_symplectic(target)
    oracle_S, oracle_N = wire_channel(ks, r)

    return {
        "theta": float(theta),
        "squeezing": r,
        "map_frobenius_error": common.frobenius_error(channel.matrix, target),
        "oracle_map_error": common.frobenius_error(oracle_S, target),
        "noise_oracle_error": common.frobenius_error(channel.noise, oracle_N),
        "noise_spectral_norm": float(np.linalg.norm(channel.noise, ord=2)),
        "noise_trace": float(np.trace(channel.noise)),
        "n_shear_steps": len(ks),
        "max_abs_shear": float(np.max(np.abs(ks))),
        "commands": len(pattern.commands),
    }


# =============================================================================
# Sweep 1: rotation angle theta at fixed resource squeezing
# =============================================================================

theta_rows = [evaluate(theta, FIXED_SQUEEZING_FOR_THETA_SWEEP) for theta in THETAS]

# =============================================================================
# Sweep 2: resource squeezing r at fixed rotation angle
# =============================================================================

squeezing_rows = [evaluate(FIXED_THETA_FOR_SQUEEZING_SWEEP, r) for r in SQUEEZING_SWEEP]

# -----------------------------------------------------------------------------
# Validation
# -----------------------------------------------------------------------------

all_rows = theta_rows + squeezing_rows
failures = [
    row for row in all_rows
    if row["map_frobenius_error"] > MAP_TOL
    or row["oracle_map_error"] > MAP_TOL
    or row["noise_oracle_error"] > NOISE_ORACLE_TOL
]
if failures:
    common.save_csv(failures, "R3_rotation_compile_FAILURES")
    raise AssertionError(f"{len(failures)} cases exceeded tolerance; see FAILURES csv")

common.save_result(theta_rows, "R3_rotation_compile_theta_sweep")
common.save_result(
    squeezing_rows,
    "R3_rotation_compile_squeezing_sweep",
    extra={"fixed_theta": FIXED_THETA_FOR_SQUEEZING_SWEEP},
)

# =============================================================================
# Final 2 x 2 publication figure
# =============================================================================

theta_vals = np.array([r["theta"] for r in theta_rows])
r_vals = np.array([r["squeezing"] for r in squeezing_rows])

fig, axes = plt.subplots(2, 2, figsize=(9.4, 7.2), facecolor="white")
axA, axB, axC, axD = axes.ravel()

floor = 1e-16

# -----------------------------------------------------------------------------
# A. Map validation vs theta (should be flat at machine precision, any r)
# -----------------------------------------------------------------------------

axA.semilogy(
    theta_vals,
    np.maximum([r["map_frobenius_error"] for r in theta_rows], floor),
    "o-", color="red", label=r"$\|S_{\rm pattern}-R(\theta)\|_F$",
)
axA.semilogy(
    theta_vals,
    np.maximum([r["oracle_map_error"] for r in theta_rows], floor),
    "s--", color="blue", label=r"$\|S_{\rm oracle}-R(\theta)\|_F$",
)
axA.axhline(MAP_TOL, color="black", linestyle="--", linewidth=0.8, label="Tolerance")
style_axis(axA, r"Rotation angle $\theta$ (rad)", "Map Frobenius error",
           f"Map validation ($r={FIXED_SQUEEZING_FOR_THETA_SWEEP}$)", "A")
axA.legend(frameon=False, loc="best")

# -----------------------------------------------------------------------------
# B. Noise mechanism vs theta: compiled/oracle noise + synthesis-shear origin
# -----------------------------------------------------------------------------

axB.semilogy(theta_vals, [r["noise_spectral_norm"] for r in theta_rows],
             "o-", color="blue", label=r"$\|N\|_2$ (compiled)")
axB.semilogy(theta_vals, [r["noise_trace"] for r in theta_rows],
             "^:", color="magenta", label=r"$\mathrm{Tr}(N)$ (compiled)")
axB.axhline(4 * np.exp(-2 * FIXED_SQUEEZING_FOR_THETA_SWEEP), color="black",
            linestyle="--", linewidth=0.8, label=r"$4e^{-2r}$ (identity wire, R2)")

axB_twin = axB.twinx()
axB_twin.semilogy(theta_vals, np.abs(np.tan(theta_vals / 2)) + 1e-12,
                   "d-.", color="cyan", alpha=0.8, label=r"$|\tan(\theta/2)|$ (shear)")
axB_twin.set_ylabel(r"$|\tan(\theta/2)|$", color="cyan")
axB_twin.tick_params(axis="y", colors="cyan")

style_axis(axB, r"Rotation angle $\theta$ (rad)", "Added covariance noise",
           "Noise floor: synthesis amplification", "B")
lines1, labels1 = axB.get_legend_handles_labels()
lines2, labels2 = axB_twin.get_legend_handles_labels()
axB.legend(lines1 + lines2, labels1 + labels2, frameon=False, loc="upper center",
           fontsize=6.5, ncol=1)

# -----------------------------------------------------------------------------
# C. Noise vs resource squeezing r (fixed theta): exponential suppression
# -----------------------------------------------------------------------------

axC.semilogy(r_vals, [r["noise_spectral_norm"] for r in squeezing_rows],
             "o-", color="blue", label=r"$\|N\|_2$ (compiled)")
axC.semilogy(r_vals, [r["noise_trace"] for r in squeezing_rows],
             "^:", color="magenta", label=r"$\mathrm{Tr}(N)$ (compiled)")
k_fixed = decompose_symplectic(independent_rotation_matrix(FIXED_THETA_FOR_SQUEEZING_SWEEP))
_, N_theory_r = zip(*(wire_channel(k_fixed, r) for r in r_vals))
axC.semilogy(r_vals, [np.linalg.norm(N, ord=2) for N in N_theory_r],
             "--", color="black", linewidth=1.0, label=r"Oracle $\|N\|_2$")
style_axis(axC, "Resource squeezing $r$", "Added covariance noise",
           rf"Squeezing scan ($\theta={FIXED_THETA_FOR_SQUEEZING_SWEEP}$)", "C")
axC.legend(frameon=False, loc="best")

# -----------------------------------------------------------------------------
# D. Map + noise-oracle validation vs r (should stay at machine precision)
# -----------------------------------------------------------------------------

axD.semilogy(r_vals, np.maximum([r["map_frobenius_error"] for r in squeezing_rows], floor),
             "o-", color="red", label="Map error")
axD.semilogy(r_vals, np.maximum([r["noise_oracle_error"] for r in squeezing_rows], floor),
             "s--", color="blue", label="Noise-oracle error")
axD.axhline(MAP_TOL, color="black", linestyle="--", linewidth=0.8, label="Tolerance")
style_axis(axD, "Resource squeezing $r$", "Numerical error",
           "Channel validation vs. $r$", "D")
axD.legend(frameon=False, loc="best")

fig.suptitle("Rotation-gate compilation: exact map, finite-squeezing noise floor",
             fontsize=11, fontweight="bold", y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.96], h_pad=2.0, w_pad=2.2)

common.save_figure(fig, "R3_rotation_compile")
plt.close(fig)

common.print_summary(
    "R3 rotation compiler validation",
    theta_points=len(theta_rows),
    squeezing_points=len(squeezing_rows),
    max_map_error=max(r["map_frobenius_error"] for r in all_rows),
    max_noise_oracle_error=max(r["noise_oracle_error"] for r in all_rows),
    max_noise_norm=max(r["noise_spectral_norm"] for r in theta_rows),
)


## R4 — Squeezing Gate Compilation

R4 checks that `Circuit(1).squeeze(0, r_logical)` reproduces the ideal single-mode
squeezing map for a wide range of (possibly large) logical squeezing, and connects
the compiled map's numerical conditioning directly to the resource cost of
synthesizing it from teleportation steps (same machinery as R1–R3).

### Target map and its intrinsic conditioning

The ideal logical squeezer is (`photographiq.gaussian.squeezing`, reproduced here
independently)

$$
S(r_{\text{logical}}) =
\begin{pmatrix} e^{-r_{\text{logical}}} & 0 \\ 0 & e^{r_{\text{logical}}} \end{pmatrix}.
$$

Because \(S\) is diagonal, its 2-norm condition number is exactly

$$
\kappa\big(S(r_{\text{logical}})\big) = e^{2|r_{\text{logical}}|},
$$

**independently of the compiler** — this is a property of the target gate itself
(strong squeezing is intrinsically an ill-conditioned linear map: it stretches one
quadrature and compresses the other by the same exponential factor).

### Shear synthesis and the noise floor

`decompose_symplectic` (R3) always takes the "near-singular" branch for a pure
squeezer, since its off-diagonal entry is exactly zero, and returns five shears

$$
k = \big(0,\;0,\;-e^{-r_{\text{logical}}},\;-e^{r_{\text{logical}}},\;-e^{-r_{\text{logical}}}\big).
$$

One shear magnitude grows as \(e^{|r_{\text{logical}}|}\), so — exactly as in R3 —
the finite-squeezing noise floor at fixed resource squeezing \(r\) is amplified by
the synthesis itself through the multi-step recursion

$$
N_{j+1} = T(k_j)\,N_j\,T(k_j)^{T} + \operatorname{diag}(0,e^{-2r}),
\qquad N_0=0,
$$

so \(\|N\|_2\) is expected to grow with \(|r_{\text{logical}}|\), tracking (not
exactly reproducing, since the growth also depends on step order) the same
\(e^{2|r_{\text{logical}}|}\) scale as \(\kappa(S)\).

### What is checked

For a sweep of \(r_{\text{logical}}\) at fixed resource squeezing \(r=1.2\):

1. Compiled map \(S_{\text{pattern}}\) vs. the independent target \(S(r_{\text{logical}})\)
   (Frobenius error) — exact for every \(r_{\text{logical}}\).
2. Compiled noise \(N_{\text{pattern}}\) vs. the closed-form oracle obtained by
   feeding the actual shears \(k\) through the independent recursion above
   (same `wire_channel` oracle as R1–R3).
3. \(\kappa(S_{\text{pattern}})\) against the closed form \(e^{2|r_{\text{logical}}|}\).
4. Finiteness of the compiled matrix/noise (`np.isfinite`) as a basic numerical
   sanity guard for large \(|r_{\text{logical}}|\).

### Success criteria

- Map and noise-oracle errors \(\lesssim 10^{-9}\) for all tested
  \(r_{\text{logical}}\in[-1.5,1.5]\).
- \(\kappa(S_{\text{pattern}})\) matches \(e^{2|r_{\text{logical}}|}\) to machine
  precision (panel B) — a strong-squeezing gate is provably ill-conditioned, not a
  compiler artifact.
- \(\|N\|_2\) grows monotonically with \(|r_{\text{logical}}|\) (panel C), the same
  synthesis-amplification mechanism identified in R3.
- All quantities remain finite over the tested range (panel D).


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg
from photographiq.compiler import decompose_symplectic
from photographiq.gaussian import wire_channel

set_reference_style()

LOGICAL_SQUEEZING = [-1.5, -1.0, -0.5, -0.2, 0.0, 0.2, 0.5, 1.0, 1.5]
RESOURCE_SQUEEZING = 1.2

MAP_TOL = 1e-9
NOISE_ORACLE_TOL = 1e-9


def independent_squeezing_matrix(r: float) -> np.ndarray:
    """Independent NumPy squeezing matrix; not photographiq.gaussian.squeezing."""
    return np.array([[np.exp(-r), 0.0], [0.0, np.exp(r)]])


rows = []
for r_logical in LOGICAL_SQUEEZING:
    target = independent_squeezing_matrix(r_logical)

    circuit = pg.Circuit(1).squeeze(0, r_logical)
    pattern = circuit.compile(squeezing=RESOURCE_SQUEEZING)
    channel = pg.gaussian_channel(pattern)

    # Independent noise oracle from the *actual* synthesis shears, same
    # recursion validated in R1-R3:
    #
    # k = (0, 0, -e^(-r_logical), -e^(r_logical), -e^(-r_logical))
    ks = decompose_symplectic(target)
    oracle_S, oracle_N = wire_channel(ks, RESOURCE_SQUEEZING)

    # Intrinsic conditioning of the TARGET gate itself:
    #
    # kappa(S) = exp(2|r_logical|), independent of the compiler
    theory_condition = float(np.exp(2 * abs(r_logical)))

    rows.append(
        {
            "logical_squeezing_r": r_logical,
            "resource_squeezing": RESOURCE_SQUEEZING,
            "map_frobenius_error": common.frobenius_error(channel.matrix, target),
            "oracle_map_error": common.frobenius_error(oracle_S, target),
            "noise_oracle_error": common.frobenius_error(channel.noise, oracle_N),
            "noise_spectral_norm": float(np.linalg.norm(channel.noise, ord=2)),
            "noise_trace": float(np.trace(channel.noise)),
            "matrix_condition_number": float(np.linalg.cond(channel.matrix)),
            "theory_condition_number": theory_condition,
            "max_abs_shear": float(np.max(np.abs(ks))),
            "matrix_finite": bool(np.isfinite(channel.matrix).all()),
            "noise_finite": bool(np.isfinite(channel.noise).all()),
        }
    )

# -----------------------------------------------------------------------------
# Validation
# -----------------------------------------------------------------------------

failures = [
    row for row in rows
    if row["map_frobenius_error"] > MAP_TOL
    or row["oracle_map_error"] > MAP_TOL
    or row["noise_oracle_error"] > NOISE_ORACLE_TOL
    or not row["matrix_finite"]
    or not row["noise_finite"]
]
if failures:
    common.save_csv(failures, "R4_squeezing_compile_FAILURES")
    raise AssertionError(f"{len(failures)} cases exceeded tolerance; see FAILURES csv")

common.save_result(rows, "R4_squeezing_compile", extra={"resource_squeezing": RESOURCE_SQUEEZING})

# =============================================================================
# Final 2 x 2 publication figure
# =============================================================================

rs = np.array([r["logical_squeezing_r"] for r in rows])
floor = 1e-16

fig, axes = plt.subplots(2, 2, figsize=(9.4, 7.2), facecolor="white")
axA, axB, axC, axD = axes.ravel()

# -----------------------------------------------------------------------------
# A. Map + noise-oracle validation (should stay at machine precision)
# -----------------------------------------------------------------------------

axA.semilogy(rs, np.maximum([r["map_frobenius_error"] for r in rows], floor),
             "o-", color="red", label=r"$\|S_{\rm pattern}-S(r)\|_F$")
axA.semilogy(rs, np.maximum([r["noise_oracle_error"] for r in rows], floor),
             "s--", color="blue", label="Noise-oracle error")
axA.axhline(MAP_TOL, color="black", linestyle="--", linewidth=0.8, label="Tolerance")
style_axis(axA, "Logical squeezing $r_{\\mathrm{logical}}$", "Numerical error",
           "Channel validation", "A")
axA.legend(frameon=False, loc="best")

# -----------------------------------------------------------------------------
# B. Intrinsic conditioning: kappa(S) = exp(2|r_logical|)
# -----------------------------------------------------------------------------

axB.semilogy(rs, [r["matrix_condition_number"] for r in rows],
             "o-", color="blue", label=r"$\kappa(S_{\rm pattern})$")
axB.semilogy(rs, [r["theory_condition_number"] for r in rows],
             "--", color="black", linewidth=1.0, label=r"$e^{2|r_{\rm logical}|}$")
style_axis(axB, "Logical squeezing $r_{\\mathrm{logical}}$", "Condition number",
           "Intrinsic gate conditioning", "B")
axB.legend(frameon=False, loc="best")

# -----------------------------------------------------------------------------
# C. Noise floor vs logical squeezing: synthesis amplification (cf. R3)
# -----------------------------------------------------------------------------

axC.semilogy(rs, [r["noise_spectral_norm"] for r in rows],
             "o-", color="blue", label=r"$\|N\|_2$ (compiled)")
axC.semilogy(rs, [r["noise_trace"] for r in rows],
             "^:", color="magenta", label=r"$\mathrm{Tr}(N)$ (compiled)")
axC.axhline(4 * np.exp(-2 * RESOURCE_SQUEEZING), color="black", linestyle="--",
            linewidth=0.8, label=r"$4e^{-2r}$ (identity wire, R2)")
style_axis(axC, "Logical squeezing $r_{\\mathrm{logical}}$", "Added covariance noise",
           f"Noise floor ($r={RESOURCE_SQUEEZING}$)", "C")
axC.legend(frameon=False, loc="best")

# -----------------------------------------------------------------------------
# D. Synthesis shear magnitude driving both B and C
# -----------------------------------------------------------------------------

axD.semilogy(rs, [r["max_abs_shear"] for r in rows],
             "o-", color="red", label=r"$\max_j|k_j|$ (compiled)")
axD.semilogy(rs, np.exp(np.abs(rs)), "--", color="black", linewidth=1.0,
             label=r"$e^{|r_{\rm logical}|}$")
style_axis(axD, "Logical squeezing $r_{\\mathrm{logical}}$", "Max shear magnitude",
           "Synthesis shear growth", "D")
axD.legend(frameon=False, loc="best")

fig.suptitle(f"Squeezing-gate compilation, resource squeezing $r={RESOURCE_SQUEEZING}$",
             fontsize=11, fontweight="bold", y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.96], h_pad=2.0, w_pad=2.2)

common.save_figure(fig, "R4_squeezing_compile")
plt.close(fig)

common.print_summary(
    "R4 squeezing compiler validation",
    points=len(rows),
    max_map_error=max(r["map_frobenius_error"] for r in rows),
    max_noise_oracle_error=max(r["noise_oracle_error"] for r in rows),
    max_condition_number=max(r["matrix_condition_number"] for r in rows),
)


## R5 — Random Single-Mode Symplectic Compilation

R5 is the broadest quantitative test of the single-mode compiler (R3/R4 checked
two specific gate families; R5 checks *arbitrary* valid single-mode Gaussian
gates). It draws 400 random real symplectic matrices, compiles each through the
same `decompose_symplectic` machinery, and measures how close the compiled
channel gets to the exact target.

### Random target construction

Every single-mode symplectic matrix with \(\det S=1\) factors as a rotation,
a squeezer, and another rotation (Euler-like decomposition, R3+R4 building
blocks):

$$
S(\varphi_1,r,\varphi_2) = R(\varphi_1)\,S_{\text{sq}}(r)\,R(\varphi_2),
\qquad
\varphi_1,\varphi_2\sim U(-\pi,\pi),\quad r\sim U(-1.6,1.6).
$$

This target is built with plain NumPy (independent of `photographiq.gaussian`),
so `determinant(S)=1` to machine precision is itself a free structural check on
the sampler, plotted alongside the compiler error.

### What is checked

For each sample, `Circuit(1).gaussian(0, S)` calls `decompose_symplectic(S)`
(R3) and compiles the resulting teleportation chain. Because \(S\) is
essentially arbitrary (not a pure rotation or squeezer), the near-singular
branch (\(|c|<10^{-8}\), R3-R4) can trigger for either family, using 5 shears
instead of 4; `decompose_symplectic` can also raise `ValueError` for a
genuinely unstable decomposition request (documented in `docs/theory.md`) —
these are recorded as decomposition failures, never silently skipped.

For every successfully compiled sample:

$$
\epsilon = \|S_{\text{pattern}} - S\|_F,
$$

using the same independent Gaussian-channel analyzer as R1–R4.

### Success criteria

- \(\epsilon\) clusters tightly near machine precision (\(10^{-16}\)–\(10^{-12}\))
  across the *entire* random parameter space, with no systematic dependence on
  the squeezing strength \(r\) — i.e. the noise floor in panel A/B is sampling
  jitter in the last few bits, not a compiler weakness that appears for
  particular gates (panel A, B).
- All sampled determinants equal 1 to machine precision (panel D) — a free
  cross-check that the random sampler itself is correct.
- Any decomposition failures are rare, explicitly reported (not hidden), and
  restricted to the near-singular corner of parameter space (panel C reports
  how many shear steps each successful compilation used).


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg
from photographiq.compiler import decompose_symplectic

set_reference_style()

N_SAMPLES = 400
SEED = 20260913
SQUEEZING = 1.0
R_RANGE = (-1.6, 1.6)
DET_TOL = 1e-10          # sampler self-consistency (exact algebraic identity)
MAP_TOL = 1e-7           # near-machine-precision compiler validation


def independent_rotation(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])


def independent_squeezing(r):
    return np.array([[np.exp(-r), 0.0], [0.0, np.exp(r)]])


def random_symplectic(generator: np.random.Generator):
    # S(phi1, r, phi2) = R(phi1) . S_sq(r) . R(phi2)
    phi1 = generator.uniform(-np.pi, np.pi)
    phi2 = generator.uniform(-np.pi, np.pi)
    r = generator.uniform(*R_RANGE)
    matrix = independent_rotation(phi1) @ independent_squeezing(r) @ independent_rotation(phi2)
    return matrix, phi1, r, phi2


generator = common.rng(SEED)
rows = []
for i in range(N_SAMPLES):
    target, phi1, r, phi2 = random_symplectic(generator)
    row = {
        "sample": i,
        "phi1": phi1,
        "r": r,
        "phi2": phi2,
        "determinant": float(np.linalg.det(target)),
        "determinant_error": abs(float(np.linalg.det(target)) - 1.0),
    }
    try:
        ks = decompose_symplectic(target)
        circuit = pg.Circuit(1).gaussian(0, target)
        pattern, trace = circuit.compile(squeezing=SQUEEZING, return_trace=True)
        channel = pg.gaussian_channel(pattern)
        error = common.frobenius_error(channel.matrix, target)
        step = trace.steps[0]
        row.update(
            {
                "decomposition_failed": False,
                "frobenius_error": error,
                "log10_error": common.safe_log10(error),
                "n_shear_steps": len(ks),
                "resource_nodes": len(step.resource_nodes),
                "measurements": len(step.measurements),
                "commands": len(pattern.commands),
            }
        )
    except ValueError as exc:
        row.update(
            {
                "decomposition_failed": True,
                "error_message": str(exc),
                "frobenius_error": None,
                "log10_error": None,
                "n_shear_steps": None,
                "resource_nodes": None,
                "measurements": None,
                "commands": None,
            }
        )
    rows.append(row)

successes = [r for r in rows if not r["decomposition_failed"]]
failed = [r for r in rows if r["decomposition_failed"]]

# -----------------------------------------------------------------------------
# Validation
# -----------------------------------------------------------------------------

bad = [r for r in successes if r["frobenius_error"] > MAP_TOL]
if bad:
    common.save_csv(bad, "R5_random_symplectic_compile_FAILURES")
    raise AssertionError(f"{len(bad)} compiled maps exceeded tolerance {MAP_TOL}; see FAILURES csv")

bad_det = [r for r in rows if r["determinant_error"] > DET_TOL]
if bad_det:
    common.save_csv(bad_det, "R5_random_symplectic_compile_DET_FAILURES")
    raise AssertionError("Random sampler produced a non-unit-determinant matrix; see DET_FAILURES csv")

common.save_result(
    rows,
    "R5_random_symplectic_compile",
    extra={
        "n_samples": N_SAMPLES,
        "seed": SEED,
        "squeezing": SQUEEZING,
        "r_range": list(R_RANGE),
        "n_successes": len(successes),
        "n_decomposition_failures": len(failed),
    },
)
if failed:
    common.save_csv(failed, "R5_random_symplectic_compile_decomposition_failures")

# =============================================================================
# Final 2 x 2 publication figure
# =============================================================================

log_errors = np.array([r["log10_error"] for r in successes])
r_vals = np.array([r["r"] for r in successes])
steps = np.array([r["n_shear_steps"] for r in successes])
det_err = np.maximum([r["determinant_error"] for r in rows], 1e-18)

fig, axes = plt.subplots(2, 2, figsize=(9.6, 7.4), facecolor="white")
axA, axB, axC, axD = axes.ravel()

# -----------------------------------------------------------------------------
# A. Main result: distribution of compiled-vs-analytic map error
# -----------------------------------------------------------------------------

axA.hist(log_errors, bins=30, color="blue", edgecolor="white", alpha=0.85)
axA.axvline(np.median(log_errors), color="red", linestyle="--", linewidth=1.2,
            label=f"median = {np.median(log_errors):.1f}")
style_axis(axA, r"$\log_{10}\|S_{\rm pattern}-S\|_F$", "Count",
           f"Map error distribution (n={len(successes)})", "A")
axA.legend(frameon=False, loc="best")

# -----------------------------------------------------------------------------
# B. No systematic dependence on squeezing strength -> genuine noise floor
# -----------------------------------------------------------------------------

axB.scatter(r_vals, log_errors, s=10, color="blue", alpha=0.5, edgecolors="none")
style_axis(axB, "Sampled squeezing $r$", r"$\log_{10}$ map error",
           "Error vs. gate squeezing strength", "B")

# -----------------------------------------------------------------------------
# C. Shear-step usage (4 = generic gate, 5 = near-singular branch, R3/R4)
# -----------------------------------------------------------------------------

values, counts = np.unique(steps, return_counts=True)
axC.bar(values, counts, color="magenta", alpha=0.8, width=0.5)
axC.set_xticks(values)
style_axis(axC, "Teleportation shear steps used", "Count",
           "Decomposition branch usage", "C")

# -----------------------------------------------------------------------------
# D. Sampler self-consistency: det(S) = 1 to machine precision
# -----------------------------------------------------------------------------

axD.semilogy(range(len(rows)), det_err, ".", color="red", markersize=3, alpha=0.6)
axD.axhline(DET_TOL, color="black", linestyle="--", linewidth=0.8, label="Tolerance")
style_axis(axD, "Sample index", r"$|\det(S)-1|$",
           "Random-sampler consistency", "D")
axD.legend(frameon=False, loc="best")

fig.suptitle(
    f"Random single-mode symplectic compilation "
    f"(n={N_SAMPLES}, {len(failed)} decomposition failures)",
    fontsize=11, fontweight="bold", y=0.99,
)
fig.tight_layout(rect=[0, 0, 1, 0.96], h_pad=2.0, w_pad=2.2)

common.save_figure(fig, "R5_random_symplectic_compile")
plt.close(fig)

common.print_summary(
    "R5 random symplectic compilation",
    n_samples=N_SAMPLES,
    n_successes=len(successes),
    n_decomposition_failures=len(failed),
    max_error=max((r["frobenius_error"] for r in successes), default=None),
    median_log10_error=float(np.median(log_errors)) if len(log_errors) else None,
)


## R6 — Two-Mode CZ Compilation

R6 checks the two-mode entangling primitive: `Circuit(2).cz(0, 1, weight)`,
lowered by `_Builder.cz` (`src/photographiq/compiler.py`) as one instantaneous
`Entangle` interaction followed by a bare four-step identity wire (R2) on
*each* output mode (so every logical circuit ends on fresh, freely-relabelable
nodes). This gives two independent, checkable predictions: an exact linear map
and a noise term with a specific, checkable structure.

### Ideal map and noise structure

The `Entangle(u, v, weight)` command implements (package docstring,
`src/photographiq/commands.py`)

$$
p_u \to p_u + g\,q_v, \qquad p_v \to p_v + g\,q_u,
$$

i.e., on \((q_0,p_0,q_1,p_1)\),

$$
\mathrm{CZ}(g)=
\begin{pmatrix}
1&0&0&0\\
0&1&g&0\\
0&0&1&0\\
g&0&0&1
\end{pmatrix}.
$$

Because the entangling interaction itself is an *exact* symplectic map (no
teleportation, no resource state), it adds **no** noise. All finite-squeezing
noise comes only from the identity wires appended afterward, so the total
noise is block-diagonal in the per-mode identity-wire noise (R2):

$$
N_{\mathrm{CZ}} =
\begin{pmatrix} N_{\text{wire}}(r) & 0 \\ 0 & N_{\text{wire}}(r)\end{pmatrix},
\qquad
N_{\text{wire}}(r) = 2e^{-2r} I_2
\quad\text{(independent of the CZ weight }g\text{)}.
$$

### The real entanglement signature

For covariance-\(I\) inputs, \(\mathrm{CZ}(g)\) leaves \(\operatorname{Cov}(q_0,q_1)=0\)
(the two \(q\)'s never mix), but generates a genuine cross-mode correlation
between the *conjugate* quadratures,

$$
\operatorname{Cov}(p_0,q_1) = \operatorname{Cov}(q_0,p_1) = g,
$$

directly from the two off-diagonal weight entries above — this, not the
\(q_0\)-\(q_1\) covariance, is the measurable signature of the entangling gate.

### What is checked

For a sweep of CZ weights \(g\):

1. Compiled map \(S_{\text{pattern}}\) vs. the independent \(\mathrm{CZ}(g)\)
   target (Frobenius error).
2. Compiled noise \(N_{\text{pattern}}\) vs. the independent block-diagonal
   wire-noise oracle above.
3. Full-state mean/covariance error for representative coherent-state inputs,
   including \(\operatorname{Cov}(p_0,q_1)\) against the exact prediction \(g\).

### Success criteria

- Map and noise-oracle errors \(\lesssim 10^{-8}\) for every \(g\) (panel A).
- \(\|N\|_2\) and \(\operatorname{Tr}(N)\) stay **constant** as \(g\) varies
  (panel B) — entangling is noiseless; only the appended identity wires
  contribute noise, and that noise cannot depend on \(g\).
- \(\operatorname{Cov}(p_0,q_1)\) tracks \(g\) exactly (unit slope, panel D),
  while \(\operatorname{Cov}(q_0,q_1)\) stays at the numerical noise floor
  (panel D) — confirming which correlation actually carries the entangling
  signature.


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg
from photographiq.gaussian import wire_channel

set_reference_style()

WEIGHTS = [-2.0, -1.0, -0.5, -0.1, 0.1, 0.5, 1.0, 1.5, 2.0]
SQUEEZING = 1.0
INPUT_ALPHAS = [(0.3 + 0.1j, -0.2 + 0.4j), (0.0, 0.5 + 0.0j)]
TOL = 1e-8


def independent_cz(weight: float) -> np.ndarray:
    """Independent CZ(g) symplectic matrix on (q0,p0,q1,p1); p_i -> p_i+g*q_j."""
    matrix = np.eye(4)
    matrix[1, 2] = weight  # p0 += g q1
    matrix[3, 0] = weight  # p1 += g q0
    return matrix


def expected_noise(squeezing: float) -> np.ndarray:
    """Independent noise oracle: logical CZ = instantaneous Entangle (exact,
    noiseless) followed by an identity wire (4 k=0 steps, R2) on each output
    mode, so the total noise is block-diagonal in the per-mode wire noise and
    cannot depend on the CZ weight g."""
    _, per_mode_noise = wire_channel([0.0] * 4, squeezing)
    noise = np.zeros((4, 4))
    noise[:2, :2] = per_mode_noise
    noise[2:, 2:] = per_mode_noise
    return noise


rows = []
for weight in WEIGHTS:
    circuit = pg.Circuit(2).cz(0, 1, weight)
    pattern = circuit.compile(squeezing=SQUEEZING)
    channel = pg.gaussian_channel(pattern)
    target = independent_cz(weight)
    noise_oracle = expected_noise(SQUEEZING)

    state_errors = []
    p0_q1_correlations = []
    q0_q1_correlations = []
    for alpha0, alpha1 in INPUT_ALPHAS:
        inputs = pg.GaussianState(
            np.array([2 * alpha0.real, 2 * alpha0.imag, 2 * alpha1.real, 2 * alpha1.imag]),
            np.eye(4),
            (0, 1),
        )
        actual = channel.apply(inputs)
        ideal_mean = target @ inputs.mean
        ideal_cov = target @ inputs.covariance @ target.T + noise_oracle
        state_errors.append(
            {
                "mean_error": common.frobenius_error(actual.mean, ideal_mean),
                "covariance_error": common.frobenius_error(actual.covariance, ideal_cov),
            }
        )
        # The genuine CZ entanglement signature is the conjugate-quadrature
        # correlation Cov(p0, q1) = g; Cov(q0, q1) stays at zero.
        p0_q1_correlations.append(float(actual.covariance[1, 2]))
        q0_q1_correlations.append(float(actual.covariance[0, 2]))

    rows.append(
        {
            "weight": weight,
            "squeezing": SQUEEZING,
            "map_frobenius_error": common.frobenius_error(channel.matrix, target),
            "noise_oracle_frobenius_error": common.frobenius_error(channel.noise, noise_oracle),
            "noise_spectral_norm": float(np.linalg.norm(channel.noise, ord=2)),
            "noise_trace": float(np.trace(channel.noise)),
            "max_state_mean_error": max(s["mean_error"] for s in state_errors),
            "max_state_covariance_error": max(s["covariance_error"] for s in state_errors),
            "mean_p0_q1_correlation": float(np.mean(p0_q1_correlations)),
            "mean_q0_q1_correlation": float(np.mean(q0_q1_correlations)),
        }
    )

# -----------------------------------------------------------------------------
# Validation
# -----------------------------------------------------------------------------

failures = [
    r for r in rows
    if r["map_frobenius_error"] > TOL
    or r["noise_oracle_frobenius_error"] > TOL
    or r["max_state_mean_error"] > TOL
    or r["max_state_covariance_error"] > TOL
    or abs(r["mean_p0_q1_correlation"] - r["weight"]) > TOL
]
if failures:
    common.save_csv(failures, "R6_cz_compile_FAILURES")
    raise AssertionError("Compiled CZ map disagreed with the analytic target; see FAILURES csv")

common.save_result(rows, "R6_cz_compile", extra={"squeezing": SQUEEZING})

# =============================================================================
# Final 2 x 2 publication figure
# =============================================================================

ws = np.array([r["weight"] for r in rows])
floor = 1e-16

fig, axes = plt.subplots(2, 2, figsize=(9.4, 7.2), facecolor="white")
axA, axB, axC, axD = axes.ravel()

# -----------------------------------------------------------------------------
# A. Map and noise-oracle validation
# -----------------------------------------------------------------------------

axA.semilogy(ws, np.maximum([r["map_frobenius_error"] for r in rows], floor),
             "o-", color="red", label=r"$\|S_{\rm pattern}-\mathrm{CZ}(g)\|_F$")
axA.semilogy(ws, np.maximum([r["noise_oracle_frobenius_error"] for r in rows], floor),
             "s--", color="blue", label="Noise-oracle error")
axA.axhline(TOL, color="black", linestyle="--", linewidth=0.8, label="Tolerance")
style_axis(axA, "CZ weight $g$", "Numerical error", "Channel validation", "A")
axA.legend(frameon=False, loc="best")

# -----------------------------------------------------------------------------
# B. Noise is weight-independent: entangling itself is noiseless
# -----------------------------------------------------------------------------

axB.plot(ws, [r["noise_spectral_norm"] for r in rows], "o-", color="blue",
          label=r"$\|N\|_2$")
axB.plot(ws, [r["noise_trace"] for r in rows], "^:", color="magenta",
          label=r"$\mathrm{Tr}(N)$")
axB.axhline(4 * np.exp(-2 * SQUEEZING), color="black", linestyle="--", linewidth=0.8,
            label=r"$4e^{-2r}$ (per-mode wire, R2)")
axB.set_ylim(0, 1.15 * max(r["noise_trace"] for r in rows))
style_axis(axB, "CZ weight $g$", "Added covariance noise",
           "Noise is independent of $g$", "B")
axB.legend(frameon=False, loc="best")

# -----------------------------------------------------------------------------
# C. Full-state validation for representative coherent inputs
# -----------------------------------------------------------------------------

axC.semilogy(ws, np.maximum([r["max_state_mean_error"] for r in rows], floor),
             "o-", color="red", label="Max mean error")
axC.semilogy(ws, np.maximum([r["max_state_covariance_error"] for r in rows], floor),
             "s--", color="blue", label="Max covariance error")
axC.axhline(TOL, color="black", linestyle="--", linewidth=0.8, label="Tolerance")
style_axis(axC, "CZ weight $g$", "Numerical error", "State-level validation", "C")
axC.legend(frameon=False, loc="best")

# -----------------------------------------------------------------------------
# D. The genuine entanglement signature: Cov(p0,q1)=g, not Cov(q0,q1)
# -----------------------------------------------------------------------------

axD.plot(ws, [r["mean_p0_q1_correlation"] for r in rows], "o-", color="red",
          label=r"$\mathrm{Cov}(p_0,q_1)$")
axD.plot(ws, ws, "--", color="black", linewidth=1.0, label=r"$g$ (theory)")
axD.plot(ws, [r["mean_q0_q1_correlation"] for r in rows], "s:", color="blue",
          alpha=0.8, label=r"$\mathrm{Cov}(q_0,q_1)$ ($\approx 0$)")
style_axis(axD, "CZ weight $g$", "Covariance", "Entanglement signature", "D")
axD.legend(frameon=False, loc="best")

fig.suptitle(f"CZ gate compilation, resource squeezing $r={SQUEEZING}$",
             fontsize=11, fontweight="bold", y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.96], h_pad=2.0, w_pad=2.2)

common.save_figure(fig, "R6_cz_compile")
plt.close(fig)

common.print_summary(
    "R6 CZ compilation",
    weights=len(rows),
    max_map_error=max(r["map_frobenius_error"] for r in rows),
    max_noise_norm=max(r["noise_spectral_norm"] for r in rows),
)


## R7 — Beamsplitter Compilation (SUM-Gate Decomposition)

R7 checks the most resource-expensive two-mode primitive: `Circuit(2).beamsplitter(0, 1, theta)`.
Unlike CZ (R6, one direct `Entangle`), the compiler builds a beamsplitter out of
SUM gates, each itself built from CZ sandwiched between two single-mode Fourier
rotations (R3's teleportation machinery, nested three layers deep).

### SUM-sandwich construction

A logical SUM gate is a CZ conjugated by \(\pm\pi/2\) rotations on the target mode
(`_Builder.sum`, `src/photographiq/compiler.py`):

$$
\mathrm{SUM}_{u\to v}(g) = R_v(-\tfrac{\pi}{2})\,\mathrm{CZ}(u,v,g)\,R_v(\tfrac{\pi}{2}).
$$

Per `docs/theory.md`, one beamsplitter of angle \(\varphi\) is

$$
\mathrm{BS}(\varphi) =
\mathrm{SUM}_{v\to u}\!\big(-\tan(\varphi/2)\big)\;
\mathrm{SUM}_{u\to v}\!\big(\sin\varphi\big)\;
\mathrm{SUM}_{v\to u}\!\big(-\tan(\varphi/2)\big),
$$

which is singular as \(\varphi\to\pm\pi\). The compiler instead applies this
sandwich **twice** with the half angle \(\varphi=\theta/2\),

$$
\mathrm{BS}(\theta) = \Big[
\mathrm{SUM}_{v\to u}\!\big(-\tan(\theta/4)\big)\,
\mathrm{SUM}_{u\to v}\!\big(\sin(\theta/2)\big)\,
\mathrm{SUM}_{v\to u}\!\big(-\tan(\theta/4)\big)
\Big]^2,
$$

which pushes the \(\tan\) singularity out to \(\theta=\pm2\pi\) — safely outside
any physical beamsplitter angle. \(\theta=\pm\pi\) itself is therefore excluded
from the sweep only as an extra numerical margin, not because of a real pole.

Six SUM gates, each three single-mode teleportation chains plus one CZ, make
this **deliberately resource-expensive** (`docs/theory.md`): every compiled
beamsplitter pays the same, angle-independent resource cost, regardless of how
small \(\theta\) is — a design choice favoring compiler simplicity/transparency
over efficiency.

### What is checked

For a sweep of \(\theta\) at fixed resource squeezing \(r=1.2\):

1. Compiled map \(S_{\text{pattern}}\) vs. the independent target
   \(\mathrm{BS}(\theta)=\begin{pmatrix}c&0&-s&0\\0&c&0&-s\\s&0&c&0\\0&s&0&c\end{pmatrix}\)
   (\(c=\cos\theta\), \(s=\sin\theta\); package sign convention).
2. The MBQC resource overhead: resource nodes, measurements and total commands
   for one compiled beamsplitter.

### Success criteria

- Map error stays small (\(\lesssim10^{-6}\), looser than R3–R6 because six
  nested single-mode syntheses compound floating-point error) for every
  \(\theta\) in the sweep (panel A).
- Resource counts are **constant** in \(\theta\) (panel B) — confirming the
  fixed, angle-independent cost predicted above, not a numerical coincidence.
- No singular behavior appears near \(\theta=\pm\pi\) (panel A), consistent with
  the half-angle construction moving the true pole to \(\pm2\pi\).


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg

set_reference_style()

THETAS = np.linspace(-np.pi + 0.05, np.pi - 0.05, 17)  # extra margin; true pole is at +-2pi
SQUEEZING = 1.2
MAP_TOL = 1e-6
DET_TOL = 1e-6


def independent_beamsplitter(theta: float) -> np.ndarray:
    """Independent beamsplitter symplectic matrix (package sign convention)."""
    c, s = np.cos(theta), np.sin(theta)
    return np.array(
        [
            [c, 0, -s, 0],
            [0, c, 0, -s],
            [s, 0, c, 0],
            [0, s, 0, c],
        ]
    )


rows = []
for theta in THETAS:
    circuit = pg.Circuit(2).beamsplitter(0, 1, float(theta))
    pattern, trace = circuit.compile(squeezing=SQUEEZING, return_trace=True)
    channel = pg.gaussian_channel(pattern)
    target = independent_beamsplitter(float(theta))
    step = trace.steps[0]
    rows.append(
        {
            "theta": float(theta),
            "squeezing": SQUEEZING,
            "map_frobenius_error": common.frobenius_error(channel.matrix, target),
            # A compiled 2-mode symplectic map must have det=1 exactly; a
            # structural check independent of the analytic BS(theta) formula.
            "determinant_error": abs(float(np.linalg.det(channel.matrix)) - 1.0),
            "noise_spectral_norm": float(np.linalg.norm(channel.noise, ord=2)),
            "noise_trace": float(np.trace(channel.noise)),
            "resource_nodes": len(step.resource_nodes),
            "measurements": len(step.measurements),
            "commands": len(pattern.commands),
        }
    )

# -----------------------------------------------------------------------------
# Validation
# -----------------------------------------------------------------------------

failures = [
    r for r in rows
    if r["map_frobenius_error"] > MAP_TOL or r["determinant_error"] > DET_TOL
]
if failures:
    common.save_csv(failures, "R7_beamsplitter_compile_FAILURES")
    raise AssertionError("Compiled beamsplitter map disagreed with the analytic target; see FAILURES csv")

common.save_result(rows, "R7_beamsplitter_compile", extra={"squeezing": SQUEEZING})

# =============================================================================
# Final 2 x 2 publication figure
# =============================================================================

ts = np.array([r["theta"] for r in rows])
floor = 1e-16

fig, axes = plt.subplots(2, 2, figsize=(9.4, 7.2), facecolor="white")
axA, axB, axC, axD = axes.ravel()

# -----------------------------------------------------------------------------
# A. Map validation across the full angle range: no pole near +-pi
# -----------------------------------------------------------------------------

axA.semilogy(ts, np.maximum([r["map_frobenius_error"] for r in rows], floor),
             "o-", color="red")
axA.axhline(MAP_TOL, color="black", linestyle="--", linewidth=0.8, label="Tolerance")
style_axis(axA, r"Beamsplitter angle $\theta$ (rad)", "Map Frobenius error",
           "Map validation (no pole near $\\pm\\pi$)", "A")
axA.legend(frameon=False, loc="best")

# -----------------------------------------------------------------------------
# B. Resource cost is angle-independent (fixed 6-SUM-gate construction)
# -----------------------------------------------------------------------------

axB.plot(ts, [r["resource_nodes"] for r in rows], "o-", color="blue", label="Resource nodes")
axB.plot(ts, [r["measurements"] for r in rows], "s--", color="magenta", label="Measurements")
axB.plot(ts, [r["commands"] for r in rows], "^:", color="cyan", label="Commands")
style_axis(axB, r"Beamsplitter angle $\theta$ (rad)", "MBQC resource count",
           "Cost is angle-independent", "B")
axB.legend(frameon=False, loc="best", fontsize=6.5)

# -----------------------------------------------------------------------------
# C. Finite-squeezing noise floor across the angle sweep
# -----------------------------------------------------------------------------

axC.semilogy(ts, [r["noise_spectral_norm"] for r in rows], "o-", color="blue",
             label=r"$\|N\|_2$")
axC.semilogy(ts, [r["noise_trace"] for r in rows], "^:", color="magenta",
             label=r"$\mathrm{Tr}(N)$")
style_axis(axC, r"Beamsplitter angle $\theta$ (rad)", "Added covariance noise",
           f"Noise floor ($r={SQUEEZING}$)", "C")
axC.legend(frameon=False, loc="best")

# -----------------------------------------------------------------------------
# D. Structural check: det(S)=1 for every compiled symplectic map
# -----------------------------------------------------------------------------

axD.semilogy(ts, np.maximum([r["determinant_error"] for r in rows], floor),
             "o-", color="red")
axD.axhline(DET_TOL, color="black", linestyle="--", linewidth=0.8, label="Tolerance")
style_axis(axD, r"Beamsplitter angle $\theta$ (rad)", r"$|\det(S_{\rm pattern})-1|$",
           "Symplecticity check", "D")
axD.legend(frameon=False, loc="best")

fig.suptitle(f"Beamsplitter compilation (SUM-gate decomposition), resource squeezing $r={SQUEEZING}$",
             fontsize=11, fontweight="bold", y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.96], h_pad=2.0, w_pad=2.2)

common.save_figure(fig, "R7_beamsplitter_compile")
plt.close(fig)

common.print_summary(
    "R7 beamsplitter compilation",
    thetas=len(rows),
    max_map_error=max(r["map_frobenius_error"] for r in rows),
    max_resource_nodes=max(r["resource_nodes"] for r in rows),
    max_commands=max(r["commands"] for r in rows),
)


## R8 — Multi-Mode Compilation Showcase (Fig. 3 candidate)

R8 is a **structural/illustrative** experiment, not a pass/fail numerical test:
it compiles one five-mode circuit mixing every gate family exercised
individually in R3–R7 (rotation, squeezing, CZ, displacement, beamsplitter) and
renders the package's own gate-to-resource provenance figure
(`photographiq.visualize_compilation`), so a reader can see, gate by gate, how
a logical circuit becomes an MBQC resource graph.

### Circuit and compiled resource graph

$$
\text{Circuit} = R_0(0.4)\;\;S_1(0.3)\;\;R_2(-0.6)\;\;\mathrm{CZ}(0,1;0.8)\;\;
D_3(0.2,-0.1)\;\;\mathrm{BS}(2,3;0.3)\;\;R_4(0.5)\;\;\mathrm{CZ}(3,4;-0.5)
$$

Each gate lowers to one or more MBQC commands (`Prepare`, `Entangle`, `Measure`,
`Displace`) via the same `_Builder` used in R3–R7; `CompilationTrace`/
`CompilationStep` (R9) record, per gate, exactly which resource nodes,
measurements and corrections it produced and which frontier nodes it consumed
and returned. The MBQC resource graph is the pattern's entanglement graph
\(G=(V,E)\), where \(V\) is every prepared/input node and \(E\) is every
`Entangle` edge.

Per `docs/theory.md`, beamsplitter compilation is deliberately the most
resource-expensive primitive (R7: six SUM gates), so this circuit is kept small
enough to stay under `visualize_compilation`'s built-in size guard
(\(\le 24\) gates, \(\le 150\) nodes) while still mixing every gate family.

### What is shown

1. The package's own compilation figure: source circuit wires on one side,
   generated MBQC nodes/edges/measurements on the other, connected by
   provenance arrows.
2. A resource-accounting bar chart (this notebook's addition) breaking down
   resource nodes, measurements and corrections **per logical gate**, so the
   relative MBQC cost of each gate family (a single rotation vs. a full
   beamsplitter) is visible quantitatively, not just visually in the graph.

### What this experiment does *not* claim

This is a descriptive/software-engineering result (illustrating the compiler's
gate-to-resource mapping), not a numerical accuracy claim — the actual map and
noise validation for each gate family is R3–R7; R9 separately verifies the
trace's bookkeeping is complete and self-consistent.


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg

set_reference_style()

SQUEEZING = 1.0


def build_circuit() -> pg.Circuit:
    circuit = pg.Circuit(5)
    circuit.rotate(0, 0.4)
    circuit.squeeze(1, 0.3)
    circuit.rotate(2, -0.6)
    circuit.cz(0, 1, 0.8)
    circuit.displace(3, q=0.2, p=-0.1)
    circuit.beamsplitter(2, 3, 0.3)
    circuit.rotate(4, 0.5)
    circuit.cz(3, 4, -0.5)
    return circuit


circuit = build_circuit()
pattern, trace = circuit.compile(squeezing=SQUEEZING, return_trace=True)
graph = pattern.graph

entangle_edges = [c for c in pattern.commands if isinstance(c, pg.Entangle)]
measurements = [c for c in pattern.commands if isinstance(c, pg.Measure)]
corrections = [c for c in pattern.commands if isinstance(c, pg.Displace)]

summary = {
    "logical_modes": circuit.modes,
    "source_gates": len(circuit.gates),
    "mbqc_nodes": len(graph.nodes),
    "measurements": len(measurements),
    "entanglement_edges": len(entangle_edges),
    "corrections": len(corrections),
    "commands": len(pattern.commands),
}
if summary["mbqc_nodes"] > 150 or summary["source_gates"] > 24:
    raise AssertionError("Showcase circuit exceeds visualize_compilation's size guard")

steps_rows = [
    {
        "gate_index": s.gate_index,
        "source_gate": s.source_gate,
        "source_modes": s.source_modes,
        "resource_nodes": len(s.resource_nodes),
        "measurements": len(s.measurements),
        "corrections": len(s.corrections),
        "commands": len(s.command_indices),
    }
    for s in trace.steps
]

common.save_json(summary, "R8_multimode_showcase_summary")
common.save_csv(steps_rows, "R8_multimode_showcase_steps")
common.write_metadata("R8_multimode_showcase")
common.save_raw(pattern.to_json(), "R8_multimode_showcase_pattern.json")

# =============================================================================
# Panel 1: the package's own gate-to-resource compilation figure
# =============================================================================

fig1 = pg.visualize_compilation(
    circuit,
    pattern,
    trace=trace,
    output=str(common.FIG_PDF / "R8_multimode_showcase.pdf"),
)
fig1.savefig(common.FIG_PNG / "R8_multimode_showcase.png", bbox_inches="tight", dpi=300)
plt.close(fig1)

# =============================================================================
# Panel 2 (this notebook's addition): per-gate resource accounting, so every
# stored quantity in steps_rows/summary is also represented visually.
# =============================================================================

labels = [f"{s['source_gate']}{s['source_modes']}" for s in steps_rows]
x = np.arange(len(steps_rows))
width = 0.27

fig2, ax = plt.subplots(figsize=(9.0, 4.2), facecolor="white")
ax.bar(x - width, [s["resource_nodes"] for s in steps_rows], width, color="blue",
       label="Resource nodes")
ax.bar(x, [s["measurements"] for s in steps_rows], width, color="magenta",
       label="Measurements")
ax.bar(x + width, [s["corrections"] for s in steps_rows], width, color="cyan",
       label="Corrections")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=30, ha="right")
style_axis(ax, "Logical gate (source, modes)", "MBQC resource count",
           "Per-gate MBQC resource cost", "A")
ax.legend(frameon=False, loc="best")
fig2.suptitle(
    f"Five-mode showcase: {summary['source_gates']} gates -> "
    f"{summary['mbqc_nodes']} nodes, {summary['commands']} commands",
    fontsize=10, fontweight="bold", y=1.0,
)
fig2.tight_layout(rect=[0, 0, 1, 0.94])
common.save_figure(fig2, "R8_multimode_showcase_resource_accounting")
plt.close(fig2)

common.print_summary("R8 multimode showcase", **summary)


## R9 — Compilation Provenance / Trace Completeness

R9 is a software-correctness check, not a physics test: every command the
compiler emits must be attributable to exactly one logical source gate. This
matters because R3–R8's noise/error bookkeeping, and any future debugging of a
compiled circuit, implicitly assumes the `CompilationTrace` is a faithful,
gap-free record of what produced what.

### The partition invariant

Let the compiled pattern (same five-mode circuit as R8) have commands indexed
\(0,\dots,M-1\) (excluding the trailing `Output` command), and let
\(I_g\subset\{0,\dots,M-1\}\) be the command indices `CompilationStep`
attributes to logical gate \(g\). Provenance is complete and non-overlapping
iff the \(I_g\) **partition** the full index set:

$$
\bigcup_g I_g = \{0,1,\dots,M-1\}, \qquad I_g \cap I_{g'} = \varnothing \;\; (g\neq g').
$$

This is a discrete, exact combinatorial identity (a partition of a finite
index set), checked here by sorting the union of every `command_indices` tuple
and comparing it to `range(M)` element-for-element — no floating-point
tolerance is meaningful for this check.

### What is checked

For every gate in the five-mode showcase circuit (R8): its resource nodes,
input/output frontier nodes, command indices, entanglement edges,
measurements and corrections are exported to CSV/JSON, and the partition
identity above is asserted directly (`AssertionError` on any gap or overlap,
not a tolerance-based comparison).

### Success criteria

- The sorted union of all `command_indices` equals `range(len(pattern.commands) - 1)`
  exactly (panel A: a gap-free, non-overlapping timeline, one color per gate).
- Every gate's command/resource-node/measurement count is consistent with its
  gate family (rotation/squeeze: 4 steps; CZ: 1 edge + two 4-step wires;
  beamsplitter: the six-SUM construction from R7) — panel B.


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg

set_reference_style()

SQUEEZING = 1.0


def build_circuit() -> pg.Circuit:
    """Identical to R8's build_circuit (kept standalone per script)."""
    circuit = pg.Circuit(5)
    circuit.rotate(0, 0.4)
    circuit.squeeze(1, 0.3)
    circuit.rotate(2, -0.6)
    circuit.cz(0, 1, 0.8)
    circuit.displace(3, q=0.2, p=-0.1)
    circuit.beamsplitter(2, 3, 0.3)
    circuit.rotate(4, 0.5)
    circuit.cz(3, 4, -0.5)
    return circuit


circuit = build_circuit()
pattern, trace = circuit.compile(squeezing=SQUEEZING, return_trace=True)

rows = []
summary_lines = []
for step in trace.steps:
    rows.append(
        {
            "gate_index": step.gate_index,
            "source_gate": step.source_gate,
            "source_modes": step.source_modes,
            "source_parameters": step.source_parameters,
            "input_nodes": step.input_nodes,
            "output_nodes": step.output_nodes,
            "resource_nodes": step.resource_nodes,
            "n_resource_nodes": len(step.resource_nodes),
            "command_indices": step.command_indices,
            "n_commands": len(step.command_indices),
            "edges": step.edges,
            "n_edges": len(step.edges),
            "measurements": step.measurements,
            "n_measurements": len(step.measurements),
            "corrections": step.corrections,
            "n_corrections": len(step.corrections),
            "synthesis_report": None if step.synthesis_report is None else str(step.synthesis_report),
        }
    )
    summary_lines.append(
        f"Gate {step.gate_index:2d} [{step.source_gate}] modes={step.source_modes} "
        f"params={step.source_parameters} : {len(step.resource_nodes)} resource nodes, "
        f"{len(step.measurements)} measurements, {len(step.corrections)} corrections, "
        f"{len(step.command_indices)} commands "
        f"({step.input_nodes} -> {step.output_nodes})"
    )

# -----------------------------------------------------------------------------
# Partition invariant: union of I_g = {0,...,M-1}, pairwise disjoint.
# An exact combinatorial identity -- checked without floating-point tolerance.
# -----------------------------------------------------------------------------

all_indices = sorted(i for step in trace.steps for i in step.command_indices)
expected = list(range(len(pattern.commands) - 1))  # exclude trailing Output
if all_indices != expected:
    raise AssertionError("Compilation trace command indices do not partition the command list")

common.save_csv(rows, "R9_compilation_trace")
common.save_json({"steps": rows, "pattern_signature": trace.pattern_signature}, "R9_compilation_trace")
common.write_metadata("R9_compilation_trace")
common.save_raw("\n".join(summary_lines), "R9_compilation_trace_summary.txt")

# =============================================================================
# Final 1 x 2 publication figure
# =============================================================================

fig, (axA, axB) = plt.subplots(1, 2, figsize=(11.0, 4.2), facecolor="white")

# -----------------------------------------------------------------------------
# A. Gap-free, non-overlapping command-index timeline (the partition itself)
# -----------------------------------------------------------------------------

colors = plt.cm.tab10(np.linspace(0, 1, len(trace.steps)))
for i, step in enumerate(trace.steps):
    idx = step.command_indices
    if not idx:
        continue
    axA.barh(0, width=len(idx), left=min(idx), height=0.6, color=colors[i],
             edgecolor="white", linewidth=0.5,
             label=f"{step.gate_index}:{step.source_gate}{step.source_modes}")
axA.set_yticks([])
axA.set_xlim(0, len(pattern.commands) - 1)
style_axis(axA, "Command index", "", "Provenance partition (one color per gate)", "A")
axA.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.25),
           ncol=4, fontsize=6.5)

# -----------------------------------------------------------------------------
# B. Per-gate command/measurement/resource-node counts
# -----------------------------------------------------------------------------

labels = [f"{r['gate_index']}:{r['source_gate']}{r['source_modes']}" for r in rows]
x = np.arange(len(rows))
width = 0.27
axB.bar(x - width, [r["n_resource_nodes"] for r in rows], width, color="blue", label="Resource nodes")
axB.bar(x, [r["n_measurements"] for r in rows], width, color="magenta", label="Measurements")
axB.bar(x + width, [r["n_commands"] for r in rows], width, color="cyan", label="Commands")
axB.set_xticks(x)
axB.set_xticklabels(labels, rotation=30, ha="right")
style_axis(axB, "Logical gate", "Count", "Per-gate provenance counts", "B")
axB.legend(frameon=False, loc="best", fontsize=7)

fig.suptitle(
    f"Compilation provenance: {len(trace.steps)} gates, "
    f"{len(pattern.commands)} commands, partition verified",
    fontsize=10, fontweight="bold", y=1.03,
)
fig.tight_layout(rect=[0, 0, 1, 0.96])
common.save_figure(fig, "R9_compilation_trace")
plt.close(fig)

common.print_summary(
    "R9 compilation provenance",
    gates=len(trace.steps),
    total_commands=len(pattern.commands),
    total_resource_nodes=sum(len(s.resource_nodes) for s in trace.steps),
)


## R10 — Compiler Resource Scaling

R10 measures how the MBQC resource cost and wall-clock compile time of
repeated single-gate-family circuits grow with the number of gates, for the
four primitives validated individually in R3, R4, R6 and R7.

### Expected scaling

Each gate family lowers to a **fixed** number of teleportation steps per gate
(R3: 4 or 5 for a generic single-mode gate; R6: 1 entangling edge + two 4-step
identity wires per CZ; R7: six SUM gates, each three single-mode chains, per
beamsplitter). Compiling \(n\) *independent, non-interacting* gates therefore
has no algorithmic reason to do anything other than concatenate their MBQC
programs, so every resource count is expected to scale **linearly**,

$$
\text{resource\_nodes}(n),\;\text{measurements}(n),\;\text{commands}(n)
\;\approx\; c\cdot n,
$$

for a family-dependent constant \(c\) (the fixed per-gate cost). Compile
*time* is not guaranteed to be linear a priori (bookkeeping, list growth, and
Python-level overhead can add sub-linear or super-linear contributions), so it
is measured, not assumed, and fit separately.

This is a software-engineering/performance result — the metric is a runtime
measurement, not a physical prediction, so no analytic "truth" exists to
validate against.

### What is measured

For each gate family, wall-clock `Circuit.compile` time (warm-up + repeated
median, `common.benchmark`-style) and the resulting resource-node,
measurement, edge and command counts, at gate counts up to 100 (20 for the
markedly more expensive beamsplitter, R7).

### What a power-law fit means here

A least-squares fit of \(\log(\text{metric})\) vs. \(\log(n)\) gives an
**empirical** exponent \(p\) in \(\text{metric}(n)\approx a\,n^{p}\) for the
*sampled range only* — this is reported as an observation, not asserted as a
proven asymptotic complexity class.

### Success criteria

- Resource-count exponents \(p\approx 1\) for every gate family (panel
  fits, top row) — confirms linear, non-interacting compilation as expected
  from the fixed per-gate teleportation cost.
- Compile-time growth is reported with its own fitted exponent per family
  (bottom-right panel), without claiming a formal complexity bound.


In [ ]:
from __future__ import annotations

import time

import numpy as np

import photographiq as pg

set_reference_style()

SQUEEZING = 1.0
GATE_COUNTS = [1, 2, 5, 10, 20, 50, 100]
BEAMSPLITTER_COUNTS = [1, 2, 5, 10, 20]
GATE_KINDS = ("rotation", "squeeze", "cz", "beamsplitter")


def build(kind: str, n: int) -> pg.Circuit:
    if kind in ("rotation", "squeeze"):
        circuit = pg.Circuit(1)
        for i in range(n):
            if kind == "rotation":
                circuit.rotate(0, 0.1 + 0.01 * i)
            else:
                circuit.squeeze(0, 0.05 + 0.005 * i)
        return circuit
    circuit = pg.Circuit(2)
    for i in range(n):
        if kind == "cz":
            circuit.cz(0, 1, 0.3 + 0.01 * i)
        else:
            circuit.beamsplitter(0, 1, 0.2 + 0.01 * i)
    return circuit


def measure_compile(circuit):
    repeats = 3 if len(circuit.gates) <= 20 else 1
    times = []
    result = None
    for _ in range(repeats):
        start = time.perf_counter()
        result = circuit.compile(squeezing=SQUEEZING, return_trace=True)
        times.append(time.perf_counter() - start)
    return result, times


def fitted_exponent(n, y):
    """Least-squares slope p of log(y) vs log(n): empirical y ~ a n^p."""
    mask = (np.asarray(n) > 0) & (np.asarray(y) > 0)
    if mask.sum() < 2:
        return float("nan")
    slope, _ = np.polyfit(np.log(np.asarray(n)[mask]), np.log(np.asarray(y)[mask]), 1)
    return float(slope)


all_rows = []
for kind, counts in (
    ("rotation", GATE_COUNTS),
    ("squeeze", GATE_COUNTS),
    ("cz", GATE_COUNTS),
    ("beamsplitter", BEAMSPLITTER_COUNTS),
):
    for n in counts:
        circuit = build(kind, n)
        (pattern, trace), times = measure_compile(circuit)
        entangle_edges = sum(1 for c in pattern.commands if isinstance(c, pg.Entangle))
        measurements = sum(1 for c in pattern.commands if isinstance(c, pg.Measure))
        row = {
            "gate_kind": kind,
            "gate_count": n,
            "resource_nodes": len(pattern.graph.nodes) - circuit.modes,
            "measurements": measurements,
            "edges": entangle_edges,
            "commands": len(pattern.commands),
            "compile_time_median_seconds": float(np.median(times)) if len(times) > 1 else times[0],
            "compile_time_seconds_raw": times,
        }
        all_rows.append(row)
        print(f"  {kind} n={n}: {row['resource_nodes']} nodes, {row['compile_time_median_seconds']:.4f}s", flush=True)

# -----------------------------------------------------------------------------
# Empirical scaling exponents (observation over the sampled range, not a
# proof of asymptotic complexity)
# -----------------------------------------------------------------------------

exponent_rows = []
for kind in GATE_KINDS:
    rows = [r for r in all_rows if r["gate_kind"] == kind]
    ns = [r["gate_count"] for r in rows]
    exponent_rows.append(
        {
            "gate_kind": kind,
            "resource_nodes_exponent": fitted_exponent(ns, [r["resource_nodes"] for r in rows]),
            "measurements_exponent": fitted_exponent(ns, [r["measurements"] for r in rows]),
            "commands_exponent": fitted_exponent(ns, [r["commands"] for r in rows]),
            "compile_time_exponent": fitted_exponent(ns, [r["compile_time_median_seconds"] for r in rows]),
        }
    )

common.save_result(all_rows, "R10_compiler_scaling")
common.save_csv(exponent_rows, "R10_compiler_scaling_exponents")

# =============================================================================
# Final 2 x 2 publication figure
# =============================================================================

metrics = [
    ("resource_nodes", "Resource nodes"),
    ("measurements", "Measurements"),
    ("commands", "Commands"),
    ("compile_time_median_seconds", "Compile time (s)"),
]
panel_labels = ["A", "B", "C", "D"]
colors = {"rotation": "red", "squeeze": "blue", "cz": "magenta", "beamsplitter": "cyan"}
markers = {"rotation": "o", "squeeze": "s", "cz": "^", "beamsplitter": "d"}

fig, axes = plt.subplots(2, 2, figsize=(9.6, 7.4), facecolor="white")

for ax, (key, label), letter in zip(axes.flat, metrics, panel_labels, strict=True):
    for kind in GATE_KINDS:
        rows = [r for r in all_rows if r["gate_kind"] == kind]
        ns = [r["gate_count"] for r in rows]
        ys = [r[key] for r in rows]
        exponent = next(e[f"{key.replace('compile_time_median_seconds', 'compile_time')}_exponent"]
                         for e in exponent_rows if e["gate_kind"] == kind)
        ax.loglog(ns, ys, markers[kind] + "-", color=colors[kind], markersize=4,
                   label=f"{kind} ($p$={exponent:.2f})")
    ax.set_xscale("log")
    ax.set_yscale("log")
    style_axis(ax, "Number of gates $n$", label, label, letter)

axes.flat[0].legend(frameon=False, fontsize=6.5, loc="best")

fig.suptitle("Compiler resource and runtime scaling (empirical log-log fits)",
             fontsize=11, fontweight="bold", y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.96], h_pad=2.2, w_pad=2.2)
common.save_figure(fig, "R10_compiler_scaling")
plt.close(fig)

common.print_summary("R10 compiler resource scaling", rows=len(all_rows))


## R11 — Adaptive Feed-Forward (Outcome-Dependent Measurement Angle)

Every gate compiled in R3–R10 uses **fixed** teleportation angles: the shear
\(k_j\) is a compile-time number, and only the final *displacement correction*
depends on measurement outcomes (`Outcome("m")` in a `Displace` command). R11
tests something structurally stronger: a pattern whose **measurement basis
itself** depends on an earlier outcome — genuine mid-circuit adaptivity, the
primitive behind conditional/non-Clifford CV-MBQC gates.

### The pattern

On a three-node line resource \(0-1-2\) (input \(0\), output \(2\)):

$$
m_0 = \text{homodyne}_q(0),
\qquad
\theta_1(m_0) = 0.05\,m_0 + \theta,
\qquad
m_1 = \text{homodyne}_{\theta_1(m_0)}(1),
$$

with the usual displacement corrections \(D_1(-m_0)\), \(D_2(-m_1)\). Because
\(\theta_1\) is a random variable (a function of the first Gaussian-distributed
outcome \(m_0\)), the transformation applied to mode 1 is not one fixed linear
map — it is an outcome-conditioned **family** of maps
\(\{T(k;\theta_1(m_0))\}\), selected after \(m_0\) is known.

### Why this is genuinely adaptive, not just corrected

A non-adaptive teleportation step (R1–R10) has a fixed measurement angle, so
its *unconditional* action is exactly one Gaussian channel \((S,N,d)\) —
outcome-dependence enters only through \(d\), which averages out in the
ensemble mean/covariance (law of total covariance, R1). Here the measurement
**generator** itself depends on the outcome, so no single \((S,N,d)\) can
represent the map for every trajectory: `gaussian_channel` (`analysis.py`)
explicitly raises `NotImplementedError` for outcome-dependent gate angles
(`docs/theory.md`). This is used here as an **operational, structural test**:
if the analyzer accepted the pattern, it would prove the "adaptive" step was
secretly non-adaptive.

### What is checked

1. The command dependency graph (`Pattern.dependencies()`) has a directed edge
   from \(m_0\)'s `Measure` command to \(m_1\)'s (its angle expression reads
   `Outcome("m0")`).
2. `gaussian_channel(pattern)` raises `NotImplementedError` (rejection is the
   expected, correct behavior here — not a bug).
3. Across several trajectory seeds, the *realized* angle
   \(\theta_1=0.05\,m_0+\theta\) actually used by each shot is recovered from
   the recorded outcome and compared with the output state's own quadrature at
   that same angle, closing the loop between "what angle was supposed to be
   used" and "what the simulator actually measured in."

### Success criteria

- The dependency edge \(m_0\to m_1\) exists (panel: dependency graph).
- The affine analyzer rejects the pattern (recorded, not silently bypassed).
- For every seed, the output state's quadrature at the *realized* adaptive
  angle matches the simulator's own report at that angle to machine precision
  — confirming the adaptive angle was genuinely applied per-shot, and that
  different outcomes \(m_0\) really do steer different measurement bases
  (panel: realized angle vs. \(m_0\), and output means across seeds).


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg
from photographiq.visualization import draw_dependencies, draw_pattern

set_reference_style()

SQUEEZING = 1.0
THETA_PARAMETER = 0.3
SEEDS = [1, 2, 3, 4, 5, 6, 7, 8]


def build_pattern():
    graph = pg.CVGraph.line(3, squeezing=SQUEEZING, inputs=(0,))
    pattern = pg.Pattern(graph)
    pattern.measure(0, pg.Homodyne.q(), key="m0")
    pattern.displace(1, q=-pg.Outcome("m0"))
    # Adaptive: the homodyne angle itself is a function of the earlier outcome.
    #
    # theta_1(m0) = 0.05 m0 + theta
    pattern.measure(1, pg.Homodyne(angle=0.05 * pg.Outcome("m0") + pg.Parameter("theta")), key="m1")
    pattern.displace(2, q=-pg.Outcome("m1"))
    pattern.append(pg.Output((2,)))
    return pattern.validate()


pattern = build_pattern()

dependency_graph = pattern.dependencies()
m0_index = next(i for i, c in enumerate(pattern.commands) if isinstance(c, pg.Measure) and c.result_key == "m0")
m1_index = next(i for i, c in enumerate(pattern.commands) if isinstance(c, pg.Measure) and c.result_key == "m1")
has_dependency_edge = dependency_graph.has_edge(m0_index, m1_index)
if not has_dependency_edge:
    raise AssertionError("Expected a causal dependency edge from m0's measurement to m1's")

# -----------------------------------------------------------------------------
# Structural proof of genuine adaptivity: an outcome-dependent measurement
# GENERATOR cannot be written as one fixed (S, N, d) Gaussian channel, so the
# affine analyzer must reject it (docs/theory.md). Acceptance here would mean
# the "adaptive" step was secretly non-adaptive.
# -----------------------------------------------------------------------------

rejected_by_affine_analyzer = False
try:
    pg.gaussian_channel(pattern)
except NotImplementedError:
    rejected_by_affine_analyzer = True
if not rejected_by_affine_analyzer:
    raise AssertionError("Expected the affine Gaussian-channel analyzer to reject this adaptive pattern")

rows = []
input_state = pg.GaussianInput.coherent(0.3 + 0.1j)
for seed in SEEDS:
    result = pg.simulate(
        pattern, backend="gaussian", inputs={0: input_state}, parameters={"theta": THETA_PARAMETER}, seed=seed
    )
    # Realized adaptive angle for THIS trajectory's outcome m0:
    #
    # theta_1 = 0.05 m0 + theta
    expected_angle = 0.05 * result.records["m0"] + THETA_PARAMETER
    q_at_expected_angle, _ = result.state.quadrature(2, expected_angle)
    rows.append(
        {
            "seed": seed,
            "m0_outcome": result.records["m0"],
            "m1_outcome": result.records["m1"],
            "adaptive_angle_used": expected_angle,
            "output_q_mean": result.state.quadrature(2)[0],
            "output_p_mean": result.state.quadrature(2, expected_angle)[0],
        }
    )

common.save_result(
    rows,
    "R11_adaptive_feedforward",
    extra={
        "pattern_json": pattern.to_json(),
        "dependency_edges": sorted(dependency_graph.edges()),
        "m0_command_index": m0_index,
        "m1_command_index": m1_index,
        "affine_analyzer_rejected_pattern": rejected_by_affine_analyzer,
        "theta_parameter": THETA_PARAMETER,
    },
)
common.save_raw(pattern.to_json(), "R11_adaptive_feedforward_pattern.json")

# =============================================================================
# Panel 1: the package's own pattern/dependency diagrams
# =============================================================================

ax = draw_pattern(pattern)
common.save_figure(ax.figure, "R11_adaptive_feedforward_pattern")
plt.close(ax.figure)

ax = draw_dependencies(pattern)
common.save_figure(ax.figure, "R11_adaptive_feedforward_dependencies")
plt.close(ax.figure)

# =============================================================================
# Panel 2 (this notebook's addition): the numeric per-seed record, so every
# stored quantity in `rows` is also represented visually.
# =============================================================================

m0_vals = np.array([r["m0_outcome"] for r in rows])
angle_vals = np.array([r["adaptive_angle_used"] for r in rows])
q_vals = np.array([r["output_q_mean"] for r in rows])
p_vals = np.array([r["output_p_mean"] for r in rows])

fig, (axA, axB) = plt.subplots(1, 2, figsize=(9.6, 4.2), facecolor="white")

order = np.argsort(m0_vals)
axA.plot(m0_vals[order], angle_vals[order], "o-", color="red", label="Realized $\\theta_1$")
axA.plot(m0_vals[order], 0.05 * m0_vals[order] + THETA_PARAMETER, "--", color="black",
         linewidth=1.0, label=r"$0.05\,m_0+\theta$")
style_axis(axA, "Outcome $m_0$", r"Adaptive angle $\theta_1$",
           "Measurement basis tracks the outcome", "A")
axA.legend(frameon=False, loc="best")

sc = axB.scatter(q_vals, p_vals, c=m0_vals, cmap="coolwarm", s=45, edgecolors="black", linewidths=0.4)
cbar = fig.colorbar(sc, ax=axB)
cbar.set_label("Outcome $m_0$")
style_axis(axB, r"$\langle q_{\rm out}\rangle$", rf"$\langle p_{{\rm out}}\rangle$ (at $\theta_1$)",
           "Output per adaptive trajectory", "B")

fig.suptitle("Adaptive feed-forward: outcome-dependent measurement basis",
             fontsize=10, fontweight="bold", y=1.02)
fig.tight_layout(rect=[0, 0, 1, 0.94])
common.save_figure(fig, "R11_adaptive_feedforward_summary")
plt.close(fig)

common.print_summary(
    "R11 adaptive feed-forward",
    seeds=len(rows),
    dependency_edge_m0_to_m1=has_dependency_edge,
    affine_analyzer_rejected_pattern=rejected_by_affine_analyzer,
)


## R12 — Dependency DAG and Causal Schedule

R12 is a structural/software-correctness check on the classical control-flow
layer that makes adaptive patterns like R11 well-defined at all: if outcome
\(m_0\) feeds a correction or measurement angle downstream, the simulator must
guarantee \(m_0\) is *known* before that command executes.

### The invariant

`Pattern.dependencies()` builds a directed graph \(D=(V,E)\) over command
indices: an edge \(u\to v\) means command \(v\) reads a classical value
(`Outcome(key)`) produced by command \(u\), or that both commands touch the
same quantum mode in sequence (`flow.dependency_graph`, `docs/theory.md`).
`Pattern.schedule()` returns one topological ordering \(\pi\) of \(V\)
consistent with \(D\). Correctness requires

$$
\forall (u,v)\in E:\quad \pi(u) < \pi(v),
$$

i.e. every dependency points strictly forward in the executed order — the
same partial-order requirement any dataflow scheduler must satisfy, here
applied to a four-node pattern where node 2's correction depends on **two**
different earlier outcomes simultaneously (\(m_0\) and \(m_1\)):

$$
D_2 = -m_1,\qquad P_2 = 0.2\,m_0,\qquad
D_3 = -m_2,\qquad P_3 = -m_0+m_1.
$$

### What is checked

1. Extract \(D\) and \(\pi\) for this multi-dependency pattern.
2. For every edge \((u,v)\in E\), verify \(\pi(u)<\pi(v)\) directly (a
   discrete order comparison, not a numerical tolerance).

### Success criteria

- Zero backward violations: the schedule is a valid topological order of the
  dependency DAG (panel A: DAG drawing; panel B: every edge lies above the
  diagonal when plotted as (source position, target position)).


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg
from photographiq.visualization import draw_dependencies

set_reference_style()

SQUEEZING = 1.0


def build_pattern():
    graph = pg.CVGraph.line(4, squeezing=SQUEEZING, inputs=(0,))
    pattern = pg.Pattern(graph)
    pattern.measure(0, pg.Homodyne.q(), key="m0")
    pattern.displace(1, q=-pg.Outcome("m0"))
    pattern.measure(1, pg.Homodyne(angle=0.1 * pg.Outcome("m0")), key="m1")
    # Node 2's correction depends on BOTH earlier outcomes.
    pattern.displace(2, q=-pg.Outcome("m1"), p=0.2 * pg.Outcome("m0"))
    pattern.measure(2, pg.Homodyne.p(), key="m2")
    pattern.displace(3, q=-pg.Outcome("m2"), p=-pg.Outcome("m0") + pg.Outcome("m1"))
    pattern.append(pg.Output((3,)))
    return pattern.validate()


pattern = build_pattern()
dag = pattern.dependencies()
schedule = pattern.schedule()

position = {command_index: position for position, command_index in enumerate(schedule)}
edges = sorted(dag.edges())

# Discrete order check: pi(u) < pi(v) for every dependency edge (u, v).
# Not a numerical tolerance -- a strict integer comparison.
violations = [(u, v) for u, v in edges if position[u] >= position[v]]

rows = [
    {
        "command_index": i,
        "command_type": type(c).__name__,
        "schedule_position": position[i],
        "dependencies": sorted(dag.predecessors(i)),
    }
    for i, c in enumerate(pattern.commands)
]

common.save_result(
    rows,
    "R12_dependency_dag",
    extra={
        "dependency_edges": edges,
        "topological_schedule": list(schedule),
        "backward_violations": violations,
        "pattern_json": pattern.to_json(),
    },
)

if violations:
    common.save_json(violations, "R12_dependency_dag_VIOLATIONS")
    raise AssertionError(f"{len(violations)} dependency edges violate the topological schedule")

# =============================================================================
# Final 1 x 2 publication figure
# =============================================================================

fig, (axA, axB) = plt.subplots(1, 2, figsize=(10.0, 4.4), facecolor="white")

# -----------------------------------------------------------------------------
# A. The package's own dependency-graph drawing
# -----------------------------------------------------------------------------

draw_dependencies(pattern, ax=axA)
axA.text(-0.08, 1.05, "A", transform=axA.transAxes, fontsize=11, fontweight="bold", va="top")

# -----------------------------------------------------------------------------
# B. Every dependency edge must lie strictly above the diagonal in schedule
#    position -- the visual form of pi(u) < pi(v).
# -----------------------------------------------------------------------------

src_pos = [position[u] for u, v in edges]
dst_pos = [position[v] for u, v in edges]
axB.plot([0, len(pattern.commands)], [0, len(pattern.commands)], "--", color="black",
         linewidth=0.8, label=r"$\pi(u)=\pi(v)$")
axB.scatter(src_pos, dst_pos, color="red", s=35, zorder=3, label="Dependency edges")
style_axis(axB, r"Schedule position $\pi(u)$ (source)", r"Schedule position $\pi(v)$ (target)",
           "Every edge points forward in the schedule", "B")
axB.legend(frameon=False, loc="best")

fig.suptitle("Dependency DAG and topological schedule (0 backward violations)",
             fontsize=10, fontweight="bold", y=1.02)
fig.tight_layout(rect=[0, 0, 1, 0.94])
common.save_figure(fig, "R12_dependency_dag")
plt.close(fig)

common.print_summary(
    "R12 dependency DAG / causal schedule",
    commands=len(pattern.commands),
    dependency_edges=len(edges),
    backward_violations=len(violations),
)


## R13 — Invalid Causality / Software-Safety Audit

R13 is a software-correctness audit, not a physics experiment: it checks that
`Pattern.validate()` (or construction itself) actually **rejects** every
structurally invalid pattern it documents rejecting, with the real validator
exception — not a hand-waved assumption that "the checks work."

### The invariants under test

Twelve independent invalid-construction classes, each violating one discrete
structural or causal invariant a valid `Pattern` must satisfy:

| Invariant | Violated by |
|---|---|
| Inputs are distinct nodes | `duplicate_inputs` |
| A node is prepared at most once | `duplicate_resource_node` |
| An `Entangle` has two distinct endpoints | `self_cz` |
| `Output` is the pattern's final command | `output_not_final` |
| `Output` lists each node once | `duplicate_outputs` |
| A node is used only before being destructively measured | `missing_or_measured_node` |
| Every `Outcome(key)` refers to an **already-produced** classical value | `future_outcome_dependency` |
| The command dependency graph is a DAG (R12) | `cyclic_dependency` |
| Every `Outcome(key)` refers to a key some `Measure` actually defines | `undefined_classical_key` |
| Classical result keys are unique | `duplicate_classical_key` |
| A `Measure` description is a supported measurement type | `unsupported_measurement_description` |
| A `Pattern` built directly from an **ideal** (\(r=\infty\)) resource graph | `ideal_graph_pattern` |

Each is a discrete, exact structural check — satisfied or violated, no
numerical tolerance involved.

### A known, harmless imprecision

Building `dependency_graph` (used by `validate()`) does not deduplicate
repeated node references within one command, so `Entangle(0, 0, ...)` and
`Output((0, 0))` both create a **self-loop**, which the earlier acyclicity
check reports generically as `"Cyclic command dependencies"` rather than the
more specific `"Self CZ is invalid"` / `"Duplicate outputs"` messages the
validator also implements. **The pattern is still correctly rejected** in both
cases (`self_cz`, `duplicate_outputs` below) — this is an imprecise
diagnostic message for this particular trigger shape, not a missed rejection
(see `ISSUES_FOUND.md` #1 for the full trace).

### Success criteria

- All 12 cases raise (`ValueError`, `NotImplementedError` or `TypeError`);
  zero silently succeed (panel: status grid, green = rejected as expected).
- The exception actually raised is recorded and categorized per case, so the
  imprecise-message cases above are visible rather than hidden (panel:
  exception type per case).


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg
from photographiq.commands import Displace, Entangle, Measure, Output, Prepare
from photographiq.expressions import Outcome
from photographiq.measurements import Homodyne

set_reference_style()


def case_duplicate_inputs():
    return pg.Pattern(inputs=(0, 0)).validate()


def case_duplicate_resource_node():
    graph = pg.CVGraph.line(2, inputs=(0,))
    pattern = pg.Pattern(graph)
    pattern.append(Prepare(1, 1.0))  # node 1 already prepared by the graph
    return pattern.validate()


def case_self_cz():
    pattern = pg.Pattern(inputs=(0,))
    pattern.append(Entangle(0, 0, 1.0))
    return pattern.validate()


def case_output_not_final():
    pattern = pg.Pattern(inputs=(0,))
    pattern.append(Output((0,)))
    pattern.append(Displace(0, q=1.0))
    return pattern.validate()


def case_duplicate_outputs():
    pattern = pg.Pattern(inputs=(0,))
    pattern.append(Output((0, 0)))
    return pattern.validate()


def case_missing_or_measured_node():
    pattern = pg.Pattern(inputs=(0,))
    pattern.measure(0, Homodyne.q(), key="m0")
    pattern.append(Displace(0, q=1.0))  # 0 was destructively measured
    return pattern.validate()


def case_future_outcome_dependency():
    pattern = pg.Pattern(inputs=(0, 1))
    pattern.append(Displace(0, q=Outcome("future")))  # "future" not yet produced
    pattern.append(Measure(1, Homodyne.q(), "future"))
    return pattern.validate()


def case_cyclic_dependency():
    pattern = pg.Pattern(inputs=(0, 1))
    pattern.append(Measure(0, Homodyne(angle=Outcome("m1")), "m0"))
    pattern.append(Measure(1, Homodyne(angle=Outcome("m0")), "m1"))
    return pattern.validate()


def case_undefined_classical_key():
    pattern = pg.Pattern(inputs=(0,))
    pattern.append(Displace(0, q=Outcome("nonexistent")))
    pattern.append(Output((0,)))
    return pattern.validate()


def case_duplicate_classical_key():
    pattern = pg.Pattern(inputs=(0, 1))
    pattern.append(Measure(0, Homodyne.q(), "m"))
    pattern.append(Measure(1, Homodyne.q(), "m"))  # same key twice
    return pattern.validate()


def case_unsupported_measurement_description():
    pattern = pg.Pattern(inputs=(0,))
    pattern.append(Measure(0, "not a measurement", "m"))
    return pattern.validate()


def case_ideal_graph_pattern():
    graph = pg.CVGraph.line(2, inputs=(0,), ideal=True)
    return pg.Pattern(graph)


CASES = [
    ("duplicate_inputs", case_duplicate_inputs),
    ("duplicate_resource_node", case_duplicate_resource_node),
    ("self_cz", case_self_cz),
    ("output_not_final", case_output_not_final),
    ("duplicate_outputs", case_duplicate_outputs),
    ("missing_or_measured_node", case_missing_or_measured_node),
    ("future_outcome_dependency", case_future_outcome_dependency),
    ("cyclic_dependency", case_cyclic_dependency),
    ("undefined_classical_key", case_undefined_classical_key),
    ("duplicate_classical_key", case_duplicate_classical_key),
    ("unsupported_measurement_description", case_unsupported_measurement_description),
    ("ideal_graph_pattern", case_ideal_graph_pattern),
]

# Cases where the raised message is documented to be a generic "Cyclic
# command dependencies" instead of the more specific diagnostic, because
# dependency_graph does not deduplicate repeated node references within one
# command (ISSUES_FOUND.md #1). Still correctly rejected either way.
IMPRECISE_MESSAGE_CASES = {"self_cz", "duplicate_outputs"}

rows = []
unexpected = []
for name, builder in CASES:
    try:
        builder()
        rows.append({"case": name, "raised": False, "exception_type": None, "message": None})
        unexpected.append(name)
    except (ValueError, NotImplementedError, TypeError) as exc:
        rows.append(
            {
                "case": name,
                "raised": True,
                "exception_type": type(exc).__name__,
                "message": str(exc),
                "imprecise_message_expected": name in IMPRECISE_MESSAGE_CASES,
            }
        )

common.save_result(rows, "R13_invalid_causality")

if unexpected:
    raise AssertionError(f"Expected these invalid patterns to be rejected but they were not: {unexpected}")

# =============================================================================
# Final figure: compliance status grid (software-safety audit, not a physics
# result -- a categorical status table is the right visualization here)
# =============================================================================

exception_types = sorted({r["exception_type"] for r in rows if r["exception_type"]})
type_color = {"ValueError": "#2c6e91", "NotImplementedError": "#8e2c6e", "TypeError": "#6e8e2c"}

fig, ax = plt.subplots(figsize=(7.6, 5.4), facecolor="white")
y = np.arange(len(rows))[::-1]

for yi, row in zip(y, rows):
    color = type_color.get(row["exception_type"], "gray")
    ax.barh(yi, 1.0, color=color, edgecolor="white", height=0.7)
    flag = " (generic message, see ISSUES_FOUND.md #1)" if row.get("imprecise_message_expected") else ""
    ax.text(1.02, yi, f"{row['exception_type']}{flag}", va="center", fontsize=7)

ax.set_yticks(y)
ax.set_yticklabels([r["case"] for r in rows], fontsize=8)
ax.set_xlim(0, 1)
ax.set_xticks([])
style_axis(ax, "", "", "All 12 invalid patterns correctly rejected", "A")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.spines["bottom"].set_visible(False)

handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in type_color.values()]
ax.legend(handles, type_color.keys(), frameon=False, loc="lower right", fontsize=7)

fig.suptitle("R13: invalid-pattern rejection audit", fontsize=10, fontweight="bold", y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.96])
common.save_figure(fig, "R13_invalid_causality")
plt.close(fig)

common.print_summary(
    "R13 invalid causality / software safety",
    cases=len(rows),
    all_rejected=len(unexpected) == 0,
)


## R14 — CV-Flow Certification

R14 tests `photographiq.cvflow.certify_cv_flow`, an implementation of the
real-linear CV-flow condition of Booth and Markham (*Quantum* **7**, 1146,
2023) for one **given** measurement order. It does not search over orders —
this experiment shows certification/rejection for two concrete orders on the
same resource, not a general flow-finding claim (`MANUSCRIPT_MAP.md`).

### The certification condition

For a resource graph with adjacency \(A\), inputs \(I\), and a proposed total
measurement order, let \(P(j)\) be the set of nodes measured up to and
including step \(j\), and let \(F(j)=V\setminus(P(j)\cup I)\) be the
"future" nodes (unmeasured, non-input) at that step. A correction vector
\(c_j\) exists exactly when the linear system

$$
A\big[P(j),\,F(j)\big]\,c_j = e_j
$$

has an exact real solution (\(e_j\) the standard basis vector singling out
node \(j\)) — i.e. the *already-measured* nodes' adjacency into the *not-yet
determined* part of the graph can be combined to reproduce, node \(j\)'s
own row exactly. A valid order admits such a \(c_j\) for every measured node;
`certify_cv_flow` returns the corrections and residuals when it succeeds, and
`None` when no exact solution exists.

### Two orders, same three-node line graph

Line graph \(0-1-2\) (input \(0\), output \(2\)):

- **Valid order** \((0,1)\): the natural teleportation direction. A
  certificate should exist with (near-)zero residual.
- **Invalid order** \((1,0)\): measuring node 1 before node 0. Node 1's only
  neighbors are 0 and 2; once node 0 is excluded (already used as an input,
  not available to supply future influence) and node 2 is the surviving
  output, no real linear combination of the single remaining row can
  reproduce the required unit vector — `certify_cv_flow` must return `None`.

This experiment recomputes the *same* linear system independently
(`numpy.linalg.lstsq`) to report an explicit least-squares residual for the
invalid order, rather than treating `None` as a black box.

### What is checked

1. `certify_cv_flow(graph, (0,1))` returns a certificate (not `None`).
2. `certify_cv_flow(graph, (1,0))` returns `None`.
3. The independent least-squares residual for order \((1,0)\) is genuinely
   nonzero (quantifying *how much* the invalid order fails, not just that it
   fails).
4. The certified order's `flow_pattern` is tied to an operationally checkable
   simulation: its **unconditional** \(M=500\)-shot ensemble mean/covariance
   (law of total covariance, R1) is compared against the independent affine
   `gaussian_channel` analyzer. A **single** conditional trajectory is also
   recorded separately — it is one random homodyne realization and is not
   expected to match the unconditional prediction closely (R1's
   conditional/unconditional distinction applies here too).

### Success criteria

- Valid order: certificate found, residuals at the numerical noise floor.
- Invalid order: no certificate, and the independent residual is
  \(O(1)\) — a genuine, not marginal, obstruction.
- The \(M=500\)-shot ensemble mean/covariance matches the analytic channel
  within sampling tolerance (\(\sim 5/\sqrt{M}\)); the single trajectory may
  deviate substantially more (panel B) — this gap is the expected
  conditional-vs-unconditional signature, not a discrepancy.


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg
from photographiq.cvflow import certify_cv_flow, flow_pattern

set_reference_style()

SQUEEZING = 1.0

graph = pg.CVGraph.line(3, squeezing=SQUEEZING, inputs=(0,), outputs=(2,))

valid_order = (0, 1)
valid_certificate = certify_cv_flow(graph, valid_order)
if valid_certificate is None:
    raise AssertionError("Expected order (0,1) to admit a CV-flow certificate")

invalid_order = (1, 0)
invalid_certificate = certify_cv_flow(graph, invalid_order)
if invalid_certificate is not None:
    raise AssertionError("Expected order (1,0) to have no CV-flow certificate")

# -----------------------------------------------------------------------------
# Diagnose *why* the invalid order fails: recompute the same linear system
# certify_cv_flow solves internally,
#
#   A[P(j), F(j)] c_j = e_j
#
# to report an explicit residual rather than treating None as a black box.
# -----------------------------------------------------------------------------

a = graph.adjacency()
indices = {n: i for i, n in enumerate(graph.nodes)}
past = invalid_order[:2]
future = tuple(n for n in graph.nodes if n not in past and n not in graph.inputs)
matrix = a[np.ix_([indices[n] for n in past], [indices[n] for n in future])]
target = np.zeros(len(past))
target[-1] = 1
solution, *_ = np.linalg.lstsq(matrix, target, rcond=None)
residual = float(np.linalg.norm(matrix @ solution - target, ord=np.inf))

# For reference, the valid order's own certified residual (should be ~0).
valid_residual = max((abs(v) for v in valid_certificate.residuals), default=0.0)

pattern = flow_pattern(graph, valid_order)
input_state = pg.GaussianInput.coherent(0.3 + 0.1j)
ideal_channel = pg.gaussian_channel(pattern)
ideal = ideal_channel.apply(input_state.state(0))

# One conditional trajectory is a single random homodyne realization, not the
# unconditional channel prediction (R1): comparing it to `ideal` directly
# need only agree within stochastic (not machine-precision) tolerance.
SHOTS = 500
SEED = 1
single = pg.simulate(pattern, backend="gaussian", inputs={0: input_state}, seed=SEED)
single_mean_error = common.frobenius_error(single.state.mean, ideal.mean)
single_covariance_error = common.frobenius_error(single.state.covariance, ideal.covariance)

# The genuine unconditional check averages over many trajectories (law of
# total covariance, R1):
#
# V_ensemble = E[V_j] + Cov(mu_j)
shots = pg.run_shots(pattern, SHOTS, backend="gaussian", inputs={0: input_state}, seed=SEED)
ensemble = shots.ensemble_state()
mean_error = common.frobenius_error(ensemble.mean, ideal.mean)
covariance_error = common.frobenius_error(ensemble.covariance, ideal.covariance)

# Sampling-justified tolerance for a Monte Carlo ensemble mean/covariance
# (not the tight algebraic tolerance used for exact identities elsewhere):
# ~1/sqrt(SHOTS) statistical fluctuation for O(1) quadrature variances.
ENSEMBLE_TOL = 5.0 / np.sqrt(SHOTS)
if mean_error > ENSEMBLE_TOL or covariance_error > ENSEMBLE_TOL:
    raise AssertionError(
        f"flow_pattern ensemble disagreed with the affine channel beyond "
        f"sampling tolerance {ENSEMBLE_TOL:.3f}: mean_error={mean_error:.3f}, "
        f"covariance_error={covariance_error:.3f}"
    )

summary = {
    "graph": "line(3), inputs=(0,), outputs=(2,)",
    "valid_order": valid_order,
    "valid_certificate_corrections": {str(k): v for k, v in valid_certificate.corrections.items()},
    "valid_certificate_residuals": valid_certificate.residuals,
    "valid_certificate_max_residual": valid_residual,
    "invalid_order": invalid_order,
    "invalid_certificate_result": None,
    "invalid_order_diagnostic_residual": residual,
    "invalid_order_diagnostic_solution": solution.tolist(),
    "flow_pattern_commands": len(pattern.commands),
    "shots": SHOTS,
    "single_trajectory_mean": single.state.mean.tolist(),
    "single_trajectory_covariance": single.state.covariance.tolist(),
    "single_trajectory_mean_error": single_mean_error,
    "single_trajectory_covariance_error": single_covariance_error,
    "ensemble_mean": ensemble.mean.tolist(),
    "ensemble_covariance": ensemble.covariance.tolist(),
    "affine_channel_mean": ideal.mean.tolist(),
    "affine_channel_covariance": ideal.covariance.tolist(),
    "flow_vs_channel_mean_error": mean_error,
    "flow_vs_channel_covariance_error": covariance_error,
}
common.save_json(summary, "R14_cvflow")
common.write_metadata("R14_cvflow")
common.save_raw(pattern.to_json(), "R14_cvflow_pattern.json")

# =============================================================================
# Final 1 x 2 publication figure
# =============================================================================

fig, (axA, axB) = plt.subplots(1, 2, figsize=(9.0, 4.2), facecolor="white")
floor = 1e-16

# -----------------------------------------------------------------------------
# A. Certification residuals: valid order ~ 0, invalid order O(1)
# -----------------------------------------------------------------------------

labels = [f"valid {valid_order}", f"invalid {invalid_order}"]
values = [max(valid_residual, floor), max(residual, floor)]
colors = ["blue", "red"]
axA.bar(labels, values, color=colors, alpha=0.85)
axA.set_yscale("log")
axA.axhline(1e-8, color="black", linestyle="--", linewidth=0.8, label="Numerical floor")
style_axis(axA, "Measurement order", "Certification residual",
           "CV-flow certificate exists only for the valid order", "A")
axA.legend(frameon=False, loc="best")

# -----------------------------------------------------------------------------
# B. flow_pattern vs. the affine channel: a SINGLE conditional trajectory
# need not match (R1); the M=500-shot ENSEMBLE should, within sampling noise.
# -----------------------------------------------------------------------------

x = np.arange(2)
width = 0.35
axB.bar(x - width / 2, [max(single_mean_error, floor), max(single_covariance_error, floor)],
        width, color="cyan", alpha=0.85, label="Single trajectory")
axB.bar(x + width / 2, [max(mean_error, floor), max(covariance_error, floor)],
        width, color="blue", alpha=0.85, label=f"Ensemble ($M$={SHOTS})")
axB.axhline(ENSEMBLE_TOL, color="black", linestyle="--", linewidth=0.8, label="Sampling tolerance")
axB.set_yscale("log")
axB.set_xticks(x)
axB.set_xticklabels(["Mean error", "Covariance error"])
style_axis(axB, "", "Frobenius error vs. affine channel",
           "flow_pattern: conditional vs. unconditional", "B")
axB.legend(frameon=False, loc="best", fontsize=7)

fig.suptitle("CV-flow certification and simulation cross-check", fontsize=10.5,
             fontweight="bold", y=1.02)
fig.tight_layout(rect=[0, 0, 1, 0.94])
common.save_figure(fig, "R14_cvflow")
plt.close(fig)

common.print_summary(
    "R14 CV-flow validation",
    valid_certificate_found=True,
    invalid_certificate_found=False,
    invalid_order_residual=residual,
    flow_vs_channel_mean_error=mean_error,
    flow_vs_channel_covariance_error=covariance_error,
)


## R15 — Raw Piquasso Gaussian Reference (Independent Backend Check)

R1–R14 validated PhotoGraphiQ's own affine `gaussian_channel` analyzer against
hand-written NumPy formulas. R15 checks something different: does
PhotoGraphiQ's **Piquasso backend adapter** (`backends/piquasso.py`) actually
reproduce what a completely independent, hand-written Piquasso program
computes for the same physical operation? This is a genuine second code path
— Piquasso's own Gaussian simulator, not PhotoGraphiQ's internal analytic
machinery — for five native gate types plus a compound circuit.

### Construction

For each gate family, two programs are built from the *same* coherent input
\(\alpha\) and the *same* physical parameters, through two independent
routes:

$$
\text{raw: } \; \texttt{pq.Program}\{\texttt{Vacuum, Displacement}(\alpha), \texttt{Gate}\}
\;\xrightarrow{\texttt{pq.GaussianSimulator}}\;
(\mu_{\text{raw}}, V_{\text{raw}}),
$$

$$
\text{PhotoGraphiQ: } \; \texttt{Pattern}\{\texttt{Gate}\}
\;\xrightarrow{\texttt{simulate(backend="piquasso")}}\;
(\mu_{\text{pg}}, V_{\text{pg}}).
$$

Native `pq.ControlledZ` is used for the raw CZ program (the correct
Gaussian-regime representation; Piquasso's `GaussianTransform` is a different
primitive used internally by PhotoGraphiQ's *Fock* backend, R28's finding
notwithstanding — not interchangeable with the Gaussian CZ tested here).

An additional independent cross-check uses the documented single-mode
analytic formula (`docs/theory.md`)

$$
\langle n\rangle = \frac{\operatorname{Tr}(V)+\mu\cdot\mu-2}{4}
\qquad(\hbar=2),
$$

applied identically to both \((\mu_{\text{raw}},V_{\text{raw}})\) and
\((\mu_{\text{pg}},V_{\text{pg}})\), giving a third, physically meaningful
scalar (mean photon number) that must also agree.

### What is checked

\(N_{\text{random}}=20\) random parameter draws for each of: displacement,
rotation, squeezing, beamsplitter, CZ, and a four-gate compound circuit
(squeeze + rotate + CZ + displace) — 120 cases total, each compared on
\(\|\mu_{\text{pg}}-\mu_{\text{raw}}\|\), \(\|V_{\text{pg}}-V_{\text{raw}}\|_F\)
and (single-mode cases) \(|\langle n\rangle_{\text{pg}}-\langle n\rangle_{\text{raw}}|\).

### Success criteria

- All three error metrics stay at machine precision (\(\lesssim10^{-9}\)) for
  every gate family and every random draw — panel A/B show the worst case per
  family; panel C shows the photon-number cross-check; panel D shows the full
  120-case distribution has no long tail (no family is silently worse than the
  others).


In [ ]:
from __future__ import annotations

import numpy as np
import piquasso as pq

import photographiq as pg

set_reference_style()

SEED = 20260913
N_RANDOM = 20
HBAR = 2.0
TOL = 1e-9


def mean_photon_number(mean: np.ndarray, covariance: np.ndarray) -> float:
    """Independent analytic single-mode formula: (Tr(V)+mu.mu-2)/4 (docs/theory.md)."""
    return float((np.trace(covariance) + mean @ mean - 2) / 4)


def raw_single_mode(alpha: complex, gate_builder):
    """Prepare a coherent state and apply one native gate via a fresh raw program."""
    with pq.Program() as program:
        pq.Q(0) | pq.Vacuum()
        pq.Q(0) | pq.Displacement(r=abs(alpha), phi=np.angle(alpha))
        gate_builder(program)
    state = pq.GaussianSimulator(d=1, config=pq.Config(hbar=HBAR)).execute(program).state
    return np.array(state.xpxp_mean_vector), np.array(state.xpxp_covariance_matrix) / 2


def raw_two_mode(alpha0: complex, alpha1: complex, gate_builder):
    with pq.Program() as program:
        pq.Q(0, 1) | pq.Vacuum()
        pq.Q(0) | pq.Displacement(r=abs(alpha0), phi=np.angle(alpha0))
        pq.Q(1) | pq.Displacement(r=abs(alpha1), phi=np.angle(alpha1))
        gate_builder(program)
    state = pq.GaussianSimulator(d=2, config=pq.Config(hbar=HBAR)).execute(program).state
    return np.array(state.xpxp_mean_vector), np.array(state.xpxp_covariance_matrix) / 2


def pg_single_mode(alpha: complex, command):
    pattern = pg.Pattern(inputs=(0,)).append(command).append(pg.Output((0,)))
    result = pg.simulate(pattern, backend="piquasso", inputs={0: pg.GaussianInput.coherent(alpha)}, seed=0)
    return result.state.mean, result.state.covariance


def pg_two_mode(alpha0: complex, alpha1: complex, commands):
    pattern = pg.Pattern(inputs=(0, 1))
    pattern.extend(commands)
    pattern.append(pg.Output((0, 1)))
    inputs = {0: pg.GaussianInput.coherent(alpha0), 1: pg.GaussianInput.coherent(alpha1)}
    result = pg.simulate(pattern, backend="piquasso", inputs=inputs, seed=0)
    return result.state.mean, result.state.covariance


def compare(name, alpha_desc, raw_mean, raw_cov, pg_mean, pg_cov):
    mean_error = common.frobenius_error(pg_mean, raw_mean)
    cov_error = common.frobenius_error(pg_cov, raw_cov)
    raw_n = mean_photon_number(raw_mean[:2], raw_cov[:2, :2]) if len(raw_mean) == 2 else None
    pg_n = mean_photon_number(pg_mean[:2], pg_cov[:2, :2]) if len(pg_mean) == 2 else None
    return {
        "gate": name,
        "params": alpha_desc,
        "mean_vector_norm_error": mean_error,
        "covariance_frobenius_error": cov_error,
        "photon_number_error": None if raw_n is None else abs(raw_n - pg_n),
    }


generator = common.rng(SEED)
rows = []

for _ in range(N_RANDOM):
    alpha = generator.uniform(-1, 1) + 1j * generator.uniform(-1, 1)
    q, p = generator.uniform(-1, 1), generator.uniform(-1, 1)
    shift = complex(q, p) / 2

    def gate_builder(program, shift=shift):
        pq.Q(0) | pq.Displacement(r=abs(shift), phi=np.angle(shift))

    raw_mean, raw_cov = raw_single_mode(alpha, gate_builder)
    pg_mean, pg_cov = pg_single_mode(alpha, pg.Displace(0, q=q, p=p))
    rows.append(compare("displacement", {"alpha": [alpha.real, alpha.imag], "q": q, "p": p}, raw_mean, raw_cov, pg_mean, pg_cov))

for _ in range(N_RANDOM):
    alpha = generator.uniform(-1, 1) + 1j * generator.uniform(-1, 1)
    angle = generator.uniform(-np.pi, np.pi)
    raw_mean, raw_cov = raw_single_mode(alpha, lambda program, angle=angle: (pq.Q(0) | pq.Phaseshifter(phi=angle)))
    pg_mean, pg_cov = pg_single_mode(alpha, pg.Rotate(0, angle))
    rows.append(compare("rotation", {"alpha": [alpha.real, alpha.imag], "angle": angle}, raw_mean, raw_cov, pg_mean, pg_cov))

for _ in range(N_RANDOM):
    alpha = generator.uniform(-1, 1) + 1j * generator.uniform(-1, 1)
    r = generator.uniform(-1.2, 1.2)
    raw_mean, raw_cov = raw_single_mode(alpha, lambda program, r=r: (pq.Q(0) | pq.Squeezing(r=r)))
    pg_mean, pg_cov = pg_single_mode(alpha, pg.Squeeze(0, r))
    rows.append(compare("squeezing", {"alpha": [alpha.real, alpha.imag], "r": r}, raw_mean, raw_cov, pg_mean, pg_cov))

for _ in range(N_RANDOM):
    alpha0 = generator.uniform(-1, 1) + 1j * generator.uniform(-1, 1)
    alpha1 = generator.uniform(-1, 1) + 1j * generator.uniform(-1, 1)
    theta = generator.uniform(-np.pi / 2, np.pi / 2)
    raw_mean, raw_cov = raw_two_mode(alpha0, alpha1, lambda program, theta=theta: (pq.Q(0, 1) | pq.Beamsplitter(theta=theta, phi=0.0)))
    pg_mean, pg_cov = pg_two_mode(alpha0, alpha1, [pg.BeamSplitter(0, 1, theta)])
    rows.append(
        compare(
            "beamsplitter",
            {"alpha0": [alpha0.real, alpha0.imag], "alpha1": [alpha1.real, alpha1.imag], "theta": theta},
            raw_mean, raw_cov, pg_mean, pg_cov,
        )
    )

for _ in range(N_RANDOM):
    alpha0 = generator.uniform(-1, 1) + 1j * generator.uniform(-1, 1)
    alpha1 = generator.uniform(-1, 1) + 1j * generator.uniform(-1, 1)
    weight = generator.uniform(-2, 2)
    raw_mean, raw_cov = raw_two_mode(alpha0, alpha1, lambda program, weight=weight: (pq.Q(0, 1) | pq.ControlledZ(s=weight)))
    pg_mean, pg_cov = pg_two_mode(alpha0, alpha1, [pg.Entangle(0, 1, weight)])
    rows.append(
        compare(
            "cz",
            {"alpha0": [alpha0.real, alpha0.imag], "alpha1": [alpha1.real, alpha1.imag], "weight": weight},
            raw_mean, raw_cov, pg_mean, pg_cov,
        )
    )

for _ in range(N_RANDOM):
    alpha0 = generator.uniform(-1, 1) + 1j * generator.uniform(-1, 1)
    alpha1 = generator.uniform(-1, 1) + 1j * generator.uniform(-1, 1)
    r, angle, weight, q, p = (
        generator.uniform(-0.6, 0.6),
        generator.uniform(-np.pi, np.pi),
        generator.uniform(-1, 1),
        generator.uniform(-1, 1),
        generator.uniform(-1, 1),
    )

    def raw_compound(program, r=r, angle=angle, weight=weight, q=q, p=p):
        shift = complex(q, p) / 2
        pq.Q(0) | pq.Squeezing(r=r)
        pq.Q(0) | pq.Phaseshifter(phi=angle)
        pq.Q(0, 1) | pq.ControlledZ(s=weight)
        pq.Q(1) | pq.Displacement(r=abs(shift), phi=np.angle(shift))

    raw_mean, raw_cov = raw_two_mode(alpha0, alpha1, raw_compound)
    pg_mean, pg_cov = pg_two_mode(
        alpha0, alpha1, [pg.Squeeze(0, r), pg.Rotate(0, angle), pg.Entangle(0, 1, weight), pg.Displace(1, q=q, p=p)]
    )
    rows.append(
        compare(
            "compound(squeeze+rotate+cz+displace)",
            {"r": r, "angle": angle, "weight": weight, "q": q, "p": p},
            raw_mean, raw_cov, pg_mean, pg_cov,
        )
    )

# -----------------------------------------------------------------------------
# Validation
# -----------------------------------------------------------------------------

failures = [r for r in rows if r["mean_vector_norm_error"] > TOL or r["covariance_frobenius_error"] > TOL]
if failures:
    common.save_csv(failures, "R15_raw_piquasso_gaussian_FAILURES")
    raise AssertionError(f"{len(failures)} cases disagreed with raw Piquasso beyond tolerance; see FAILURES csv")

common.save_result(rows, "R15_raw_piquasso_gaussian", extra={"n_random_per_gate": N_RANDOM, "seed": SEED})

gates = sorted(set(r["gate"] for r in rows))
summary_table = [
    {
        "gate": gate,
        "n": len([r for r in rows if r["gate"] == gate]),
        "max_mean_error": max(r["mean_vector_norm_error"] for r in rows if r["gate"] == gate),
        "max_covariance_error": max(r["covariance_frobenius_error"] for r in rows if r["gate"] == gate),
        "max_photon_number_error": max(
            (r["photon_number_error"] for r in rows if r["gate"] == gate and r["photon_number_error"] is not None),
            default=None,
        ),
    }
    for gate in gates
]
common.save_csv(summary_table, "R15_raw_piquasso_gaussian_summary_table")

# =============================================================================
# Final 2 x 2 publication figure
# =============================================================================

x = np.arange(len(gates))
floor = 1e-18

fig, axes = plt.subplots(2, 2, figsize=(10.0, 7.4), facecolor="white")
axA, axB, axC, axD = axes.ravel()

# -----------------------------------------------------------------------------
# A. Worst-case covariance error per gate family
# -----------------------------------------------------------------------------

axA.bar(x, [max(t["max_covariance_error"], floor) for t in summary_table], color="blue")
axA.set_yscale("log")
axA.axhline(TOL, color="black", linestyle="--", linewidth=0.8, label="Tolerance")
axA.set_xticks(x)
axA.set_xticklabels(gates, rotation=30, ha="right", fontsize=7)
style_axis(axA, "", r"$\max\|V_{\rm pg}-V_{\rm raw}\|_F$", "Covariance error", "A")
axA.legend(frameon=False, loc="best")

# -----------------------------------------------------------------------------
# B. Worst-case mean error per gate family
# -----------------------------------------------------------------------------

axB.bar(x, [max(t["max_mean_error"], floor) for t in summary_table], color="red")
axB.set_yscale("log")
axB.axhline(TOL, color="black", linestyle="--", linewidth=0.8, label="Tolerance")
axB.set_xticks(x)
axB.set_xticklabels(gates, rotation=30, ha="right", fontsize=7)
style_axis(axB, "", r"$\max\|\mu_{\rm pg}-\mu_{\rm raw}\|$", "Mean error", "B")
axB.legend(frameon=False, loc="best")

# -----------------------------------------------------------------------------
# C. Independent photon-number cross-check (single-mode gates only)
# -----------------------------------------------------------------------------

photon_gates = [t["gate"] for t in summary_table if t["max_photon_number_error"] is not None]
photon_errors = [max(t["max_photon_number_error"], floor) for t in summary_table if t["max_photon_number_error"] is not None]
axC.bar(np.arange(len(photon_gates)), photon_errors, color="magenta")
axC.set_yscale("log")
axC.axhline(TOL, color="black", linestyle="--", linewidth=0.8, label="Tolerance")
axC.set_xticks(np.arange(len(photon_gates)))
axC.set_xticklabels(photon_gates, rotation=30, ha="right", fontsize=7)
style_axis(axC, "", r"$\max|\langle n\rangle_{\rm pg}-\langle n\rangle_{\rm raw}|$",
           "Photon-number cross-check", "C")
axC.legend(frameon=False, loc="best")

# -----------------------------------------------------------------------------
# D. Full 120-case error distribution (no hidden long tail)
# -----------------------------------------------------------------------------

all_cov = np.maximum([r["covariance_frobenius_error"] for r in rows], floor)
all_mean = np.maximum([r["mean_vector_norm_error"] for r in rows], floor)
axD.scatter(np.log10(all_mean), np.log10(all_cov), s=14, color="blue", alpha=0.6, edgecolors="none")
axD.axhline(np.log10(TOL), color="black", linestyle="--", linewidth=0.8)
axD.axvline(np.log10(TOL), color="black", linestyle="--", linewidth=0.8, label="Tolerance")
style_axis(axD, r"$\log_{10}$(mean error)", r"$\log_{10}$(covariance error)",
           f"All {len(rows)} cases", "D")
axD.legend(frameon=False, loc="best")

fig.suptitle("PhotoGraphiQ Piquasso backend vs. independent raw Piquasso programs",
             fontsize=11, fontweight="bold", y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.96], h_pad=2.2, w_pad=2.2)
common.save_figure(fig, "R15_raw_piquasso_gaussian")
plt.close(fig)

common.print_summary(
    "R15 raw Piquasso Gaussian reference",
    total_cases=len(rows),
    max_mean_error=max(r["mean_vector_norm_error"] for r in rows),
    max_covariance_error=max(r["covariance_frobenius_error"] for r in rows),
)


## R16 — Independent NumPy Analytic Reference (Three-Way Backend Check)

R15 checked the Piquasso backend against a raw Piquasso program. R16 checks
**both** Gaussian backends (`"gaussian"`, PhotoGraphiQ's own NumPy simulator,
and `"piquasso"`) plus a raw Piquasso program, all at once, against a target
built with plain NumPy matrices (independent of `photographiq.gaussian`) —
the strongest form of cross-validation used in this suite: three numerically
independent implementations against one analytic ground truth.

### Construction

For a coherent-state input \((\mu_{\text{in}}=I\cdot(\dots), V_{\text{in}}=I)\)
and a hand-written symplectic target \(S\) (rotation/squeezing/beamsplitter/CZ
matrices, reproduced independently — same formulas as R3, R4, R6, R7, but
typed again here with zero shared code), the analytic prediction is the
noiseless affine map

$$
\mu_{\text{analytic}} = S\,\mu_{\text{in}} + \delta,
\qquad
V_{\text{analytic}} = S\,V_{\text{in}}\,S^{T}.
$$

(no `+N`: single fixed-angle gates applied directly to a state, not compiled
through finite-squeezing teleportation, so there is no resource noise here —
unlike R3–R10.) This is compared against:

$$
(\mu_{\text{gaussian}}, V_{\text{gaussian}}),\quad
(\mu_{\text{piquasso}}, V_{\text{piquasso}}),\quad
(\mu_{\text{raw-piquasso}}, V_{\text{raw-piquasso}}),
$$

for rotation, squeezing, displacement, beamsplitter, CZ, and a
squeeze+rotate+CZ compound (whose target matrix is itself an explicit NumPy
matrix product, checked structurally against the individual gate matrices).

### What is checked

Six error metrics per case (mean/covariance error for each of the three
execution paths against the shared analytic target) — 36 numbers total.

### Success criteria

- Every one of the 36 errors is at machine precision (\(\lesssim10^{-9}\)):
  panel A/B show mean and covariance error, grouped by gate and by execution
  path, so any single path or gate family that disagreed would be visible
  immediately rather than averaged away.


In [ ]:
from __future__ import annotations

import numpy as np
import piquasso as pq

import photographiq as pg

set_reference_style()

HBAR = 2.0
TOL = 1e-9


# --------------------------------------------------------------------------- #
# Independent NumPy target matrices (not photographiq.gaussian)
# --------------------------------------------------------------------------- #


def target_rotation(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])


def target_squeezing(r):
    return np.array([[np.exp(-r), 0.0], [0.0, np.exp(r)]])


def target_displacement_delta(q, p):
    return np.array([q, p])


def target_beamsplitter(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, 0, -s, 0], [0, c, 0, -s], [s, 0, c, 0], [0, s, 0, c]])


def target_cz(weight):
    matrix = np.eye(4)
    matrix[1, 2] = weight
    matrix[3, 0] = weight
    return matrix


def raw_piquasso(alphas, gate_builder, modes):
    with pq.Program() as program:
        pq.Q(*range(modes)) | pq.Vacuum()
        for i, alpha in enumerate(alphas):
            pq.Q(i) | pq.Displacement(r=abs(alpha), phi=np.angle(alpha))
        gate_builder()
    state = pq.GaussianSimulator(d=modes, config=pq.Config(hbar=HBAR)).execute(program).state
    return np.array(state.xpxp_mean_vector), np.array(state.xpxp_covariance_matrix) / 2


def run_case(name, params, modes, alphas, target_matrix, target_delta, pg_commands, raw_builder):
    mean_in = np.empty(2 * modes)
    mean_in[0::2] = [2 * a.real for a in alphas]
    mean_in[1::2] = [2 * a.imag for a in alphas]
    cov_in = np.eye(2 * modes)

    analytic_mean = target_matrix @ mean_in + target_delta
    analytic_cov = target_matrix @ cov_in @ target_matrix.T

    inputs = {i: pg.GaussianInput.coherent(a) for i, a in enumerate(alphas)}
    rows = {}
    for backend in ("gaussian", "piquasso"):
        pattern = pg.Pattern(inputs=tuple(range(modes)))
        pattern.extend(pg_commands)
        pattern.append(pg.Output(tuple(range(modes))))
        result = pg.simulate(pattern, backend=backend, inputs=inputs, seed=0)
        rows[f"{backend}_mean_error"] = common.frobenius_error(result.state.mean, analytic_mean)
        rows[f"{backend}_covariance_error"] = common.frobenius_error(result.state.covariance, analytic_cov)

    raw_mean, raw_cov = raw_piquasso(alphas, raw_builder, modes)
    rows["raw_piquasso_mean_error"] = common.frobenius_error(raw_mean, analytic_mean)
    rows["raw_piquasso_covariance_error"] = common.frobenius_error(raw_cov, analytic_cov)

    return {"gate": name, "params": params, **rows}


rows = []

rows.append(
    run_case(
        "rotation", {"theta": 0.6}, 1, [0.4 + 0.2j],
        target_rotation(0.6), np.zeros(2), [pg.Rotate(0, 0.6)],
        lambda: (pq.Q(0) | pq.Phaseshifter(phi=0.6)),
    )
)
rows.append(
    run_case(
        "squeezing", {"r": 0.5}, 1, [0.3 - 0.1j],
        target_squeezing(0.5), np.zeros(2), [pg.Squeeze(0, 0.5)],
        lambda: (pq.Q(0) | pq.Squeezing(r=0.5)),
    )
)
rows.append(
    run_case(
        "displacement", {"q": 0.4, "p": -0.3}, 1, [0.1 + 0.1j],
        np.eye(2), target_displacement_delta(0.4, -0.3), [pg.Displace(0, q=0.4, p=-0.3)],
        lambda: (pq.Q(0) | pq.Displacement(r=abs(complex(0.4, -0.3) / 2), phi=np.angle(complex(0.4, -0.3) / 2))),
    )
)
rows.append(
    run_case(
        "beamsplitter", {"theta": 0.35}, 2, [0.3 + 0.1j, -0.2 + 0.2j],
        target_beamsplitter(0.35), np.zeros(4), [pg.BeamSplitter(0, 1, 0.35)],
        lambda: (pq.Q(0, 1) | pq.Beamsplitter(theta=0.35, phi=0.0)),
    )
)
rows.append(
    run_case(
        "cz", {"weight": 0.7}, 2, [0.2 + 0.3j, 0.1 - 0.4j],
        target_cz(0.7), np.zeros(4), [pg.Entangle(0, 1, 0.7)],
        lambda: (pq.Q(0, 1) | pq.ControlledZ(s=0.7)),
    )
)
compound_matrix = target_cz(0.4) @ np.block(
    [[target_rotation(0.3), np.zeros((2, 2))], [np.zeros((2, 2)), np.eye(2)]]
) @ np.block([[target_squeezing(0.2), np.zeros((2, 2))], [np.zeros((2, 2)), np.eye(2)]])
rows.append(
    run_case(
        "compound(squeeze+rotate+cz)", {"r": 0.2, "angle": 0.3, "weight": 0.4}, 2,
        [0.2 + 0.0j, -0.1 + 0.2j], compound_matrix, np.zeros(4),
        [pg.Squeeze(0, 0.2), pg.Rotate(0, 0.3), pg.Entangle(0, 1, 0.4)],
        lambda: (
            pq.Q(0) | pq.Squeezing(r=0.2),
            pq.Q(0) | pq.Phaseshifter(phi=0.3),
            pq.Q(0, 1) | pq.ControlledZ(s=0.4),
        ),
    )
)

# -----------------------------------------------------------------------------
# Validation
# -----------------------------------------------------------------------------

error_keys = [k for k in rows[0] if k.endswith("_error")]
failures = [r for r in rows if any(r[k] > TOL for k in error_keys)]
if failures:
    common.save_csv(failures, "R16_numpy_analytic_reference_FAILURES")
    raise AssertionError("A backend disagreed with the independent analytic target; see FAILURES csv")

common.save_result(rows, "R16_numpy_analytic_reference")

# =============================================================================
# Final 1 x 2 publication figure
# =============================================================================

gates = [r["gate"] for r in rows]
x = np.arange(len(gates))
width = 0.25
floor = 1e-18
paths = [("gaussian", "blue", "NumPy Gaussian backend"),
         ("piquasso", "magenta", "Piquasso Gaussian backend"),
         ("raw_piquasso", "red", "Raw Piquasso")]

fig, (axA, axB) = plt.subplots(1, 2, figsize=(11.2, 4.6), facecolor="white")

for ax, metric, title, letter in (
    (axA, "covariance_error", "Covariance error vs. NumPy analytic target", "A"),
    (axB, "mean_error", "Mean error vs. NumPy analytic target", "B"),
):
    for i, (key, color, label) in enumerate(paths):
        values = np.maximum([r[f"{key}_{metric}"] for r in rows], floor)
        ax.bar(x + (i - 1) * width, values, width, color=color, label=label)
    ax.set_yscale("log")
    ax.axhline(TOL, color="black", linestyle="--", linewidth=0.8, label="Tolerance")
    ax.set_xticks(x)
    ax.set_xticklabels(gates, rotation=25, ha="right", fontsize=7)
    style_axis(ax, "", f"$\\log$-scale {metric.replace('_', ' ')}", title, letter)

axA.legend(frameon=False, fontsize=7, loc="best")

fig.suptitle("Three independent execution paths vs. one NumPy analytic target",
             fontsize=10.5, fontweight="bold", y=1.02)
fig.tight_layout(rect=[0, 0, 1, 0.93], w_pad=2.2)
common.save_figure(fig, "R16_numpy_analytic_reference")
plt.close(fig)

common.print_summary(
    "R16 independent NumPy analytic reference",
    cases=len(rows),
    max_error=max(max(r[k] for k in error_keys) for r in rows),
)


## R17 — Graphix Structural Comparison (Structure Only)

R17 checks that PhotoGraphiQ's CV-MBQC resource graphs and dependency
structure match an independently, by-hand-built Graphix (qubit MBQC) pattern
for the same abstract topology. This is a **structural**, not physical,
comparison — Graphix works over qubits and has no notion of a CV quadrature,
amplitude or Wigner function, so nothing numerical is ever compared here
(`docs/validation/index.md`); only the discrete combinatorial objects that
both frameworks share are checked: the resource graph, measurement order and
classical-dependency domains.

### What "structural equivalence" means

For a line topology (\(0-1-\dots-(n-1)\), input 0) and a star topology (hub 0,
\(k\) leaves), both an equivalent Graphix pattern (`N`, `E`, `M`, `X`, `Z`
commands) and a PhotoGraphiQ CV `Pattern` are built by hand from the same
topology description. Four discrete invariants are then compared:

1. **Graph isomorphism**: \(G_{\text{graphix}}\cong G_{\text{cv}}\)
   (`networkx.is_isomorphic`) and canonical edge-set equality.
2. **I/O sets**: identical input and output node tuples.
3. **Measurement order**: the sequence of measured nodes matches exactly.
4. **Dependency domains**: every Graphix `s_domain`/`domain` entry (which
   node's outcome a later `M`/`X`/`Z` command depends on) corresponds to a
   real edge in PhotoGraphiQ's own `Pattern.dependencies()` DAG (R12).

### What this experiment does *not* claim

Per `MANUSCRIPT_MAP.md`: R17 is never evidence for CV amplitude, quadrature
or Wigner-function correctness — only for the shared combinatorial layer
(who depends on whom, and the shape of the entanglement graph) being
consistent between an independent qubit-MBQC reference implementation and
PhotoGraphiQ's CV resource graphs.

### Success criteria

- All four checks hold for every one of 4 topologies (line \(n=3,5\); star
  with 3, 5 leaves) — panel C: a compliance grid, one row per case.
- The resource-graph layouts themselves are shown for one line and one star
  case (panels A, B) so the claimed isomorphism is visually inspectable, not
  only asserted.


In [ ]:
from __future__ import annotations

import networkx as nx
import numpy as np

set_reference_style()

if not common.HAS_GRAPHIX:
    print("SKIPPED: graphix is not installed; see requirements_notes.md")
else:
    import graphix
    from graphix import command

    import photographiq as pg

    def build_line_case(n: int):
        graph = pg.CVGraph.line(n, inputs=(0,))
        cv = pg.Pattern(graph)
        cv.measure(0, pg.Homodyne.q())
        for i in range(1, n - 1):
            cv.measure(i, pg.Homodyne(0.2 + 0.1 * i + pg.Outcome(i - 1)))
        cv.displace(n - 1, q=pg.Outcome(n - 2), p=(pg.Outcome(0) if n > 2 else 0.0))
        cv.append(pg.Output((n - 1,)))
        cv.validate()

        dv = graphix.Pattern(input_nodes=[0])
        for i in range(1, n):
            dv.add(command.N(i))
        for i in range(n - 1):
            dv.add(command.E((i, i + 1)))
        dv.add(command.M(0))
        for i in range(1, n - 1):
            dv.add(command.M(i, s_domain={i - 1}))
        dv.add(command.X(n - 1, domain={n - 2}))
        if n > 2:
            dv.add(command.Z(n - 1, domain={0}))
        return cv, dv

    def build_star_case(leaves: int):
        graph = pg.CVGraph.star(leaves, inputs=(0,))
        leaf_nodes = tuple(range(1, leaves + 1))
        cv = pg.Pattern(graph)
        cv.measure(0, pg.Homodyne.q())
        for leaf in leaf_nodes:
            cv.displace(leaf, q=pg.Outcome(0))
        cv.append(pg.Output(leaf_nodes))
        cv.validate()

        dv = graphix.Pattern(input_nodes=[0])
        for leaf in leaf_nodes:
            dv.add(command.N(leaf))
        for leaf in leaf_nodes:
            dv.add(command.E((0, leaf)))
        dv.add(command.M(0))
        for leaf in leaf_nodes:
            dv.add(command.X(leaf, domain={0}))
        return cv, dv

    def compare(name, cv, dv):
        og = dv.to_opengraph()
        isomorphic = nx.is_isomorphic(og.graph, cv.graph.network)
        edges_match = common.canonical_edges(og.graph) == common.canonical_edges(cv.graph.network)
        io_match = tuple(dv.input_nodes) == cv.inputs and tuple(dv.output_nodes) == cv.outputs
        graphix_measure_order = [c.node for c in dv if isinstance(c, command.M)]
        cv_measure_order = [c.node for c in cv.commands if isinstance(c, pg.Measure)]
        order_match = graphix_measure_order == cv_measure_order

        cv_measurements = [(i, c) for i, c in enumerate(cv.commands) if isinstance(c, pg.Measure)]
        dependency_graph = cv.dependencies()
        domain_checks = []
        for gx_command in dv:
            if isinstance(gx_command, command.M) and gx_command.s_domain:
                target_index = next(i for i, c in cv_measurements if c.node == gx_command.node)
                for source_node in gx_command.s_domain:
                    source_index = next(i for i, c in cv_measurements if c.node == source_node)
                    domain_checks.append(dependency_graph.has_edge(source_index, target_index))
            if isinstance(gx_command, command.X) and gx_command.domain:
                for source_node in gx_command.domain:
                    source_index = next(i for i, c in cv_measurements if c.node == source_node)
                    # The corresponding correction is the Displace on gx_command.node.
                    target_index = next(
                        i for i, c in enumerate(cv.commands)
                        if isinstance(c, pg.Displace) and c.node == gx_command.node
                    )
                    domain_checks.append(dependency_graph.has_edge(source_index, target_index))
        domains_match = all(domain_checks) and len(domain_checks) > 0

        return {
            "case": name,
            "nodes": len(cv.graph.nodes),
            "edges": cv.graph.network.number_of_edges(),
            "graph_isomorphic": isomorphic,
            "canonical_edges_match": edges_match,
            "input_output_match": io_match,
            "measurement_order_match": order_match,
            "dependency_domain_checks": len(domain_checks),
            "dependency_domains_match": domains_match,
        }

    line5_cv, line5_dv = build_line_case(5)
    star5_cv, star5_dv = build_star_case(5)
    rows = [
        compare("line_n3", *build_line_case(3)),
        compare("line_n5", line5_cv, line5_dv),
        compare("star_leaves_3", *build_star_case(3)),
        compare("star_leaves_5", star5_cv, star5_dv),
    ]

    failures = [
        r for r in rows
        if not (r["graph_isomorphic"] and r["canonical_edges_match"] and r["input_output_match"]
                and r["measurement_order_match"] and r["dependency_domains_match"])
    ]
    if failures:
        common.save_csv(failures, "R17_graphix_structure_FAILURES")
        raise AssertionError("Structural comparison against Graphix failed; see FAILURES csv")

    common.save_result(
        rows,
        "R17_graphix_structure",
        extra={
            "note": "Structural comparison only: node/edge/I-O/order/domain equivalence. "
            "Graphix never validates CV amplitudes or quadrature statistics."
        },
    )

    # =========================================================================
    # Final 1 x 3 publication figure
    # =========================================================================

    fig, (axA, axB, axC) = plt.subplots(1, 3, figsize=(13.0, 4.4), facecolor="white")

    nx.draw_networkx(line5_cv.graph.network, nx.spring_layout(line5_cv.graph.network, seed=0),
                      ax=axA, node_color="#b9d8db", node_size=500, font_size=8)
    axA.set_title("Line ($n$=5) resource graph", pad=6)
    axA.set_axis_off()
    axA.text(-0.05, 1.05, "A", transform=axA.transAxes, fontsize=11, fontweight="bold", va="top")

    nx.draw_networkx(star5_cv.graph.network, nx.spring_layout(star5_cv.graph.network, seed=0),
                      ax=axB, node_color="#d8b9d0", node_size=500, font_size=8)
    axB.set_title("Star (5 leaves) resource graph", pad=6)
    axB.set_axis_off()
    axB.text(-0.05, 1.05, "B", transform=axB.transAxes, fontsize=11, fontweight="bold", va="top")

    checks = ["graph_isomorphic", "canonical_edges_match", "input_output_match",
              "measurement_order_match", "dependency_domains_match"]
    grid = np.array([[1.0 if r[c] else 0.0 for c in checks] for r in rows])
    axC.imshow(grid, cmap="Greens", vmin=0, vmax=1.3, aspect="auto")
    axC.set_xticks(range(len(checks)))
    axC.set_xticklabels([c.replace("_", "\n") for c in checks], fontsize=6.5)
    axC.set_yticks(range(len(rows)))
    axC.set_yticklabels([r["case"] for r in rows], fontsize=8)
    for yi in range(len(rows)):
        for xi in range(len(checks)):
            axC.text(xi, yi, "OK", ha="center", va="center", fontsize=7, color="black")
    axC.set_title("Structural compliance (all pass)", pad=6)
    axC.text(-0.15, 1.05, "C", transform=axC.transAxes, fontsize=11, fontweight="bold", va="top")

    fig.suptitle("PhotoGraphiQ vs. Graphix: structural equivalence (never CV amplitudes)",
                 fontsize=10.5, fontweight="bold", y=1.03)
    fig.tight_layout(rect=[0, 0, 1, 0.94], w_pad=2.2)
    common.save_figure(fig, "R17_graphix_structure")
    plt.close(fig)

    common.print_summary("R17 Graphix structural comparison", cases=len(rows), all_passed=not failures)


## R18 — Gaussian Backend Cross-Check (Exact vs. Finite-Fock Approximation)

R18 tests two qualitatively different claims side by side, and is careful not
to conflate them: (1) the NumPy Gaussian backend and the Piquasso Gaussian
backend must agree **exactly** on any physical (non-adaptive) Gaussian
workload, since both claim to solve the identical linear-optics problem
exactly; (2) the experimental `piquasso-mixed-fock` backend can only
**approximate** a Gaussian state at finite Fock cutoff, so it is held to a
looser, explicitly-reported tolerance, never the exact one.

### Two-mode random workloads

15 random two-mode circuits (4 gates each, drawn from rotate/squeeze/
beamsplitter/CZ/displace with independently random parameters) act on
low-energy coherent inputs (\(|\alpha|\lesssim0.35\)) so a modest Fock cutoff
\(c=14\) can plausibly represent the state.

### What is checked

- **Exact agreement** (NumPy vs. Piquasso Gaussian backends): mean vector,
  covariance matrix, and the documented mean-photon-number formula
  (\(\langle n\rangle=(\operatorname{Tr}V+\mu\cdot\mu-2)/4\), R15) must match
  to \(\lesssim10^{-8}\) — both are exact linear-algebra solutions of the same
  affine channel, so any disagreement would indicate a genuine backend bug,
  not truncation.
- **Approximate agreement** (Gaussian vs. `piquasso-mixed-fock` at
  \(c=14\)): photon number and \(q\)-quadrature mean are compared with a
  looser, reported (not asserted-tight) tolerance, since finite-cutoff Fock
  truncation is expected to introduce a small but nonzero residual. Cases
  that legitimately exceed the backend's own retained-norm guard are recorded
  as "not ok", not silently skipped (`common.safe_cutoff_run`).

### Success criteria

- Zero exact-backend disagreements above \(10^{-8}\) (panel A: mean,
  covariance and photon-number error distributions for the exact pair).
- Mixed-Fock photon-number and \(q\)-mean errors are small and unimodal
  (panel B), consistent with finite-cutoff truncation rather than a
  correctness bug; any guard-triggered ("not ok") cases are counted, not
  hidden.


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg

set_reference_style()

SEED = 20260913
N_RANDOM = 15
FOCK_CUTOFF = 14
GATE_KINDS = ("rotate", "squeeze", "beamsplitter", "cz", "displace")
EXACT_TOL = 1e-8


def random_pattern(generator, n_gates=4):
    pattern = pg.Pattern(inputs=(0, 1))
    for _ in range(n_gates):
        kind = generator.choice(GATE_KINDS)
        if kind == "rotate":
            pattern.append(pg.Rotate(int(generator.integers(0, 2)), float(generator.uniform(-np.pi, np.pi))))
        elif kind == "squeeze":
            pattern.append(pg.Squeeze(int(generator.integers(0, 2)), float(generator.uniform(-0.4, 0.4))))
        elif kind == "beamsplitter":
            pattern.append(pg.BeamSplitter(0, 1, float(generator.uniform(-np.pi / 2, np.pi / 2))))
        elif kind == "cz":
            pattern.append(pg.Entangle(0, 1, float(generator.uniform(-1, 1))))
        else:
            pattern.append(
                pg.Displace(int(generator.integers(0, 2)), q=float(generator.uniform(-0.3, 0.3)), p=float(generator.uniform(-0.3, 0.3)))
            )
    pattern.append(pg.Output((0, 1)))
    return pattern.validate()


generator = common.rng(SEED)
rows = []
for i in range(N_RANDOM):
    pattern = random_pattern(generator)
    alpha0 = generator.uniform(-0.25, 0.25) + 1j * generator.uniform(-0.25, 0.25)
    alpha1 = generator.uniform(-0.25, 0.25) + 1j * generator.uniform(-0.25, 0.25)
    inputs = {0: pg.GaussianInput.coherent(alpha0), 1: pg.GaussianInput.coherent(alpha1)}

    gaussian = pg.simulate(pattern, backend="gaussian", inputs=inputs, seed=0).state
    piquasso = pg.simulate(pattern, backend="piquasso", inputs=inputs, seed=0).state

    mean_error = common.frobenius_error(piquasso.mean, gaussian.mean)
    cov_error = common.frobenius_error(piquasso.covariance, gaussian.covariance)
    photon_error = max(abs(piquasso.photon_number(n) - gaussian.photon_number(n)) for n in (0, 1))

    row = {
        "sample": i,
        "n_gates": len(pattern.commands) - 1,
        "gaussian_vs_piquasso_mean_error": mean_error,
        "gaussian_vs_piquasso_covariance_error": cov_error,
        "gaussian_vs_piquasso_photon_number_error": photon_error,
    }

    fock_failure = common.safe_cutoff_run(
        lambda: pg.simulate(pattern, backend="piquasso-mixed-fock", cutoff=FOCK_CUTOFF, inputs=inputs, seed=0).state,
        label=f"sample {i} mixed-fock",
    )
    if fock_failure["ok"]:
        fock_state = fock_failure["value"]
        row["mixed_fock_photon_number_error"] = max(
            abs(fock_state.photon_number(n) - gaussian.photon_number(n)) for n in (0, 1)
        )
        row["mixed_fock_q_error"] = max(
            abs(fock_state.quadrature(n)[0] - gaussian.quadrature(n)[0]) for n in (0, 1)
        )
        row["mixed_fock_cutoff"] = FOCK_CUTOFF
        row["mixed_fock_ok"] = True
    else:
        row.update(
            mixed_fock_photon_number_error=None, mixed_fock_q_error=None,
            mixed_fock_cutoff=FOCK_CUTOFF, mixed_fock_ok=False, mixed_fock_error=fock_failure["error"],
        )
    rows.append(row)

# -----------------------------------------------------------------------------
# Validation: exact backends only (mixed-Fock is a reported approximation,
# never asserted at the exact tolerance)
# -----------------------------------------------------------------------------

exact_failures = [
    r for r in rows
    if r["gaussian_vs_piquasso_mean_error"] > EXACT_TOL
    or r["gaussian_vs_piquasso_covariance_error"] > EXACT_TOL
    or r["gaussian_vs_piquasso_photon_number_error"] > EXACT_TOL
]
if exact_failures:
    common.save_csv(exact_failures, "R18_gaussian_backends_FAILURES")
    raise AssertionError("NumPy Gaussian and Piquasso Gaussian backends disagreed beyond tolerance; see FAILURES csv")

common.save_result(rows, "R18_gaussian_backends", extra={"seed": SEED, "fock_cutoff": FOCK_CUTOFF})

ok_rows = [r for r in rows if r["mixed_fock_ok"]]
n_guard_triggered = len(rows) - len(ok_rows)

# =============================================================================
# Final 2 x 2 publication figure
# =============================================================================

floor = 1e-18
fig, axes = plt.subplots(2, 2, figsize=(9.6, 7.2), facecolor="white")
axA, axB, axC, axD = axes.ravel()

# -----------------------------------------------------------------------------
# A/B. Exact backend agreement: covariance and mean/photon-number errors
# -----------------------------------------------------------------------------

axA.hist(np.log10(np.maximum([r["gaussian_vs_piquasso_covariance_error"] for r in rows], floor)),
         bins=12, color="blue", edgecolor="white")
axA.axvline(np.log10(EXACT_TOL), color="black", linestyle="--", linewidth=0.8, label="Tolerance")
style_axis(axA, r"$\log_{10}$(covariance error)", "Count",
           "Exact backends: NumPy vs. Piquasso", "A")
axA.legend(frameon=False, loc="best")

axB.hist(np.log10(np.maximum([r["gaussian_vs_piquasso_mean_error"] for r in rows], floor)),
         bins=12, color="red", alpha=0.85, label="Mean error")
axB.hist(np.log10(np.maximum([r["gaussian_vs_piquasso_photon_number_error"] for r in rows], floor)),
         bins=12, color="magenta", alpha=0.55, label=r"$\langle n\rangle$ error")
axB.axvline(np.log10(EXACT_TOL), color="black", linestyle="--", linewidth=0.8, label="Tolerance")
style_axis(axB, r"$\log_{10}$(error)", "Count",
           "Exact backends: mean & photon number", "B")
axB.legend(frameon=False, loc="best", fontsize=7)

# -----------------------------------------------------------------------------
# C/D. Mixed-Fock APPROXIMATION at finite cutoff (looser tolerance, reported)
# -----------------------------------------------------------------------------

if ok_rows:
    axC.hist(np.log10(np.maximum([r["mixed_fock_photon_number_error"] for r in ok_rows], floor)),
             bins=12, color="cyan", edgecolor="white")
style_axis(axC, r"$\log_{10}$(photon-number error)", "Count",
           f"Mixed-Fock ($c={FOCK_CUTOFF}$) vs. exact Gaussian", "C")

if ok_rows:
    axD.hist(np.log10(np.maximum([r["mixed_fock_q_error"] for r in ok_rows], floor)),
             bins=12, color="orange", edgecolor="white")
style_axis(axD, r"$\log_{10}(q\text{-mean error})$", "Count",
           f"Mixed-Fock ($n_{{ok}}$={len(ok_rows)}/{len(rows)})", "D")

fig.suptitle("Backend cross-checks: exact agreement (top) vs. finite-cutoff approximation (bottom)",
             fontsize=10.5, fontweight="bold", y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.95], h_pad=2.2, w_pad=2.2)
common.save_figure(fig, "R18_gaussian_backends")
plt.close(fig)

common.print_summary(
    "R18 Gaussian backend cross-check",
    samples=len(rows),
    max_exact_covariance_error=max(r["gaussian_vs_piquasso_covariance_error"] for r in rows),
    mixed_fock_successes=len(ok_rows),
    mixed_fock_guard_triggered=n_guard_triggered,
    median_mixed_fock_photon_error=float(np.median([r["mixed_fock_photon_number_error"] for r in ok_rows])) if ok_rows else None,
)


## R19 — Photon Subtraction: Exact Binomial Heralding Probability

R19 validates the package's *physical* photon-subtraction primitive
(`non_gaussian.photon_subtraction`) against an exact, independently-derived
closed form — the strongest kind of non-Gaussian check in this suite because
the reference is not a numerical approximation but an exact combinatorial
identity.

### The physical model

A vacuum-tap beamsplitter of angle \(\theta\) couples the signal mode
(prepared in Fock state \(|n\rangle\)) to a vacuum ancilla; the ancilla is
then photon-counted. Because a beamsplitter is **number-conserving**, and the
ancilla starts in vacuum, detecting \(k\) photons on the ancilla projects the
signal onto exactly \(n-k\) photons — this is a *physical* (finite-tap,
number-resolving) heralded subtraction, not the ideal, unphysical annihilation
operator \(\hat a\) (`docs/non_gaussian_design.md`).

### Exact heralding probability

For input \(|n\rangle\) through a beamsplitter of transmissivity
\(\cos^2\theta\), detecting \(k\) ancilla photons has probability

$$
P(k\mid n,\theta) = \binom{n}{k}\,\sin^{2k}\theta\,\cos^{2(n-k)}\theta,
$$

an exact binomial distribution (SciPy `comb`, computed independently of
`photographiq.non_gaussian`) — the same combinatorics as independently
splitting \(n\) indistinguishable photons into two number-conserving output
ports with per-photon "success" probability \(\sin^2\theta\).

### What is checked

For \(n\in\{1,2,3\}\), a sweep of tap angles \(\theta\in(0,\pi/2)\), and every
possible heralded count \(k=0,\dots,n\) (135 total cases): the simulated
heralding probability against \(P(k\mid n,\theta)\), and the conditional
signal-mode photon number against the exact deterministic prediction
\(n-k\).

### Success criteria

- Absolute probability error \(\lesssim10^{-9}\) for all 135 cases (panels
  A–C: probability curves per \(n\); panel D: the full error distribution
  across \(\theta\)).
- Conditional photon number equals \(n-k\) **exactly** (panel E: measured vs.
  predicted, all points on the diagonal) — this is a number-conservation
  identity, not a statistical estimate.
- Relative error stays at the same machine-precision floor even where the
  probability itself is small (panel F) — confirming the small-probability
  tail is not numerically unstable.


In [ ]:
from __future__ import annotations

import numpy as np
from scipy.special import comb

import photographiq as pg

set_reference_style()

THETAS = np.linspace(0.05, np.pi / 2 - 0.05, 15)
PHOTON_NUMBERS = (1, 2, 3)
CUTOFF = 8
TOL = 1e-9


def analytic_probability(n, k, theta):
    # Exact binomial heralding: P(k|n,theta) = C(n,k) sin(theta)^2k cos(theta)^2(n-k)
    return float(comb(n, k) * np.sin(theta) ** (2 * k) * np.cos(theta) ** (2 * (n - k)))


rows = []
for n in PHOTON_NUMBERS:
    for theta in THETAS:
        pattern = pg.non_gaussian.photon_subtraction(float(theta))
        for k in range(n + 1):
            result = pg.simulate(
                pattern,
                backend="piquasso-fock",
                cutoff=CUTOFF,
                inputs={"in": pg.FockInput.number(n)},
                measurement_outcomes={"count": k},
                seed=0,
            )
            pg_probability = result.measurement_statistics["count"]["value"]
            analytic = analytic_probability(n, k, float(theta))
            conditional_n = result.state.photon_number("in")
            rows.append(
                {
                    "n": n,
                    "theta": float(theta),
                    "k": k,
                    "analytic_probability": analytic,
                    "photographiq_probability": pg_probability,
                    "absolute_error": abs(pg_probability - analytic),
                    "relative_error": common.relative_error(pg_probability, analytic, floor=1e-12),
                    "conditional_photon_number": conditional_n,
                    "expected_conditional_photon_number": n - k,
                    "conditional_photon_number_error": abs(conditional_n - (n - k)),
                }
            )

# -----------------------------------------------------------------------------
# Validation
# -----------------------------------------------------------------------------

failures = [
    r for r in rows
    if r["absolute_error"] > TOL or r["conditional_photon_number_error"] > TOL
]
if failures:
    common.save_csv(failures, "R19_photon_subtraction_probability_FAILURES")
    raise AssertionError(f"{len(failures)} cases disagreed with analytic predictions; see FAILURES csv")

common.save_result(rows, "R19_photon_subtraction_probability", extra={"cutoff": CUTOFF})

# =============================================================================
# Final 2 x 3 publication figure
# =============================================================================

floor = 1e-18
fig, axes = plt.subplots(2, 3, figsize=(13.0, 7.2), facecolor="white")
(axA, axB, axC, axD, axE, axF) = axes.ravel()

# -----------------------------------------------------------------------------
# A-C. Herald probability: analytic binomial vs. simulated, per n
# -----------------------------------------------------------------------------

colors = ["red", "blue", "magenta", "cyan"]
for ax, n, letter in zip((axA, axB, axC), PHOTON_NUMBERS, "ABC", strict=True):
    for k in range(n + 1):
        subset = [r for r in rows if r["n"] == n and r["k"] == k]
        ts = [r["theta"] for r in subset]
        ax.plot(ts, [r["analytic_probability"] for r in subset], "-", color=colors[k % len(colors)],
                label=f"$k$={k} analytic")
        ax.plot(ts, [r["photographiq_probability"] for r in subset], "o", markersize=3,
                color=colors[k % len(colors)], label=f"$k$={k} PhotoGraphiQ")
    style_axis(ax, r"Tap angle $\theta$", "Herald probability" if letter == "A" else "",
               f"$n$={n}", letter)
    ax.legend(fontsize=5.5, frameon=False, ncol=2)

# -----------------------------------------------------------------------------
# D. Absolute probability error across the full sweep
# -----------------------------------------------------------------------------

for n, color in zip(PHOTON_NUMBERS, ("red", "blue", "magenta")):
    subset = [r for r in rows if r["n"] == n]
    axD.semilogy([r["theta"] for r in subset], np.maximum([r["absolute_error"] for r in subset], floor),
                 ".", color=color, alpha=0.6, markersize=4, label=f"$n$={n}")
axD.axhline(TOL, color="black", linestyle="--", linewidth=0.8, label="Tolerance")
style_axis(axD, r"Tap angle $\theta$", "Absolute probability error", "Probability validation", "D")
axD.legend(frameon=False, fontsize=6.5)

# -----------------------------------------------------------------------------
# E. Exact number-conservation identity: conditional n = n - k
# -----------------------------------------------------------------------------

axE.plot([r["expected_conditional_photon_number"] for r in rows],
         [r["conditional_photon_number"] for r in rows], "o", color="red", markersize=4, alpha=0.5)
lims = [-0.3, max(PHOTON_NUMBERS) + 0.3]
axE.plot(lims, lims, "--", color="black", linewidth=0.8, label="$n-k$ (exact)")
axE.set_xlim(lims)
axE.set_ylim(lims)
style_axis(axE, "Predicted conditional photon number $n-k$", "Measured conditional photon number",
           "Number conservation", "E")
axE.legend(frameon=False, loc="best")

# -----------------------------------------------------------------------------
# F. Relative error stays at the numerical floor even for small probabilities
# -----------------------------------------------------------------------------

analytic_vals = np.array([r["analytic_probability"] for r in rows])
rel_errs = np.maximum([r["relative_error"] for r in rows], floor)
axF.semilogy(analytic_vals, rel_errs, "o", color="blue", markersize=4, alpha=0.5)
axF.axhline(TOL, color="black", linestyle="--", linewidth=0.8, label="Tolerance")
style_axis(axF, "Analytic probability", "Relative error", "Small-probability stability", "F")
axF.legend(frameon=False, loc="best")

fig.suptitle("Photon subtraction: exact binomial heralding probability",
             fontsize=11, fontweight="bold", y=0.99)
fig.tight_layout(rect=[0, 0, 1, 0.96], h_pad=2.4, w_pad=2.2)
common.save_figure(fig, "R19_photon_subtraction_probability")
plt.close(fig)

common.print_summary(
    "R19 photon subtraction analytic probability",
    cases=len(rows),
    max_absolute_error=max(r["absolute_error"] for r in rows),
    max_conditional_photon_error=max(r["conditional_photon_number_error"] for r in rows),
)


## R20 — Photon Subtraction on Squeezed Vacuum (Even-to-Odd Parity Flip)

R20 exercises the same physical subtraction primitive as R19 on a genuinely
non-classical Gaussian input, and checks a qualitative, unambiguous physical
signature rather than a numerical curve: single-photon heralded subtraction
should flip squeezed vacuum's photon-number parity from even to odd — the
same mechanism that produces optical cat states from squeezed light.

### Why parity must flip

\(q\)-squeezed vacuum \(\hat S(-r)|0\rangle\) has support **only** on even
photon numbers (it is built from vacuum by the two-photon squeezing
operator), so its parity \(\langle(-1)^{\hat n}\rangle=+1\). A physical,
weak-tap (\(\theta\) small) heralded subtraction of one photon acts
approximately as \(\hat a\,|\psi\rangle\) restricted to the successfully
detected branch; removing one photon from a purely-even distribution leaves
support only on **odd** photon numbers, so \(\langle(-1)^{\hat n}\rangle$
should flip sign to \(-1\). This is the standard route to preparing optical
Schrödinger-cat-like states from squeezed light, and it is checked here as a
qualitative pass/fail (sign flip), not a fitted curve.

### What is checked

1. Fock-basis photon-number distribution before/after subtraction (should
   move from purely-even to odd-dominant support).
2. Parity before (\(\approx+1\)) and after (\(\approx-1\)) subtraction —
   the pass/fail criterion.
3. Wigner functions before/after (the negativity that appears after
   subtraction is itself a signature of non-classicality).
4. **Finite-cutoff convergence** of this *specific conditional branch*
   (`count=1` fixed at every tested cutoff, so all cutoffs compare the same
   heralded outcome): fidelity and trace distance to the previous, coarser
   cutoff, and the retained-norm safety margin, as the Fock cutoff is
   increased from 12 to 36.

### Success criteria

- Parity flips sign (before \(>0.5\), after \(<-0.5\)) — panel A/summary.
- The post-subtraction Wigner function shows visible negative regions
  (panel C) — the standard non-classicality signature.
- Fidelity-to-previous-cutoff climbs toward 1 and trace-distance-to-previous
  falls toward 0 as the cutoff grows (panel D) — the truncated calculation
  has converged, not merely been evaluated once.


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg

set_reference_style()

SQUEEZING_R = 0.6
THETA = 0.15  # small tap angle: closer to ideal (low-attenuation) subtraction
CUTOFF = 24
CONVERGENCE_CUTOFFS = [12, 16, 20, 28, 36]

squeezed_input = pg.GaussianInput.squeezed(-SQUEEZING_R)  # q-squeezed: even support

before_pattern = pg.Pattern(inputs=("in",)).append(pg.Output(("in",))).validate()
before = pg.simulate(
    before_pattern, backend="piquasso-fock", cutoff=CUTOFF, inputs={"in": squeezed_input}, seed=0
).state

subtraction_pattern = pg.non_gaussian.photon_subtraction(THETA)
after_result = pg.simulate(
    subtraction_pattern,
    backend="piquasso-fock",
    cutoff=CUTOFF,
    inputs={"in": squeezed_input},
    measurement_outcomes={"count": 1},
    seed=0,
)
after = after_result.state

before_parity = before.parity()
after_parity = after.parity()
before_mean_n = before.photon_number("in")
after_mean_n = after.photon_number("in")

# Qualitative, unambiguous physical signature: even -> odd parity flip.
parity_flipped = before_parity > 0.5 and after_parity < -0.5
if not parity_flipped:
    raise AssertionError(
        f"Expected an even-to-odd parity flip; got before={before_parity:.4f}, after={after_parity:.4f}"
    )

before_probs = before.probabilities
after_probs = after.probabilities
max_n = max(max(b[0] for b in before_probs), max(b[0] for b in after_probs)) + 1
distribution_rows = [
    {
        "n": n,
        "before_probability": before_probs.get((n,), 0.0),
        "after_probability": after_probs.get((n,), 0.0),
    }
    for n in range(max_n)
]

summary = {
    "squeezing_r": SQUEEZING_R,
    "tap_theta": THETA,
    "cutoff": CUTOFF,
    "before_parity": before_parity,
    "after_parity": after_parity,
    "before_mean_photon_number": before_mean_n,
    "after_mean_photon_number": after_mean_n,
    "herald_probability": after_result.measurement_statistics["count"]["value"],
    "parity_flipped_even_to_odd": parity_flipped,
}
common.save_json(summary, "R20_subtracted_squeezed_state_summary")
common.save_csv(distribution_rows, "R20_subtracted_squeezed_state_distribution")
common.write_metadata("R20_subtracted_squeezed_state")

q_axis = np.linspace(-5, 5, 121)
p_axis = np.linspace(-5, 5, 121)
before_wigner = before.wigner(q_axis, p_axis, node="in")
after_wigner = after.wigner(q_axis, p_axis, node="in")

# Cutoff convergence for this fixed conditional branch (measurement_outcomes
# fixes count=1 so every cutoff compares the same heralded branch).
study = pg.cutoff_convergence(
    subtraction_pattern,
    CONVERGENCE_CUTOFFS,
    backend="piquasso-fock",
    inputs={"in": squeezed_input},
    measurement_outcomes={"count": 1},
)
convergence_rows = [
    {
        "cutoff": row["cutoff"],
        "norm": row["norm"],
        "minimum_retained_norm": row["minimum_retained_norm"],
        "maximum_boundary_population": row["maximum_boundary_population"],
        "fidelity_to_previous": row["fidelity_to_previous"],
        "trace_distance_to_previous": row["trace_distance_to_previous"],
        "comparable_to_previous": row["comparable_to_previous"],
    }
    for row in study.rows
]
common.save_csv(convergence_rows, "R20_subtracted_squeezed_state_convergence")

# =============================================================================
# Final 2 x 2 publication figure
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(10.4, 9.0), facecolor="white")
axA, axB, axC, axD = axes.ravel()

# -----------------------------------------------------------------------------
# A. Fock distribution: purely even (before) -> odd-dominant (after)
# -----------------------------------------------------------------------------

width = 0.35
ns = [r["n"] for r in distribution_rows]
axA.bar(np.array(ns) - width / 2, [r["before_probability"] for r in distribution_rows], width,
        label=f"Before ($\\langle(-1)^n\\rangle$={before_parity:.2f})", color="blue")
axA.bar(np.array(ns) + width / 2, [r["after_probability"] for r in distribution_rows], width,
        label=f"After ($\\langle(-1)^n\\rangle$={after_parity:.2f})", color="red")
style_axis(axA, "Photon number $n$", "Probability", "Fock distribution: even to odd", "A")
axA.legend(frameon=False, fontsize=7.5)

# -----------------------------------------------------------------------------
# B/C. Wigner functions: negativity appears only after subtraction
# -----------------------------------------------------------------------------

for ax, grid, title, letter in ((axB, before_wigner, "Before (squeezed vacuum)", "B"),
                                 (axC, after_wigner, "After subtraction (odd, non-classical)", "C")):
    extent = max(abs(grid.values.min()), abs(grid.values.max()))
    mesh = ax.pcolormesh(grid.q, grid.p, grid.values, shading="auto", cmap="RdBu_r", vmin=-extent, vmax=extent)
    ax.set_aspect("equal")
    style_axis(ax, "$q$", "$p$", title, letter)
    fig.colorbar(mesh, ax=ax, fraction=0.046)

# -----------------------------------------------------------------------------
# D. Finite-cutoff convergence of this conditional branch
# -----------------------------------------------------------------------------

# The first (baseline) cutoff has no predecessor to compare against, so
# fidelity/trace_distance_to_previous is None there -- plot from the second
# cutoff onward.
comparable = [r for r in convergence_rows if r["fidelity_to_previous"] is not None]
cutoffs = [r["cutoff"] for r in comparable]
axD.plot(cutoffs, [r["fidelity_to_previous"] for r in comparable], "o-", color="red",
         label="Fidelity to previous cutoff")
axD_twin = axD.twinx()
axD_twin.semilogy(cutoffs, np.maximum([r["trace_distance_to_previous"] for r in comparable], 1e-16),
                   "s--", color="blue", label="Trace distance to previous")
axD_twin.set_ylabel("Trace distance", color="blue")
axD_twin.tick_params(axis="y", colors="blue")
style_axis(axD, "Fock cutoff", "Fidelity to previous cutoff",
           "Conditional-branch convergence", "D")
lines1, labels1 = axD.get_legend_handles_labels()
lines2, labels2 = axD_twin.get_legend_handles_labels()
axD.legend(lines1 + lines2, labels1 + labels2, frameon=False, fontsize=7, loc="center right")

fig.suptitle(f"Photon subtraction on squeezed vacuum ($r$={SQUEEZING_R}, $\\theta$={THETA})",
             fontsize=11, fontweight="bold", y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.97], h_pad=2.4, w_pad=2.4)
common.save_figure(fig, "R20_subtracted_squeezed_state")
plt.close(fig)

common.print_summary(
    "R20 photon subtraction on squeezed vacuum",
    before_parity=before_parity,
    after_parity=after_parity,
    parity_flipped=parity_flipped,
    final_fidelity_to_previous_cutoff=convergence_rows[-1]["fidelity_to_previous"],
)


## R21 — Cat States: Independent Coherent-State Expansion

R21 checks the package's finite-Fock cat-state resource (`CatResource`,
internally `FockInput.cat`) against an amplitude-level reference built with
zero shared code: an explicit NumPy expansion of \(|\alpha\rangle\pm|-\alpha\rangle\)
in the Fock basis, rather than reusing `FockInput.cat` itself.

### Independent reference construction

A coherent state has the well-known Fock expansion \(|\alpha\rangle=
e^{-|\alpha|^2/2}\sum_n \frac{\alpha^n}{\sqrt{n!}}|n\rangle\), so an even/odd
cat state is, up to normalization,

$$
|\text{cat}_\pm(\alpha)\rangle \;\propto\; |\alpha\rangle \pm |-\alpha\rangle
\;=\; e^{-\alpha^2/2}\sum_n \frac{\alpha^n}{\sqrt{n!}}\big(1\pm(-1)^n\big)\,|n\rangle,
$$

computed here directly in log-space (`gammaln`) for numerical stability, and
normalized independently of the package's own normalization path. Because
\(1\pm(-1)^n\) vanishes for odd/even \(n\) respectively, the even cat has
**only even** photon-number support and the odd cat only odd support — the
defining parity signature checked quantitatively here (not just in R20's
qualitative flip).

### What is checked

For \(\alpha\in\{0.5,1.0,1.5,2.0\}\) and both parities, at Fock cutoff 32:

1. **State fidelity** \(|\langle\psi_{\text{analytic}}|\psi_{\text{pg}}\rangle|^2\)
   against the independent expansion above.
2. **Parity sign**: \(\operatorname{sign}\langle(-1)^{\hat n}\rangle\) must
   match the resource's declared parity exactly.
3. Mean photon number, retained norm and boundary population (finite-cutoff
   truncation diagnostics) across the full \(\alpha\) sweep.

### Success criteria

- Fidelity \(>1-10^{-6}\) for every \((\alpha,\text{parity})\) pair (panel C)
  — an amplitude-level agreement, not just a matching first moment.
- Parity sign matches exactly for all 8 cases (panel D).
- Mean photon number grows with \(\alpha\) as expected for a coherent-like
  superposition (panel E), and the retained Fock-truncation norm/boundary
  population stay controlled even at the largest tested \(\alpha=2\) (panel
  F) — confirming cutoff 32 is adequate over the whole sweep, not just at
  small \(\alpha\).
- Representative Wigner functions (panels A, B) show the expected even/odd
  interference-fringe structure at \(\alpha=1.5\).


In [ ]:
from __future__ import annotations

import numpy as np
from scipy.special import gammaln

import photographiq as pg

set_reference_style()

ALPHAS = [0.5, 1.0, 1.5, 2.0]
CUTOFF = 32
REPRESENTATIVE_ALPHA = 1.5
FIDELITY_TOL = 1e-6


def analytic_cat(alpha: float, cutoff: int, parity: int) -> np.ndarray:
    """Independent NumPy |alpha>+parity|-alpha> expansion (not FockInput.cat)."""
    n = np.arange(cutoff)
    log_coherent = n * np.log(alpha) - gammaln(n + 1) / 2 - 0.5 * alpha**2 if alpha > 0 else None
    if alpha == 0:
        vector = np.zeros(cutoff)
        vector[0] = 1 + parity
    else:
        vector = np.exp(log_coherent) * (1 + parity * (-1.0) ** n)
    norm = np.linalg.norm(vector)
    return vector / norm


rows = []
wigner_states = {}
for alpha in ALPHAS:
    for parity in (1, -1):
        resource = pg.CatResource(alpha, parity=parity)
        pattern = pg.Pattern(inputs=()).append(pg.Prepare(0, state=resource)).append(pg.Output((0,))).validate()
        result = pg.simulate(pattern, backend="piquasso-fock", cutoff=CUTOFF, seed=0)
        state = result.state

        analytic = analytic_cat(alpha, CUTOFF, parity)
        actual_amplitudes = dict(zip(state.basis, state.state_vector, strict=True))
        actual_vector = np.array([actual_amplitudes.get((n,), 0j) for n in range(CUTOFF)])
        overlap = np.vdot(analytic, actual_vector)
        fidelity = float(abs(overlap) ** 2)

        expected_parity = float(parity)
        actual_parity = state.parity()

        rows.append(
            {
                "alpha": alpha,
                "parity_label": "even" if parity == 1 else "odd",
                "fidelity_vs_independent_analytic": fidelity,
                "expected_ideal_parity": expected_parity,
                "measured_parity": actual_parity,
                "mean_photon_number": state.photon_number(0),
                "norm": state.norm,
                "retained_norm": state.retained_norms[-1] if state.retained_norms else None,
                "boundary_population": state.diagnostics[-1]["boundary_population"] if state.diagnostics else None,
            }
        )
        if alpha == REPRESENTATIVE_ALPHA:
            wigner_states[parity] = state

# -----------------------------------------------------------------------------
# Validation
# -----------------------------------------------------------------------------

parity_sign_failures = [r for r in rows if np.sign(r["measured_parity"]) != np.sign(r["expected_ideal_parity"])]
fidelity_failures = [r for r in rows if r["fidelity_vs_independent_analytic"] < 1 - FIDELITY_TOL]
if parity_sign_failures or fidelity_failures:
    common.save_csv(parity_sign_failures + fidelity_failures, "R21_cat_state_FAILURES")
    raise AssertionError("Cat state parity sign or independent-fidelity check failed; see FAILURES csv")

common.save_result(rows, "R21_cat_state", extra={"cutoff": CUTOFF})

# =============================================================================
# Final 2 x 3 publication figure
# =============================================================================

q_axis = np.linspace(-6, 6, 121)
p_axis = np.linspace(-6, 6, 121)

fig, axes = plt.subplots(2, 3, figsize=(13.4, 8.0), facecolor="white")
(axA, axB, axC, axD, axE, axF) = axes.ravel()

# -----------------------------------------------------------------------------
# A/B. Representative Wigner functions: even/odd interference fringes
# -----------------------------------------------------------------------------

for ax, (parity, label), letter in zip((axA, axB), ((1, "Even cat"), (-1, "Odd cat")), "AB"):
    grid = wigner_states[parity].wigner(q_axis, p_axis, node=0)
    extent = max(abs(grid.values.min()), abs(grid.values.max()))
    mesh = ax.pcolormesh(grid.q, grid.p, grid.values, shading="auto", cmap="RdBu_r", vmin=-extent, vmax=extent)
    ax.set_aspect("equal")
    style_axis(ax, "$q$", "$p$", f"{label}, $\\alpha$={REPRESENTATIVE_ALPHA}", letter)
    fig.colorbar(mesh, ax=ax, fraction=0.046)

# -----------------------------------------------------------------------------
# C. Fidelity vs. independent analytic expansion (amplitude-level agreement)
# -----------------------------------------------------------------------------

for parity, label, color, marker in ((1, "Even", "red", "o"), (-1, "Odd", "blue", "s")):
    subset = [r for r in rows if r["expected_ideal_parity"] == parity]
    axC.plot([r["alpha"] for r in subset],
             [1 - r["fidelity_vs_independent_analytic"] for r in subset],
             marker + "-", color=color, label=label)
axC.set_yscale("log")
axC.axhline(FIDELITY_TOL, color="black", linestyle="--", linewidth=0.8, label="Tolerance")
style_axis(axC, r"$\alpha$", "$1-$fidelity", "Fidelity vs. independent expansion", "C")
axC.legend(frameon=False, fontsize=7)

# -----------------------------------------------------------------------------
# D. Parity sign check across the full sweep
# -----------------------------------------------------------------------------

for parity, label, color, marker in ((1, "Even (expect +1)", "red", "o"), (-1, "Odd (expect -1)", "blue", "s")):
    subset = [r for r in rows if r["expected_ideal_parity"] == parity]
    axD.plot([r["alpha"] for r in subset], [r["measured_parity"] for r in subset],
              marker + "-", color=color, label=label)
axD.axhline(1.0, color="black", linestyle=":", linewidth=0.8)
axD.axhline(-1.0, color="black", linestyle=":", linewidth=0.8)
style_axis(axD, r"$\alpha$", r"$\langle(-1)^{\hat n}\rangle$", "Parity sign check", "D")
axD.legend(frameon=False, fontsize=7)

# -----------------------------------------------------------------------------
# E. Mean photon number grows with alpha (physical sanity)
# -----------------------------------------------------------------------------

for parity, label, color, marker in ((1, "Even", "red", "o"), (-1, "Odd", "blue", "s")):
    subset = [r for r in rows if r["expected_ideal_parity"] == parity]
    axE.plot([r["alpha"] for r in subset], [r["mean_photon_number"] for r in subset],
              marker + "-", color=color, label=label)
style_axis(axE, r"$\alpha$", r"$\langle n\rangle$", "Mean photon number", "E")
axE.legend(frameon=False, fontsize=7)

# -----------------------------------------------------------------------------
# F. Finite-cutoff truncation diagnostics across the sweep
# -----------------------------------------------------------------------------

for parity, label, color, marker in ((1, "Even", "red", "o"), (-1, "Odd", "blue", "s")):
    subset = [r for r in rows if r["expected_ideal_parity"] == parity]
    axF.plot([r["alpha"] for r in subset], [r["retained_norm"] for r in subset],
              marker + "-", color=color, label=f"{label}: retained norm")
axF.axhline(1.0, color="black", linestyle=":", linewidth=0.8)
style_axis(axF, r"$\alpha$", "Retained Fock-truncation norm",
           f"Truncation adequacy (cutoff={CUTOFF})", "F")
axF.legend(frameon=False, fontsize=7)

fig.suptitle("Cat-state resources: independent amplitude-level validation",
             fontsize=11, fontweight="bold", y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.97], h_pad=2.4, w_pad=2.4)
common.save_figure(fig, "R21_cat_state")
plt.close(fig)

common.print_summary(
    "R21 cat state",
    cases=len(rows),
    min_fidelity_to_analytic=min(r["fidelity_vs_independent_analytic"] for r in rows),
    all_parity_signs_correct=not parity_sign_failures,
)


## R22 — Native Cubic Phase Gate vs. Dense Matrix Exponential

R22 checks the native `CubicPhase(gamma)` command — Piquasso's implementation
of the genuinely non-Gaussian gate \(\exp(i\gamma\hat q^3/6)\) — against a
completely independent, brute-force reference: a dense truncated ladder
operator built directly with NumPy, exponentiated with SciPy. This is a
finite-cutoff-**matched** comparison (both sides truncated at the same
dimension), not an infinite-cutoff convergence claim (see R33/R34 for that).

### Independent reference

At truncation dimension \(c\), build the ladder operator
\(a_{jk}=\sqrt{k}\,\delta_{j,k-1}\) (\(j,k=0,\dots,c-1\)), the quadrature
\(\hat q=a+a^{T}\) (\(\hbar=2\) convention), and

$$
U_{\text{ref}}(\gamma,c) = \exp\!\Big(\tfrac{i\gamma}{6}\hat q^{3}\Big)
\quad\text{(SciPy \texttt{expm}, dense matrix exponential)}.
$$

Applied to the *same* truncated coherent-state input vector
\(|\psi_{\text{in}}\rangle\) (extracted from PhotoGraphiQ's own Fock backend
at that cutoff, so both sides start from bit-identical truncated amplitudes),
this gives an independent target
\(|\psi_{\text{ref}}\rangle=U_{\text{ref}}(\gamma,c)|\psi_{\text{in}}\rangle\)
to compare against the native gate's actual output \(|\psi_{\text{pg}}\rangle\).

### What is checked

For \(\gamma\in\{0.02,0.05,0.1,0.2\}\) and cutoffs \(c\in\{16,24,32,48\}\):
state fidelity \(|\langle\psi_{\text{ref}}|\psi_{\text{pg}}\rangle|^2\), a
phase-aligned state-vector error, and the Fock-truncation retained norm.

### Success criteria

- Fidelity approaches 1 as the cutoff grows for every \(\gamma\) (panel A);
  the low-energy coherent input (\(\alpha=0.3+0.1i\)) means even the smallest
  tested cutoff should already be close.
- The fidelity landscape over the full \((\gamma,c)\) grid (panel B) shows no
  isolated failure — larger \(\gamma\) (stronger nonlinearity, more spread
  into higher photon numbers) requires a larger cutoff to converge, and the
  heatmap should show that trend smoothly, not abruptly.
- Phase-aligned state-vector error (panel C) tracks \(\sqrt{1-\text{fidelity}}\)
  consistently, and retained norm (panel D) stays near 1 at every tested
  cutoff — confirming truncation, not gate-implementation error, is the only
  limiting factor.


In [ ]:
from __future__ import annotations

import numpy as np
from scipy.linalg import expm

import photographiq as pg

set_reference_style()

GAMMAS = [0.02, 0.05, 0.1, 0.2]
CUTOFFS = [16, 24, 32, 48]
ALPHA = 0.3 + 0.1j


def independent_cubic_unitary(gamma: float, cutoff: int) -> np.ndarray:
    """Independent dense NumPy/SciPy reference: exp(i*gamma*q^3/6) at fixed cutoff."""
    a = np.diag(np.sqrt(np.arange(1, cutoff)), k=1)
    q = a + a.T
    return expm(1j * gamma * (q @ q @ q) / 6)


def truncated_input_vector(cutoff: int) -> np.ndarray:
    pattern = pg.Pattern(inputs=(0,)).append(pg.Output((0,))).validate()
    state = pg.simulate(
        pattern, backend="piquasso-fock", cutoff=cutoff, inputs={0: pg.GaussianInput.coherent(ALPHA)}, seed=0
    ).state
    amplitudes = dict(zip(state.basis, state.state_vector, strict=True))
    return np.array([amplitudes.get((n,), 0j) for n in range(cutoff)])


rows = []
for cutoff in CUTOFFS:
    before = truncated_input_vector(cutoff)
    for gamma in GAMMAS:
        pattern = pg.Pattern(inputs=(0,)).append(pg.CubicPhase(0, gamma)).append(pg.Output((0,))).validate()
        state = pg.simulate(
            pattern, backend="piquasso-fock", cutoff=cutoff, inputs={0: pg.GaussianInput.coherent(ALPHA)}, seed=0
        ).state
        amplitudes = dict(zip(state.basis, state.state_vector, strict=True))
        actual = np.array([amplitudes.get((n,), 0j) for n in range(cutoff)])

        unitary = independent_cubic_unitary(gamma, cutoff)
        predicted = unitary @ before
        predicted /= np.linalg.norm(predicted)
        overlap = np.vdot(predicted, actual)
        fidelity = float(np.clip(abs(overlap) ** 2, 0, 1))
        state_vector_norm_error = float(np.linalg.norm(actual - predicted * (overlap.conjugate() / abs(overlap))))

        rows.append(
            {
                "gamma": gamma,
                "cutoff": cutoff,
                "fidelity_to_independent_reference": fidelity,
                "state_vector_norm_error_phase_aligned": state_vector_norm_error,
                "trace_distance_estimate": float(np.sqrt(max(0.0, 1 - fidelity))),
                "retained_norm": state.retained_norms[-1] if state.retained_norms else None,
            }
        )

# -----------------------------------------------------------------------------
# Validation: at the LARGEST tested cutoff, truncation error should be
# negligible for this low-energy input, so fidelity to the independent
# reference must be near-unit. Smaller cutoffs are not held to this bound --
# their fidelity legitimately depends on how much gamma spreads the state
# into higher photon numbers (panel B).
# -----------------------------------------------------------------------------

FIDELITY_TOL = 0.999
largest_cutoff = max(CUTOFFS)
failures = [
    r for r in rows
    if r["cutoff"] == largest_cutoff and r["fidelity_to_independent_reference"] < FIDELITY_TOL
]
if failures:
    common.save_csv(failures, "R22_cubic_phase_direct_FAILURES")
    raise AssertionError(
        f"{len(failures)} cases at the largest cutoff ({largest_cutoff}) fell below "
        f"fidelity {FIDELITY_TOL}; see FAILURES csv"
    )

common.save_result(rows, "R22_cubic_phase_direct", extra={"alpha": [ALPHA.real, ALPHA.imag]})

# =============================================================================
# Final 2 x 2 publication figure
# =============================================================================

floor = 1e-16
fig, axes = plt.subplots(2, 2, figsize=(10.0, 7.4), facecolor="white")
axA, axB, axC, axD = axes.ravel()

# -----------------------------------------------------------------------------
# A. Infidelity vs cutoff, per gamma
# -----------------------------------------------------------------------------

colors = {0.02: "red", 0.05: "blue", 0.1: "magenta", 0.2: "cyan"}
for gamma in GAMMAS:
    subset = [r for r in rows if r["gamma"] == gamma]
    axA.semilogy([r["cutoff"] for r in subset],
                 np.maximum([1 - r["fidelity_to_independent_reference"] for r in subset], floor),
                 "o-", color=colors[gamma], label=f"$\\gamma$={gamma}")
style_axis(axA, "Fock cutoff", "1 - fidelity", "Convergence vs. cutoff", "A")
axA.legend(frameon=False, fontsize=7)

# -----------------------------------------------------------------------------
# B. Fidelity heatmap over the full (gamma, cutoff) grid
# -----------------------------------------------------------------------------

grid = np.array([[next(r["fidelity_to_independent_reference"] for r in rows
                        if r["gamma"] == g and r["cutoff"] == c) for g in GAMMAS] for c in CUTOFFS])
im = axB.imshow(grid, cmap="viridis", vmin=0.9, vmax=1.0, aspect="auto", origin="lower")
axB.set_xticks(range(len(GAMMAS)))
axB.set_xticklabels(GAMMAS)
axB.set_yticks(range(len(CUTOFFS)))
axB.set_yticklabels(CUTOFFS)
style_axis(axB, r"$\gamma$", "Fock cutoff", "Fidelity landscape", "B")
fig.colorbar(im, ax=axB, fraction=0.046, label="Fidelity")

# -----------------------------------------------------------------------------
# C. Phase-aligned state-vector error vs cutoff
# -----------------------------------------------------------------------------

for gamma in GAMMAS:
    subset = [r for r in rows if r["gamma"] == gamma]
    axC.semilogy([r["cutoff"] for r in subset],
                 np.maximum([r["state_vector_norm_error_phase_aligned"] for r in subset], floor),
                 "o-", color=colors[gamma], label=f"$\\gamma$={gamma}")
style_axis(axC, "Fock cutoff", "Phase-aligned state-vector error", "Amplitude-level error", "C")
axC.legend(frameon=False, fontsize=7)

# -----------------------------------------------------------------------------
# D. Retained Fock-truncation norm vs cutoff
# -----------------------------------------------------------------------------

for gamma in GAMMAS:
    subset = [r for r in rows if r["gamma"] == gamma]
    axD.plot([r["cutoff"] for r in subset], [r["retained_norm"] for r in subset],
              "o-", color=colors[gamma], label=f"$\\gamma$={gamma}")
axD.axhline(1.0, color="black", linestyle=":", linewidth=0.8)
style_axis(axD, "Fock cutoff", "Retained norm", "Truncation adequacy", "D")
axD.legend(frameon=False, fontsize=7)

fig.suptitle(f"Native CubicPhase vs. dense matrix exponential (coherent input $\\alpha$={ALPHA})",
             fontsize=10.5, fontweight="bold", y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.96], h_pad=2.4, w_pad=2.4)
common.save_figure(fig, "R22_cubic_phase_direct")
plt.close(fig)

common.print_summary(
    "R22 cubic phase direct execution",
    cases=len(rows),
    min_fidelity=min(r["fidelity_to_independent_reference"] for r in rows),
    max_fidelity=max(r["fidelity_to_independent_reference"] for r in rows),
)


## R23 — Finite-Resource Cubic MBQC Injection

R22 tested the *native* `CubicPhase` gate directly. R23 tests how well the
gate can be **synthesized inside MBQC** by finite-squeezing resource
injection (`non_gaussian.cubic_injection`) — a fundamentally different,
harder question: injection is a *conditional filter times a cubic phase*, not
an exact finite-energy unitary (`docs/non_gaussian.md`), so it is expected to
only approach the ideal gate as the resource squeezing grows.

### Isolating the resource-envelope error

`cubic_injection(gamma, squeezing=r)` measures an ancilla and applies
outcome-dependent corrections `QuadraticPhase(-2*gamma*m)` and
`Displace(p=-gamma*m^2)`. Fixing the ancilla outcome at exactly \(m=0\) makes
both corrections vanish identically, isolating the **residual finite-resource
Gaussian envelope error** — the part of the injection error that comes purely
from the ancilla's finite squeezing, decoupled from feed-forward correction
bookkeeping (which is tested structurally in R11–R14).

### What is checked

Fidelity of the injected gate's output to the same independent
matrix-exponential ideal-`CubicPhase` reference as R22
(\(\gamma=0.15\), coherent input \(\alpha=0.2\)), across resource squeezing
\(r\in\{0.1,\dots,2.0\}\) and Fock cutoffs \(\{24,32,48\}\). Because larger
\(r\) and sharper ancillas can occasionally hit the Fock backend's own
retained-norm safety guard (a real, physical resource limit, not a bug — the
same mechanism documented in `ISSUES_FOUND.md` #2), guard-triggered cases are
recorded as `ok=False` with the guard's message, never silently dropped.

### Success criteria

- Fidelity to the ideal `CubicPhase(gamma)` **increases monotonically** (up to
  sampling/cutoff noise) with resource squeezing \(r\) — the defining
  signature of a finite-resource approximation converging toward its
  infinite-squeezing limit (panel A).
- Retained norm stays controlled across the successful cases (panel B); any
  guard-triggered cases are counted explicitly, not hidden (panel C).
- Trace distance to the ideal gate (panel D, log scale) shrinks with \(r\)
  more clearly than raw fidelity does near 1.

### What this experiment does *not* claim

Per `MANUSCRIPT_MAP.md`: this is a finite-resource **approximation**; its
fidelities are not a claim about an exact or fault-tolerant cubic gate.


In [ ]:
from __future__ import annotations

import numpy as np
from scipy.linalg import expm

import photographiq as pg

set_reference_style()

GAMMA = 0.15
ALPHA = 0.2 + 0.0j
SQUEEZING_VALUES = [0.1, 0.3, 0.6, 1.0, 1.5, 2.0]
CUTOFFS = [24, 32, 48]
FIXED_OUTCOME = 0.0


def independent_cubic_unitary(gamma: float, cutoff: int) -> np.ndarray:
    a = np.diag(np.sqrt(np.arange(1, cutoff)), k=1)
    q = a + a.T
    return expm(1j * gamma * (q @ q @ q) / 6)


def truncated_input_vector(cutoff: int) -> np.ndarray:
    pattern = pg.Pattern(inputs=(0,)).append(pg.Output((0,))).validate()
    state = pg.simulate(
        pattern, backend="piquasso-fock", cutoff=cutoff, inputs={0: pg.GaussianInput.coherent(ALPHA)}, seed=0
    ).state
    amplitudes = dict(zip(state.basis, state.state_vector, strict=True))
    return np.array([amplitudes.get((n,), 0j) for n in range(cutoff)])


def vector_from_state(state, node, cutoff):
    amplitudes = dict(zip(state.basis, state.state_vector, strict=True))
    return np.array([amplitudes.get((n,), 0j) for n in range(cutoff)])


rows = []
for cutoff in CUTOFFS:
    before = truncated_input_vector(cutoff)
    ideal_after = independent_cubic_unitary(GAMMA, cutoff) @ before
    ideal_after /= np.linalg.norm(ideal_after)

    for r in SQUEEZING_VALUES:
        pattern = pg.non_gaussian.cubic_injection(GAMMA, squeezing=r)
        # A fixed ancilla outcome m=0 makes the outcome-dependent corrections
        # QuadraticPhase(-2*gamma*m) and Displace(p=-gamma*m^2) vanish
        # identically, isolating the residual finite-squeezing envelope error.
        failure = common.safe_cutoff_run(
            lambda: pg.simulate(
                pattern, backend="piquasso-fock", cutoff=cutoff,
                inputs={"in": pg.GaussianInput.coherent(ALPHA)},
                measurement_outcomes={"m": FIXED_OUTCOME}, seed=0,
            ),
            label=f"cutoff={cutoff} r={r}",
        )
        if not failure["ok"]:
            rows.append(
                {"gamma": GAMMA, "resource_squeezing": r, "cutoff": cutoff, "ok": False, "error": failure["error"]}
            )
            continue
        result = failure["value"]
        actual = vector_from_state(result.state, "in", cutoff)
        overlap = np.vdot(ideal_after, actual)
        fidelity = float(np.clip(abs(overlap) ** 2, 0, 1))
        rows.append(
            {
                "gamma": GAMMA,
                "resource_squeezing": r,
                "cutoff": cutoff,
                "ok": True,
                "fidelity_to_ideal_cubic_phase": fidelity,
                "trace_distance_estimate": float(np.sqrt(max(0.0, 1 - fidelity))),
                "outcome_density": result.measurement_statistics["m"]["value"],
                "outcome_kind": result.measurement_statistics["m"]["kind"],
                "retained_norm": result.state.retained_norms[-1] if result.state.retained_norms else None,
            }
        )

common.save_result(rows, "R23_cubic_injection", extra={"gamma": GAMMA, "alpha": [ALPHA.real, ALPHA.imag], "fixed_outcome": FIXED_OUTCOME})

ok_rows = [r for r in rows if r["ok"]]
not_ok_rows = [r for r in rows if not r["ok"]]

# =============================================================================
# Final 2 x 2 publication figure
# =============================================================================

floor = 1e-16
colors = {24: "red", 32: "blue", 48: "magenta"}

fig, axes = plt.subplots(2, 2, figsize=(10.0, 7.4), facecolor="white")
axA, axB, axC, axD = axes.ravel()

# -----------------------------------------------------------------------------
# A. Main result: fidelity increases with resource squeezing (Fig. 7 candidate)
# -----------------------------------------------------------------------------

for cutoff in CUTOFFS:
    subset = [r for r in ok_rows if r["cutoff"] == cutoff]
    axA.plot([r["resource_squeezing"] for r in subset], [r["fidelity_to_ideal_cubic_phase"] for r in subset],
              "o-", color=colors[cutoff], label=f"cutoff={cutoff}")
style_axis(axA, "Resource squeezing $r$", "Fidelity to ideal CubicPhase($\\gamma$)",
           f"Finite-resource injection ($\\gamma$={GAMMA}, $m$=0)", "A")
axA.legend(frameon=False, fontsize=7)

# -----------------------------------------------------------------------------
# B. Retained norm of successful cases
# -----------------------------------------------------------------------------

for cutoff in CUTOFFS:
    subset = [r for r in ok_rows if r["cutoff"] == cutoff]
    axB.plot([r["resource_squeezing"] for r in subset], [r["retained_norm"] for r in subset],
              "o-", color=colors[cutoff], label=f"cutoff={cutoff}")
axB.axhline(1.0, color="black", linestyle=":", linewidth=0.8)
style_axis(axB, "Resource squeezing $r$", "Retained norm", "Truncation adequacy", "B")
axB.legend(frameon=False, fontsize=7)

# -----------------------------------------------------------------------------
# C. Guard-triggered (not silently dropped) case count per cutoff
# -----------------------------------------------------------------------------

ok_counts = [len([r for r in ok_rows if r["cutoff"] == c]) for c in CUTOFFS]
not_ok_counts = [len([r for r in not_ok_rows if r["cutoff"] == c]) for c in CUTOFFS]
x = np.arange(len(CUTOFFS))
axC.bar(x, ok_counts, color="blue", label="ok")
axC.bar(x, not_ok_counts, bottom=ok_counts, color="red", label="norm-guard triggered")
axC.set_xticks(x)
axC.set_xticklabels(CUTOFFS)
style_axis(axC, "Fock cutoff", "Case count", "Norm-guard outcomes (not hidden)", "C")
axC.legend(frameon=False, fontsize=7)

# -----------------------------------------------------------------------------
# D. Trace distance (log scale): approach to the ideal gate
# -----------------------------------------------------------------------------

for cutoff in CUTOFFS:
    subset = [r for r in ok_rows if r["cutoff"] == cutoff]
    axD.semilogy([r["resource_squeezing"] for r in subset],
                 np.maximum([r["trace_distance_estimate"] for r in subset], floor),
                 "o-", color=colors[cutoff], label=f"cutoff={cutoff}")
style_axis(axD, "Resource squeezing $r$", "Trace distance to ideal gate",
           "Approach to the infinite-squeezing limit", "D")
axD.legend(frameon=False, fontsize=7)

fig.suptitle("Finite-resource cubic MBQC injection (approximation, not an exact gate)",
             fontsize=10.5, fontweight="bold", y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.96], h_pad=2.4, w_pad=2.4)
common.save_figure(fig, "R23_cubic_injection")
plt.close(fig)

common.print_summary(
    "R23 cubic MBQC injection (finite-resource approximation)",
    cases=len(rows),
    ok=len(ok_rows),
    min_fidelity=min((r["fidelity_to_ideal_cubic_phase"] for r in ok_rows), default=None),
    max_fidelity=max((r["fidelity_to_ideal_cubic_phase"] for r in ok_rows), default=None),
)


## R24 — Native Kerr Gate: Exact Phase Rule

R24 checks `Kerr(kappa)` (Piquasso's native \(\exp(i\kappa\hat n^2)\)) against
the simplest possible exact reference for a Fock-diagonal operator: the phase
rule applies **independently to each number state**,

$$
\hat n^2|n\rangle = n^2|n\rangle
\quad\Longrightarrow\quad
\exp(i\kappa\hat n^2)\,|n\rangle = e^{i\kappa n^2}|n\rangle,
$$

so for an arbitrary superposition \(|\psi\rangle=\sum_n c_n|n\rangle\), the
exact output is \(\sum_n c_n\,e^{i\kappa n^2}|n\rangle\) — a diagonal phase
rule computed independently here with plain NumPy, not by calling the Kerr
gate itself.

### The test state

A custom, non-trivial superposition \((|0\rangle+|1\rangle+|2\rangle)/\sqrt3\)
(`FockInput` with explicit amplitudes) is used instead of a coherent or
Gaussian input, so the check exercises the gate on an arbitrary, manifestly
non-Gaussian state — exactly the regime a diagonal phase gate is meant for.

### What is checked

For \(\kappa\in[-2,2]\) (17 values): the full complex state-vector error
\(\|\,|\psi_{\text{pg}}\rangle - |\psi_{\text{exact}}\rangle\,\|\) and the
fidelity \(|\langle\psi_{\text{exact}}|\psi_{\text{pg}}\rangle|^2\) against
the exact phase rule above.

### Success criteria

- State-vector error \(\lesssim10^{-8}\) and fidelity \(>1-10^{-10}\) for
  every \(\kappa\) (panels A, B) — since the cutoff (8) is far larger than
  the 3-dimensional support of the test state, there is no truncation error
  to confound the comparison: any nonzero error here would be a genuine gate
  bug, not a finite-cutoff artifact.


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg

set_reference_style()

KAPPAS = np.linspace(-2.0, 2.0, 17)
CUTOFF = 8
TOL_FIDELITY = 1 - 1e-10
TOL_VECTOR = 1e-8

amplitudes_in = np.array([1, 1, 1], dtype=complex) / np.sqrt(3)
fock_input = pg.FockInput(tuple(amplitudes_in))

rows = []
for kappa in KAPPAS:
    pattern = pg.Pattern(inputs=(0,)).append(pg.Kerr(0, float(kappa))).append(pg.Output((0,))).validate()
    state = pg.simulate(pattern, backend="piquasso-fock", cutoff=CUTOFF, inputs={0: fock_input}, seed=0).state
    actual_map = dict(zip(state.basis, state.state_vector, strict=True))
    actual = np.array([actual_map.get((n,), 0j) for n in range(CUTOFF)])

    # Exact diagonal phase rule: |n> -> exp(i kappa n^2) |n>, computed
    # independently of the Kerr gate itself.
    expected = np.zeros(CUTOFF, dtype=complex)
    for n in range(len(amplitudes_in)):
        expected[n] = amplitudes_in[n] * np.exp(1j * kappa * n**2)

    overlap = np.vdot(expected, actual)
    fidelity = float(np.clip(abs(overlap) ** 2, 0, 1))
    vector_error = float(np.linalg.norm(actual - expected))
    rows.append(
        {
            "kappa": float(kappa),
            "fidelity_to_exact_phase_rule": fidelity,
            "state_vector_error": vector_error,
        }
    )

# -----------------------------------------------------------------------------
# Validation
# -----------------------------------------------------------------------------

failures = [r for r in rows if r["fidelity_to_exact_phase_rule"] < TOL_FIDELITY or r["state_vector_error"] > TOL_VECTOR]
if failures:
    common.save_csv(failures, "R24_kerr_direct_FAILURES")
    raise AssertionError(f"{len(failures)} Kerr cases disagreed with the exact phase rule; see FAILURES csv")

common.save_result(rows, "R24_kerr_direct", extra={"input_amplitudes": [[a.real, a.imag] for a in amplitudes_in], "cutoff": CUTOFF})

# =============================================================================
# Final 1 x 2 publication figure
# =============================================================================

floor = 1e-16
fig, (axA, axB) = plt.subplots(1, 2, figsize=(9.6, 4.2), facecolor="white")

axA.semilogy([r["kappa"] for r in rows], np.maximum([r["state_vector_error"] for r in rows], floor),
             "o-", color="red")
axA.axhline(TOL_VECTOR, color="black", linestyle="--", linewidth=0.8, label="Tolerance")
style_axis(axA, r"Kerr coefficient $\kappa$", "State-vector error vs. exact phase rule",
           r"$(|0\rangle+|1\rangle+|2\rangle)/\sqrt{3}$ under Kerr", "A")
axA.legend(frameon=False, loc="best")

axB.semilogy([r["kappa"] for r in rows],
             np.maximum([1 - r["fidelity_to_exact_phase_rule"] for r in rows], floor),
             "o-", color="blue")
axB.axhline(1 - TOL_FIDELITY, color="black", linestyle="--", linewidth=0.8, label="Tolerance")
style_axis(axB, r"Kerr coefficient $\kappa$", "1 - fidelity", "Fidelity to exact phase rule", "B")
axB.legend(frameon=False, loc="best")

fig.suptitle("Native Kerr gate: exact diagonal phase rule", fontsize=10.5,
             fontweight="bold", y=1.02)
fig.tight_layout(rect=[0, 0, 1, 0.93], w_pad=2.2)
common.save_figure(fig, "R24_kerr_direct")
plt.close(fig)

common.print_summary(
    "R24 Kerr direct execution",
    kappas=len(rows),
    max_state_vector_error=max(r["state_vector_error"] for r in rows),
    min_fidelity=min(r["fidelity_to_exact_phase_rule"] for r in rows),
)


## R25 — Kerr Lie-Product Synthesis Convergence

`Kerr` is native at the Fock-backend level (R24), but the *compiler*
(`compile_circuit`, R3) cannot lower it directly — `synthesize_kerr` instead
approximates \(\exp(i\kappa\hat n^2)\) by a Lie-product formula built from
rotation, quadratic-phase and cubic-phase primitives. R25 isolates the
**synthesis (product-formula) error alone**: the synthesized circuit's gates
are translated directly into physical Fock-backend commands and executed
without ever compiling to an MBQC pattern, so no finite-squeezing injection
noise (R23) contaminates this measurement.

### Product-formula convergence

A first-order Lie product approximates a sum of non-commuting generators by
alternating exponentials of each term; splitting into more, smaller steps
reduces the commutator error. `compile_circuit`'s own docstring records the
expected leading-order scaling of the *synthesized amplitude* error,

$$
\big\|\,|\psi_{\text{synth}}\rangle - |\psi_{\text{exact}}\rangle\,\big\|
\; = \; O\!\big(\text{steps}^{-1/2}\big),
$$

which — since fidelity relates to amplitude error via
\(1-F \approx \|\Delta\psi\|^2\) for small deviations — predicts the
**infidelity** should fall roughly as \(O(\text{steps}^{-1})\). This
experiment checks that prediction empirically with a log-log fit, rather than
merely asserting convergence occurs.

### What is checked

For \(\kappa=0.05\) on the same test state as R24
(\((|0\rangle+|1\rangle+|2\rangle)/\sqrt3\), cutoff 16), across
`synthesis_steps`\(\in\{1,2,4,8,16,32,64\}\): fidelity to the exact phase
rule (R24's reference), primitive-gate count, and wall-clock runtime.

### Success criteria

- Infidelity \(1-F\) decreases monotonically with more synthesis steps
  (panel A) and the fitted log-log exponent is reported and compared to the
  \(-1\) prediction above — as an empirical observation over the tested
  range, not a proof of the asymptotic rate.
- Primitive-gate count and runtime grow with `synthesis_steps` (panels B, C)
  — the fidelity/cost trade-off inherent to product-formula synthesis.
- Retained Fock-truncation norm stays near 1 throughout (panel D), confirming
  the convergence trend in panel A is a synthesis effect, not a cutoff
  artifact.


In [ ]:
from __future__ import annotations

import time
import warnings

import numpy as np

import photographiq as pg
from photographiq.synthesis import synthesize_kerr

set_reference_style()

KAPPA = 0.05
CUTOFF = 16
STEPS = [1, 2, 4, 8, 16, 32, 64]


def physical_command_for(name, mode, params):
    """Translate one synthesized Circuit gate into a direct physical command."""
    if name == "displace":
        return pg.Displace(mode, *params)
    if name == "cubic_phase":
        return pg.CubicPhase(mode, params[0])
    if name == "symplectic":
        matrix = np.asarray(params[0])
        if np.allclose(matrix, [[1.0, 0.0], [matrix[1, 0], 1.0]], atol=1e-12):
            return pg.QuadraticPhase(mode, float(matrix[1, 0]))
        if np.allclose(matrix @ matrix.T, np.eye(2), atol=1e-10):
            angle = float(np.arctan2(matrix[1, 0], matrix[0, 0]))
            return pg.Rotate(mode, angle)
        raise NotImplementedError("Unexpected symplectic gate shape in Kerr synthesis output")
    raise NotImplementedError(f"Unexpected synthesized gate kind: {name}")


amplitudes_in = np.array([1, 1, 1], dtype=complex) / np.sqrt(3)
fock_input = pg.FockInput(tuple(amplitudes_in))
expected = np.zeros(CUTOFF, dtype=complex)
for n in range(len(amplitudes_in)):
    expected[n] = amplitudes_in[n] * np.exp(1j * KAPPA * n**2)

rows = []
for steps in STEPS:
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", RuntimeWarning)
        circuit, report = synthesize_kerr(KAPPA, steps=steps, modes=1, mode=0)
        pattern = pg.Pattern(inputs=(0,))
        for name, modes, params in circuit.gates:
            pattern.append(physical_command_for(name, modes[0], params))
        pattern.append(pg.Output((0,)))
        pattern.validate()

        start = time.perf_counter()
        state = pg.simulate(pattern, backend="piquasso-fock", cutoff=CUTOFF, inputs={0: fock_input}, seed=0).state
        elapsed = time.perf_counter() - start

    actual_map = dict(zip(state.basis, state.state_vector, strict=True))
    actual = np.array([actual_map.get((n,), 0j) for n in range(CUTOFF)])
    overlap = np.vdot(expected, actual)
    fidelity = float(np.clip(abs(overlap) ** 2, 0, 1))
    rows.append(
        {
            "synthesis_steps": steps,
            "fidelity_to_exact_kerr": fidelity,
            "trace_distance_estimate": float(np.sqrt(max(0.0, 1 - fidelity))),
            "primitive_count": report.primitive_count,
            "synthesis_order": report.order,
            "runtime_seconds": elapsed,
            "retained_norm": state.retained_norms[-1] if state.retained_norms else None,
        }
    )
    print(f"  steps={steps}: fidelity={fidelity:.8f}, primitives={report.primitive_count}, {elapsed:.3f}s", flush=True)

# -----------------------------------------------------------------------------
# Convergence should be monotonic in synthesis_steps (up to numerical noise
# at very high fidelity, where floating-point roundoff can dominate)
# -----------------------------------------------------------------------------

infidelities = np.array([1 - r["fidelity_to_exact_kerr"] for r in rows])
steps_arr = np.array(STEPS)

# Empirical log-log exponent: infidelity ~ a * steps^p (observation over the
# sampled range only; compiler.py documents an expected leading-order
# amplitude error O(steps^-1/2), i.e. infidelity ~ O(steps^-1)).
finite = infidelities > 1e-15
if finite.sum() >= 2:
    fit_slope, fit_intercept = np.polyfit(np.log(steps_arr[finite]), np.log(infidelities[finite]), 1)
else:
    fit_slope, fit_intercept = float("nan"), float("nan")

common.save_result(
    rows, "R25_kerr_synthesis",
    extra={
        "kappa": KAPPA, "cutoff": CUTOFF,
        "note": "synthesis-only execution; MBQC injection noise excluded (see R23)",
        "fitted_loglog_exponent": float(fit_slope),
        "documented_leading_order_exponent": -1.0,
    },
)

# =============================================================================
# Final 2 x 2 publication figure
# =============================================================================

floor = 1e-16
fig, axes = plt.subplots(2, 2, figsize=(10.0, 7.4), facecolor="white")
axA, axB, axC, axD = axes.ravel()

# -----------------------------------------------------------------------------
# A. Infidelity vs synthesis steps with empirical log-log fit
# -----------------------------------------------------------------------------

axA.loglog(steps_arr, np.maximum(infidelities, floor), "o-", color="red", label="Measured")
if finite.sum() >= 2:
    fit_line = np.exp(fit_intercept) * steps_arr.astype(float) ** fit_slope
    axA.loglog(steps_arr, fit_line, "--", color="black",
               label=f"fit: $p$={fit_slope:.2f} (doc.: $p$=$-1$)")
style_axis(axA, "Synthesis steps", "1 - fidelity", "Product-formula convergence", "A")
axA.legend(frameon=False, fontsize=7)

# -----------------------------------------------------------------------------
# B. Primitive-gate count (resource cost)
# -----------------------------------------------------------------------------

axB.loglog(steps_arr, [r["primitive_count"] for r in rows], "o-", color="blue")
style_axis(axB, "Synthesis steps", "Primitive-gate count", "Synthesis resource cost", "B")

# -----------------------------------------------------------------------------
# C. Wall-clock runtime
# -----------------------------------------------------------------------------

axC.loglog(steps_arr, [r["runtime_seconds"] for r in rows], "o-", color="magenta")
style_axis(axC, "Synthesis steps", "Runtime (s)", "Wall-clock cost", "C")

# -----------------------------------------------------------------------------
# D. Retained Fock-truncation norm: confirms A is a synthesis, not cutoff, effect
# -----------------------------------------------------------------------------

axD.plot(steps_arr, [r["retained_norm"] for r in rows], "o-", color="cyan")
axD.axhline(1.0, color="black", linestyle=":", linewidth=0.8)
style_axis(axD, "Synthesis steps", "Retained norm", "Cutoff adequacy", "D")

fig.suptitle(f"Kerr Lie-product synthesis, $\\kappa$={KAPPA} (direct physical execution)",
             fontsize=10.5, fontweight="bold", y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.96], h_pad=2.4, w_pad=2.4)
common.save_figure(fig, "R25_kerr_synthesis")
plt.close(fig)

common.print_summary(
    "R25 Kerr synthesis convergence",
    steps=STEPS,
    min_infidelity=float(infidelities.min()),
    max_infidelity=float(infidelities.max()),
    fitted_loglog_exponent=float(fit_slope),
)


## R26 — GKP Resource Characterization (Finite-Energy, No Fault-Tolerance Claim)

R26 characterizes finite-energy square-lattice GKP resources
(`GKPResource`) against their own intended continuous-variable target, using
four independent diagnostics. **No fault-tolerance or threshold claim is made
anywhere in this experiment** (`docs/theory.md`, `MANUSCRIPT_MAP.md`) — GKP
states here are always finite-energy approximations, characterized as such.

### The finite-energy GKP wavefunction

An ideal GKP codeword is an infinite comb of \(\delta\)-functions spaced by
\(\sqrt{2\pi}\) (\(\hbar=2\)); a *finite-energy* GKP resource instead
convolves that comb with a Gaussian envelope of width `envelope` and gives
each tooth a finite `peak_width`. The package's target wavefunction
\(\psi(q)\) is compared to its own **Fock-truncated reconstruction**,
\(\big|\sum_n c_n\phi_n(q)\big|^2\), built independently from the package's
own Hermite basis functions (`fock_measurements.wavefunctions`) rather than
by re-deriving the target analytically — this checks that `.project(cutoff)`
faithfully represents the *intended* target in the truncated Fock space, via

$$
L_1 = \int \big|\,|\psi_{\text{target}}(q)|^2 - |\psi_{\text{Fock}}(q)|^2\,\big|\,dq.
$$

### Stabilizers and logical distinguishability

`gkp.stabilizers` returns the expectation of the ideal displacement
stabilizers \(e^{i\sqrt{2\pi}\hat p}\), \(e^{i\sqrt{2\pi}\hat q}\); for an
ideal code these have unit modulus, and finite energy reduces them below 1.
`GKPCode.gram` gives the codeword overlap matrix; \(|\langle0_L|1_L\rangle|\)
quantifies how distinguishable the two logical states actually are at finite
energy — the practical quantity a fault-tolerance argument would need small,
though none is claimed here.

### What is checked

Three envelope/peak-width combinations (sharp/default/fuzzy) × 2 logical
values, at cutoff 48: captured mass, both stabilizer magnitudes, the
target-vs-Fock \(L_1\) distance, mean photon number, and the \(|0\rangle_L\)-\(|1\rangle_L\)
overlap; plus a cutoff-sensitivity sweep (24–96) at the default width.

### Success criteria

- Captured mass \(\to1\) and \(1-\text{captured mass}\) shrinks monotonically
  with cutoff (panel B) — the resource fits inside the Fock truncation.
- Stabilizer magnitudes are close to 1 and **decrease** as the code gets
  "fuzzier" (larger peak width/envelope, panel C) — a genuine finite-energy
  trade-off, not a numerical artifact.
- \(L_1\) distance between the target and its Fock reconstruction stays small
  across all three widths (panel D), confirming `.project()` is faithful.
- Logical overlap \(|\langle0_L|1_L\rangle|\) **grows** with fuzziness (panel
  E) — sharper codes are more distinguishable, exactly as expected; this
  number is reported, never claimed to meet any fault-tolerance threshold.
- Mean photon number (resource cost) grows as the code sharpens (panel F).


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg
from photographiq.fock_measurements import wavefunctions

set_reference_style()

PARAMETER_SETS = [
    {"peak_width": 0.3, "envelope": 0.3, "label": "sharp"},
    {"peak_width": 0.4, "envelope": 0.4, "label": "default"},
    {"peak_width": 0.6, "envelope": 0.6, "label": "fuzzy"},
]
DEFAULT_CUTOFF = 48
CUTOFF_SENSITIVITY = [24, 32, 48, 64, 96]


def reconstructed_probability(amplitudes: np.ndarray, q_grid: np.ndarray) -> np.ndarray:
    """|sum_n c_n phi_n(q)|^2 using the package's own Hermite wavefunctions."""
    cutoff = len(amplitudes)
    basis = np.array([wavefunctions(x, cutoff) for x in q_grid])  # (len(q), cutoff)
    values = basis @ amplitudes
    return np.abs(values) ** 2


rows = []
representative_data = None

for params in PARAMETER_SETS:
    for logical in (0, 1):
        resource = pg.GKPResource(logical, peak_width=params["peak_width"], envelope=params["envelope"])
        fock_input, captured_mass = resource.project(DEFAULT_CUTOFF)

        pattern = pg.Pattern(inputs=(0,)).append(pg.Output((0,))).validate()
        state = pg.simulate(pattern, backend="piquasso-fock", cutoff=DEFAULT_CUTOFF, inputs={0: fock_input}, seed=0).state
        stabilizer_values = pg.gkp.stabilizers(state)

        q_target, psi_target = resource.wavefunction()
        q_grid = np.linspace(q_target.min(), q_target.max(), 2000)
        target_probability = np.interp(q_grid, q_target, psi_target**2)
        fock_probability = reconstructed_probability(np.array(fock_input.amplitudes), q_grid)
        l1_distance = float(np.trapezoid(np.abs(target_probability - fock_probability), q_grid))

        rows.append(
            {
                "label": params["label"],
                "peak_width": params["peak_width"],
                "envelope": params["envelope"],
                "logical": logical,
                "cutoff": DEFAULT_CUTOFF,
                "captured_mass": captured_mass,
                "q_translation_stabilizer_abs": abs(stabilizer_values["q_translation"]),
                "p_translation_stabilizer_abs": abs(stabilizer_values["p_translation"]),
                "target_vs_fock_probability_l1": l1_distance,
                "mean_photon_number": state.photon_number(0),
            }
        )
        if params["label"] == "default" and logical == 0:
            representative_data = (q_grid, target_probability, fock_probability, params)

    code = pg.GKPCode(peak_width=params["peak_width"], envelope=params["envelope"], cutoff=DEFAULT_CUTOFF)
    rows.append(
        {
            "label": params["label"], "peak_width": params["peak_width"], "envelope": params["envelope"],
            "logical": "overlap|0><1|", "cutoff": DEFAULT_CUTOFF, "captured_mass": None,
            "q_translation_stabilizer_abs": None, "p_translation_stabilizer_abs": None,
            "target_vs_fock_probability_l1": None,
            "mean_photon_number": None,
            "logical_overlap_abs": abs(code.gram[0, 1]),
        }
    )

cutoff_rows = []
for cutoff in CUTOFF_SENSITIVITY:
    resource = pg.GKPResource(0, peak_width=0.4, envelope=0.4)
    _, mass = resource.project(cutoff)
    cutoff_rows.append({"cutoff": cutoff, "captured_mass": mass})

common.save_result(rows, "R26_gkp_resource")
common.save_csv(cutoff_rows, "R26_gkp_resource_cutoff_sensitivity")

numeric_rows = [r for r in rows if r["captured_mass"] is not None]
overlap_rows = [r for r in rows if r.get("logical_overlap_abs") is not None]

# =============================================================================
# Final 2 x 3 publication figure
# =============================================================================

fig, axes = plt.subplots(2, 3, figsize=(13.4, 8.0), facecolor="white")
(axA, axB, axC, axD, axE, axF) = axes.ravel()

# -----------------------------------------------------------------------------
# A. Representative wavefunction: intended target vs. Fock reconstruction
# -----------------------------------------------------------------------------

q_grid, target_probability, fock_probability, params = representative_data
axA.plot(q_grid, target_probability, label="Target $|\\psi(q)|^2$", color="red")
axA.plot(q_grid, fock_probability, "--", label=f"Fock reconstruction (c={DEFAULT_CUTOFF})", color="blue")
style_axis(axA, "$q$", "Probability density", f"GKP $|0\\rangle_L$, {params['label']}", "A")
axA.legend(frameon=False, fontsize=7)

# -----------------------------------------------------------------------------
# B. Cutoff sensitivity of captured mass
# -----------------------------------------------------------------------------

axB.semilogy([r["cutoff"] for r in cutoff_rows], np.maximum([1 - r["captured_mass"] for r in cutoff_rows], 1e-16),
             "o-", color="magenta")
style_axis(axB, "Fock cutoff", "1 - captured mass", "Cutoff sensitivity (width=0.4)", "B")

# -----------------------------------------------------------------------------
# C. Stabilizer magnitude vs. code sharpness
# -----------------------------------------------------------------------------

labels = [p["label"] for p in PARAMETER_SETS]
x = np.arange(len(labels))
for logical, marker in ((0, "o"), (1, "s")):
    subset = [r for r in numeric_rows if r["logical"] == logical]
    q_vals = [next(r["q_translation_stabilizer_abs"] for r in subset if r["label"] == lab) for lab in labels]
    p_vals = [next(r["p_translation_stabilizer_abs"] for r in subset if r["label"] == lab) for lab in labels]
    axC.plot(x, q_vals, marker + "-", color="red", label=f"$|q$-stab$|$, logical={logical}")
    axC.plot(x, p_vals, marker + "--", color="blue", label=f"$|p$-stab$|$, logical={logical}")
axC.set_xticks(x)
axC.set_xticklabels(labels)
axC.axhline(1.0, color="black", linestyle=":", linewidth=0.8)
style_axis(axC, "Code sharpness", "Stabilizer magnitude", "Finite-energy stabilizer decay", "C")
axC.legend(frameon=False, fontsize=6)

# -----------------------------------------------------------------------------
# D. Target vs. Fock reconstruction L1 distance
# -----------------------------------------------------------------------------

for logical, marker in ((0, "o"), (1, "s")):
    subset = [r for r in numeric_rows if r["logical"] == logical]
    vals = [next(r["target_vs_fock_probability_l1"] for r in subset if r["label"] == lab) for lab in labels]
    axD.plot(x, vals, marker + "-", label=f"logical={logical}")
axD.set_xticks(x)
axD.set_xticklabels(labels)
style_axis(axD, "Code sharpness", "$L_1$ distance", "Target vs. Fock reconstruction", "D")
axD.legend(frameon=False, fontsize=7)

# -----------------------------------------------------------------------------
# E. Logical overlap |<0_L|1_L>| (reported, not compared to any threshold)
# -----------------------------------------------------------------------------

overlap_vals = [next(r["logical_overlap_abs"] for r in overlap_rows if r["label"] == lab) for lab in labels]
axE.plot(x, overlap_vals, "o-", color="magenta")
axE.set_xticks(x)
axE.set_xticklabels(labels)
style_axis(axE, "Code sharpness", r"$|\langle 0_L|1_L\rangle|$", "Logical distinguishability", "E")

# -----------------------------------------------------------------------------
# F. Mean photon number (resource cost) vs. code sharpness
# -----------------------------------------------------------------------------

for logical, marker in ((0, "o"), (1, "s")):
    subset = [r for r in numeric_rows if r["logical"] == logical]
    vals = [next(r["mean_photon_number"] for r in subset if r["label"] == lab) for lab in labels]
    axF.plot(x, vals, marker + "-", label=f"logical={logical}")
axF.set_xticks(x)
axF.set_xticklabels(labels)
style_axis(axF, "Code sharpness", r"$\langle n\rangle$", "Resource photon-number cost", "F")
axF.legend(frameon=False, fontsize=7)

fig.suptitle("GKP resource characterization (finite-energy; no fault-tolerance claim)",
             fontsize=11, fontweight="bold", y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.96], h_pad=2.6, w_pad=2.4)
common.save_figure(fig, "R26_gkp_resource")
plt.close(fig)

common.print_summary(
    "R26 GKP resource characterization",
    cases=len(rows),
    min_captured_mass=min(r["captured_mass"] for r in numeric_rows),
    max_stabilizer_deviation_from_unity=max(1 - r["q_translation_stabilizer_abs"] for r in numeric_rows),
)


## R27 — GKP Logical Shifts and Stabilizers

R27 checks two independent claims: (1) logical lattice displacements
(`gkp.logical_displacement`) act on finite-energy GKP codewords the way the
ideal stabilizer algebra predicts, up to a controlled finite-energy
deviation; (2) the `NearestCellDecoder` implements exactly the floor-based
decoding rule its name promises, verified against an independent NumPy
formula for every tested outcome.

### Stabilizer phase prediction under a logical shift

The ideal stabilizers are \(\hat S_p=e^{i\sqrt{2\pi}\hat p}\),
\(\hat S_q=e^{i\sqrt{2\pi}\hat q}\) (spacing \(d=\sqrt{2\pi}\), so
\(d^2=2\pi\)). A logical \(X\) shift (\(\hat q\to\hat q+d\)) commutes with
\(\hat S_p\) up to a phase:

$$
\hat S_p\,\hat D_q(d) = e^{i d\sqrt{2\pi}}\,\hat D_q(d)\,\hat S_p
= e^{i\cdot2\pi}\,\hat D_q(d)\,\hat S_p = \hat D_q(d)\,\hat S_p,
$$

so \(\langle\hat S_p\rangle\) after an \(X\) shift should equal
\(\langle\hat S_p\rangle\) **before** the shift, times exactly 1 — i.e. the
shift should leave the \(p\)-stabilizer's expectation value **unchanged** for
an ideal code (and symmetrically for a \(Z\) shift and \(\hat S_q\)). This
factor-of-1 prediction is computed independently with NumPy and checked
directly against the simulated before/after stabilizer values — finite
energy is expected to introduce a small, reported deviation, not exact
invariance.

### Nearest-cell decoding

`NearestCellDecoder` should implement the same floor-based rule that would be
written from scratch: cell \(=\lfloor\text{outcome}/d+\tfrac12\rfloor\),
residual \(=\text{outcome}-\text{cell}\cdot d\), bit \(=\text{cell}\bmod2\) —
checked at 61 outcomes spanning \(\pm3d\) around the ideal comb.

### What is checked

1. Stabilizer magnitudes before/after each logical shift, and the complex
   agreement with the phase-invariance prediction above (a genuine
   finite-energy deviation, reported with its own size).
2. Decoded bit and residual against the independent floor-based formula,
   for every one of 61 outcomes (exact combinatorial/arithmetic check, no
   physics tolerance involved).

### Success criteria

- Decoder bit and residual match the independent formula **exactly** for all
  61 outcomes (panel A: the characteristic decoding staircase; panel B: zero
  residual error).
- Stabilizer magnitudes stay close to their pre-shift values (panel C) and
  the phase-invariance deviation is small and consistent with the same
  finite-energy scale seen in R26 (panel D) — not growing without bound.


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg
from photographiq.gkp import SPACING, logical_displacement, stabilizers

set_reference_style()

CUTOFF = 48
PEAK_WIDTH = ENVELOPE = 0.35
PHASE_TOL = 0.02  # finite-energy deviation from the ideal factor-of-1 prediction


def apply_logical_gate(fock_input, gate, cutoff):
    pattern = pg.Pattern(inputs=(0,)).append(gate).append(pg.Output((0,))).validate()
    return pg.simulate(pattern, backend="piquasso-fock", cutoff=cutoff, inputs={0: fock_input}, seed=0).state


rows = []
decoder = pg.NearestCellDecoder()

for logical in (0, 1):
    resource = pg.GKPResource(logical, peak_width=PEAK_WIDTH, envelope=ENVELOPE)
    fock_input, mass = resource.project(CUTOFF)
    base_state = apply_logical_gate(fock_input, pg.Displace(0, q=0.0, p=0.0), CUTOFF)
    base_stabilizers = stabilizers(base_state)

    for gate_name in ("X", "Z"):
        shifted_state = apply_logical_gate(fock_input, logical_displacement(0, gate_name), CUTOFF)
        shifted_stabilizers = stabilizers(shifted_state)

        # Independent prediction: an X shift (q -> q+SPACING) multiplies the
        # p-translation stabilizer by exp(i*SPACING^2) = exp(i*2*pi) = 1
        # (identity, since SPACING^2=2*pi); a Z shift analogously predicts
        # the q-translation stabilizer is unchanged. Finite energy/truncation
        # gives a small, reported deviation from this exact prediction.
        phase_factor = complex(np.exp(1j * SPACING**2)) if gate_name == "X" else complex(np.exp(-1j * SPACING**2))
        watched_key = "p_translation" if gate_name == "X" else "q_translation"
        base_val = base_stabilizers[watched_key]
        shifted_val = shifted_stabilizers[watched_key]
        predicted_val = base_val * phase_factor
        phase_prediction_error = abs(shifted_val - predicted_val)

        rows.append(
            {
                "logical": logical,
                "gate": gate_name,
                "cutoff": CUTOFF,
                "captured_mass": mass,
                "base_q_stabilizer_abs": abs(base_stabilizers["q_translation"]),
                "base_p_stabilizer_abs": abs(base_stabilizers["p_translation"]),
                "shifted_q_stabilizer_abs": abs(shifted_stabilizers["q_translation"]),
                "shifted_p_stabilizer_abs": abs(shifted_stabilizers["p_translation"]),
                "watched_stabilizer": watched_key,
                "phase_prediction_error": phase_prediction_error,
            }
        )

# -----------------------------------------------------------------------------
# Decode a family of raw homodyne-like outcomes around the ideal 0/1 combs
# and check the nearest-cell rule against an independent formula.
# -----------------------------------------------------------------------------

decode_rows = []
for outcome in np.linspace(-3 * SPACING, 3 * SPACING, 61):
    result = decoder.decode(float(outcome))
    independent_cell = int(np.floor(outcome / SPACING + 0.5))
    independent_residual = float(outcome - independent_cell * SPACING)
    independent_bit = independent_cell % 2
    decode_rows.append(
        {
            "outcome": float(outcome),
            "decoded_bit": result.bit,
            "independent_bit": independent_bit,
            "residual": result.residual,
            "independent_residual": independent_residual,
            "residual_error": abs(result.residual - independent_residual),
            "bit_match": result.bit == independent_bit,
        }
    )

# -----------------------------------------------------------------------------
# Validation
# -----------------------------------------------------------------------------

bit_failures = [r for r in decode_rows if not r["bit_match"] or r["residual_error"] > 1e-9]
if bit_failures:
    common.save_csv(bit_failures, "R27_gkp_stabilizers_decode_FAILURES")
    raise AssertionError("Nearest-cell decoder disagreed with the independent floor-based formula; see FAILURES csv")

phase_failures = [r for r in rows if r["phase_prediction_error"] > PHASE_TOL]
if phase_failures:
    common.save_csv(phase_failures, "R27_gkp_stabilizers_phase_FAILURES")
    raise AssertionError(f"Stabilizer phase deviated from prediction beyond {PHASE_TOL}; see FAILURES csv")

common.save_result(rows, "R27_gkp_stabilizers", extra={"cutoff": CUTOFF, "peak_width": PEAK_WIDTH, "envelope": ENVELOPE})
common.save_csv(decode_rows, "R27_gkp_stabilizers_decode_sweep")

# =============================================================================
# Final 2 x 2 publication figure
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(10.0, 7.6), facecolor="white")
axA, axB, axC, axD = axes.ravel()

# -----------------------------------------------------------------------------
# A. Nearest-cell decoding staircase
# -----------------------------------------------------------------------------

axA.plot([r["outcome"] for r in decode_rows], [r["decoded_bit"] for r in decode_rows], "o", markersize=3, color="blue")
for k in range(-3, 4):
    axA.axvline((k + 0.5) * SPACING, color="grey", linestyle=":", linewidth=0.7)
style_axis(axA, "Raw homodyne outcome", "Decoded logical bit", "Nearest-cell GKP decoding", "A")

# -----------------------------------------------------------------------------
# B. Decoder residual validation (exact, no tolerance)
# -----------------------------------------------------------------------------

axB.semilogy([r["outcome"] for r in decode_rows], np.maximum([r["residual_error"] for r in decode_rows], 1e-18),
             "o", markersize=3, color="red")
axB.axhline(1e-9, color="black", linestyle="--", linewidth=0.8, label="Tolerance")
style_axis(axB, "Raw homodyne outcome", "Residual error vs. independent formula",
           "Decoder validation", "B")
axB.legend(frameon=False, loc="best")

# -----------------------------------------------------------------------------
# C. Stabilizer magnitude before/after logical shift
# -----------------------------------------------------------------------------

labels = [f"L{r['logical']},{r['gate']}" for r in rows]
x = np.arange(len(rows))
base_vals = [r[f"base_{r['watched_stabilizer'][0]}_stabilizer_abs"] for r in rows]
shifted_vals = [r[f"shifted_{r['watched_stabilizer'][0]}_stabilizer_abs"] for r in rows]
width = 0.35
axC.bar(x - width / 2, base_vals, width, color="blue", label="Before shift")
axC.bar(x + width / 2, shifted_vals, width, color="red", label="After shift")
axC.set_xticks(x)
axC.set_xticklabels(labels, fontsize=7)
style_axis(axC, "", "Watched stabilizer magnitude", "Stabilizer stability under logical shift", "C")
axC.legend(frameon=False, fontsize=7)

# -----------------------------------------------------------------------------
# D. Deviation from the exact phase-invariance prediction
# -----------------------------------------------------------------------------

axD.bar(x, [r["phase_prediction_error"] for r in rows], color="magenta")
axD.axhline(PHASE_TOL, color="black", linestyle="--", linewidth=0.8, label="Tolerance")
axD.set_xticks(x)
axD.set_xticklabels(labels, fontsize=7)
style_axis(axD, "", "$|$shifted $-$ predicted$|$", "Finite-energy phase deviation", "D")
axD.legend(frameon=False, fontsize=7)

fig.suptitle(f"GKP logical shifts and stabilizers (width={PEAK_WIDTH})",
             fontsize=10.5, fontweight="bold", y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.96], h_pad=2.6, w_pad=2.4)
common.save_figure(fig, "R27_gkp_stabilizers")
plt.close(fig)

common.print_summary(
    "R27 GKP logical shifts/stabilizers",
    cases=len(rows),
    decode_points=len(decode_rows),
    min_base_stabilizer=min(min(r["base_q_stabilizer_abs"], r["base_p_stabilizer_abs"]) for r in rows),
    max_phase_prediction_error=max(r["phase_prediction_error"] for r in rows),
)


## R28 — GKP Syndrome Extraction: Ensemble vs. Single-Shot (Finite-Energy)

R28 runs the package's actual physical syndrome-extraction circuit
(`gkp.correction_pattern`, a one-ancilla SUM-gate measurement) on a GKP
\(|0\rangle_L\) codeword that has been physically displaced by a known shift
\(\delta\), and checks whether the decoded correction recovers \(\delta\)
(mod the lattice spacing) — the basic operation any GKP-based error
correction depends on.

### Why this must be an ensemble measurement, not a single shot

The syndrome ancilla is *itself* a finite-energy GKP-like resource with its
own multi-peak comb structure (`GKPResource.wavefunction`). A single
homodyne outcome samples across the **entire** ancilla comb, so one
trajectory's decoded residual has large shot-to-shot variance set by the
ancilla's `peak_width` — this is a **design lesson**, not a defect
(`ISSUES_FOUND.md` #3): only the **ensemble** average over repeated
trajectories is expected to be centered on the true \(\delta\). Circular
(not linear) statistics are used because the decoded residual lives on a
periodic domain of width `SPACING`:

$$
\bar\theta = \operatorname{Arg}\Big\langle e^{\,2\pi i\,\text{residual}/\text{SPACING}}\Big\rangle,
\qquad
\sigma_\theta = \sqrt{-2\ln\big|\langle e^{\,2\pi i\,\text{residual}/\text{SPACING}}\rangle\big|}\;\cdot\frac{\text{SPACING}}{2\pi}.
$$

### What is checked

For \(\delta\in[-0.6,0.6]\) (9 values) and \(N=16\) seeded trajectories each:
the circular-mean decoded residual against the exact expectation
\(\delta \bmod \text{SPACING}\) (mapped to \([-\text{SPACING}/2,\text{SPACING}/2)\)),
with a **statistically justified** tolerance (\(5\times\) the standard error
of the circular mean, plus a small finite-energy floor) — a fundamentally
different tolerance philosophy from the machine-precision checks used for
exact algebraic identities (R1–R16).

Because two GKP-scale Fock modes (input + ancilla) share one total-photon
cutoff, some trajectories can legitimately hit the Fock backend's
retained-norm safety guard (`ISSUES_FOUND.md` #2); these are recorded as
excluded seeds (`ok_seeds<N`), never silently dropped or averaged in.

### Success criteria

- The ensemble circular mean tracks the ideal sawtooth
  \(\delta\bmod\text{SPACING}\) within statistical tolerance (panel A).
- Correction reduces the mean \(|q|\) well below the uncorrected \(|\delta|\)
  (panel B) — the extraction is doing useful error correction, not nothing.
- Circular spread \(\sigma_\theta\) (panel C) quantifies the single-shot noise
  directly, making explicit how much averaging was needed for panel A to be
  meaningful.
- `ok_seeds` stays reasonably close to \(N=16\) for every \(\delta\) (panel D)
  — any norm-guard exclusions are visible, not hidden inside a silently
  smaller sample.

### What this experiment does *not* claim

Per `MANUSCRIPT_MAP.md`: this is a finite-ancilla, finite-energy
approximation of one syndrome-extraction round — not a fault-tolerance or
threshold claim.


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg
from photographiq.gkp import SPACING, decode_shift

set_reference_style()

CUTOFF = 80  # two GKP-scale modes share one total-photon cutoff (input + ancilla)
PEAK_WIDTH = ENVELOPE = 0.35
DELTAS = np.linspace(-0.6, 0.6, 9)
N_SEEDS = 16


def displaced_fock_vector(fock_input, delta_q, cutoff):
    pattern = pg.Pattern(inputs=(0,)).append(pg.Displace(0, q=delta_q)).append(pg.Output((0,))).validate()
    state = pg.simulate(pattern, backend="piquasso-fock", cutoff=cutoff, inputs={0: fock_input}, seed=0).state
    amplitudes = dict(zip(state.basis, state.state_vector, strict=True))
    vector = np.array([amplitudes.get((n,), 0j) for n in range(cutoff)])
    vector /= np.linalg.norm(vector)
    return pg.FockInput(tuple(vector))


def circular_stats(residuals: np.ndarray, spacing: float):
    # Circular mean/std on the periodic residual domain of width `spacing`:
    #
    # theta_bar = Arg< exp(2 pi i residual / spacing) >
    # sigma     = sqrt(-2 ln|<exp(2 pi i residual / spacing)>|) * spacing / 2pi
    angles = residuals / spacing * 2 * np.pi
    mean_vector = np.mean(np.exp(1j * angles))
    circular_mean = float(np.angle(mean_vector) * spacing / (2 * np.pi))
    circular_std = float(np.sqrt(max(0.0, -2 * np.log(abs(mean_vector)))) * spacing / (2 * np.pi))
    return circular_mean, circular_std


code_zero = pg.GKPResource(0, peak_width=PEAK_WIDTH, envelope=ENVELOPE)
fock_zero, _ = code_zero.project(CUTOFF)
ancilla_plus = pg.gkp.superposition(1, 1, cutoff=CUTOFF, peak_width=PEAK_WIDTH, envelope=ENVELOPE)
pattern = pg.gkp.correction_pattern(quadrature="q", resource=ancilla_plus)

rows = []
for delta in DELTAS:
    delta = float(delta)
    shifted_input = displaced_fock_vector(fock_zero, delta, CUTOFF)

    residuals, corrected_q, ok_seeds = [], [], 0
    for seed in range(N_SEEDS):
        failure = common.safe_cutoff_run(
            lambda seed=seed: pg.simulate(
                pattern, backend="piquasso-fock", cutoff=CUTOFF, inputs={"in": shifted_input}, seed=3000 + 97 * seed
            ),
            label=f"delta={delta} seed={seed}",
        )
        if not failure["ok"]:
            continue
        result = failure["value"]
        residual, _ = decode_shift(result.records["syndrome"])
        residuals.append(residual)
        corrected_q.append(result.state.quadrature("in")[0])
        ok_seeds += 1

    expected_residual = float(((delta + SPACING / 2) % SPACING) - SPACING / 2)
    if residuals:
        circular_mean, circular_std = circular_stats(np.array(residuals), SPACING)
    else:
        circular_mean = circular_std = None
    rows.append(
        {
            "delta": delta,
            "ok_seeds": ok_seeds,
            "expected_residual": expected_residual,
            "circular_mean_decoded_residual": circular_mean,
            "circular_std_decoded_residual": circular_std,
            "circular_mean_error": None if circular_mean is None else abs(circular_mean - expected_residual),
            "mean_abs_corrected_q": float(np.mean(np.abs(corrected_q))) if corrected_q else None,
            "abs_uncorrected_q": abs(delta),
        }
    )

ok_rows = [r for r in rows if r["ok_seeds"] > 0]

# -----------------------------------------------------------------------------
# Validation: statistical tolerance, not machine precision (Monte Carlo
# ensemble estimate). 5x the standard error of the circular mean, plus a
# small finite-energy floor, gives headroom against a single flaky run while
# still catching a genuinely biased estimator.
# -----------------------------------------------------------------------------

failures = [
    r for r in ok_rows
    if r["circular_mean_error"] > 5 * r["circular_std_decoded_residual"] / np.sqrt(max(1, r["ok_seeds"])) + 0.05
]
if failures:
    common.save_csv(failures, "R28_gkp_syndrome_FAILURES")
    raise AssertionError(f"{len(failures)} deltas show a biased syndrome estimate beyond statistical tolerance; see FAILURES csv")

common.save_result(rows, "R28_gkp_syndrome", extra={"cutoff": CUTOFF, "peak_width": PEAK_WIDTH, "envelope": ENVELOPE, "n_seeds": N_SEEDS})

# =============================================================================
# Final 2 x 2 publication figure
# =============================================================================

fig, axes = plt.subplots(2, 2, figsize=(10.0, 7.6), facecolor="white")
axA, axB, axC, axD = axes.ravel()

# -----------------------------------------------------------------------------
# A. Main result: ensemble circular mean tracks the ideal sawtooth
# -----------------------------------------------------------------------------

axA.errorbar(
    [r["delta"] for r in ok_rows], [r["circular_mean_decoded_residual"] for r in ok_rows],
    yerr=[r["circular_std_decoded_residual"] / np.sqrt(r["ok_seeds"]) for r in ok_rows],
    fmt="o-", color="red", capsize=3, label=f"Ensemble mean (N={N_SEEDS})",
)
axA.plot([r["delta"] for r in ok_rows], [r["expected_residual"] for r in ok_rows], "--", color="black",
          label=r"Ideal ($\delta$ mod SPACING)")
style_axis(axA, r"Imposed displacement $\delta$", "Decoded residual (mean $\\pm$ SEM)",
           "Ensemble syndrome decoding", "A")
axA.legend(frameon=False, fontsize=7)

# -----------------------------------------------------------------------------
# B. Correction actually reduces the mean |q| below the uncorrected value
# -----------------------------------------------------------------------------

axB.plot([r["delta"] for r in ok_rows], [r["mean_abs_corrected_q"] for r in ok_rows], "o-", color="blue",
          label=r"$\langle|q_{\rm corrected}|\rangle$")
axB.plot([r["delta"] for r in ok_rows], [r["abs_uncorrected_q"] for r in ok_rows], "--", color="black",
          label=r"$|\delta|$ (uncorrected)")
style_axis(axB, r"Imposed displacement $\delta$", "Residual $|q|$ after correction",
           "Correction effectiveness", "B")
axB.legend(frameon=False, fontsize=7)

# -----------------------------------------------------------------------------
# C. Single-shot spread: why an ensemble average is necessary
# -----------------------------------------------------------------------------

axC.plot([r["delta"] for r in ok_rows], [r["circular_std_decoded_residual"] for r in ok_rows], "o-", color="magenta")
style_axis(axC, r"Imposed displacement $\delta$", r"Circular std. $\sigma_\theta$",
           "Single-shot decoding noise", "C")

# -----------------------------------------------------------------------------
# D. Validation margin and sample completeness (norm-guard exclusions shown)
# -----------------------------------------------------------------------------

tolerance = [5 * r["circular_std_decoded_residual"] / np.sqrt(max(1, r["ok_seeds"])) + 0.05 for r in ok_rows]
axD.plot([r["delta"] for r in ok_rows], [r["circular_mean_error"] for r in ok_rows], "o-", color="red",
          label="Circular mean error")
axD.plot([r["delta"] for r in ok_rows], tolerance, "--", color="black", label="Statistical tolerance")
axD_twin = axD.twinx()
axD_twin.bar([r["delta"] for r in rows], [r["ok_seeds"] for r in rows], width=0.08, color="cyan", alpha=0.35,
             label=f"ok seeds (of {N_SEEDS})")
axD_twin.set_ylabel("ok seeds", color="cyan")
axD_twin.set_ylim(0, N_SEEDS * 1.2)
style_axis(axD, r"Imposed displacement $\delta$", "Circular mean error",
           "Validation margin & sample completeness", "D")
lines1, labels1 = axD.get_legend_handles_labels()
lines2, labels2 = axD_twin.get_legend_handles_labels()
axD.legend(lines1 + lines2, labels1 + labels2, frameon=False, fontsize=6.5, loc="upper center")

fig.suptitle(f"GKP $q$-syndrome extraction, finite ancilla (cutoff={CUTOFF}, width={PEAK_WIDTH})",
             fontsize=10.5, fontweight="bold", y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.96], h_pad=2.6, w_pad=2.6)
common.save_figure(fig, "R28_gkp_syndrome")
plt.close(fig)

common.print_summary(
    "R28 GKP syndrome extraction (ensemble statistics)",
    deltas=len(rows),
    seeds_per_delta=N_SEEDS,
    max_circular_mean_error=max((r["circular_mean_error"] for r in ok_rows), default=None),
    median_circular_std=float(np.median([r["circular_std_decoded_residual"] for r in ok_rows])) if ok_rows else None,
)


## R29 — Finite-Squeezing Noise Floor for Fixed Logical Circuits (Fig. 5)

R3/R4 showed the finite-squeezing noise floor for a *single* gate depends on
the synthesis shears needed to realize it. R29 fixes two representative
*compiled logical circuits* (a single-mode rotate+squeeze, and a two-mode CZ)
and sweeps the MBQC **resource** squeezing \(r\) used to compile them, to show
the noise floor as a function of \(r\) for a realistic multi-gate circuit —
the main-text noise-scaling result.

### Fixed circuits, varying resource

Because `Circuit.rotate` then `Circuit.squeeze` compose as
\(S_{\text{sq}}(0.3)\,R(0.5)\) (squeeze acts on the already-rotated state,
matching construction order), the single-mode target is

$$
S_{\text{single}} = S_{\text{sq}}(0.3)\,R(0.5),
$$

and the two-mode target is \(\mathrm{CZ}(0.6)\) (R6). Both are independently
built with NumPy. For each resource squeezing \(r\in[0.1,3.0]\) (log-spaced),
`gaussian_channel` gives the exact map \(S\) and noise \(N\); as established
in R3/R4, \(S\) is expected to stay fixed (only the synthesis shears, not
\(r\), enter the map), while \(N\) shrinks with \(r\).

### What is checked

Map error against the fixed analytic target (should stay at machine
precision for every \(r\)); \(\operatorname{Tr}(N)\), \(\|N\|_2\), and the
largest eigenvalue of \(N\) (its principal noise direction); and the resulting
output quadrature variances for a fixed coherent input through the
single-mode circuit.

### Success criteria

- Map error \(\lesssim10^{-7}\) for every \(r\), for both circuits (panel A)
  — confirms the logical operation itself is exactly \(r\)-independent, so
  every trend below is genuinely a noise effect.
- \(\operatorname{Tr}(N)\) falls with \(r\) (panel B) but the CZ circuit's
  noise floor sits at a different level than the rotate+squeeze circuit's,
  reflecting their different synthesis shear costs (R3/R4/R6).
- Output \(q\)/\(p\) variances (panel C) approach the ideal noiseless
  prediction as \(r\) grows.
- Spectral norm and max eigenvalue of \(N\) (panel D) track \(\operatorname{Tr}(N)\)
  consistently — the noise is not developing a hidden, disproportionately
  large single direction as \(r\) changes.


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg

set_reference_style()

RESOURCE_SQUEEZING = np.geomspace(0.1, 3.0, 20)
LOGICAL_ANGLE = 0.5
LOGICAL_SQUEEZE = 0.3
CZ_WEIGHT = 0.6
INPUT_ALPHA = 0.3 + 0.2j
MAP_TOL = 1e-7


def independent_rotation(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]])


def independent_squeezing(r):
    return np.array([[np.exp(-r), 0.0], [0.0, np.exp(r)]])


# Circuit.rotate() is applied before Circuit.squeeze() in construction order,
# so the composed map is S(squeeze) @ R(angle) (squeeze acts on the already-
# rotated state), not R @ S.
single_mode_target = independent_squeezing(LOGICAL_SQUEEZE) @ independent_rotation(LOGICAL_ANGLE)
two_mode_target = np.eye(4)
two_mode_target[1, 2] = CZ_WEIGHT
two_mode_target[3, 0] = CZ_WEIGHT

single_rows, two_rows = [], []
for r in RESOURCE_SQUEEZING:
    r = float(r)
    circuit = pg.Circuit(1).rotate(0, LOGICAL_ANGLE).squeeze(0, LOGICAL_SQUEEZE)
    pattern = circuit.compile(squeezing=r)
    channel = pg.gaussian_channel(pattern)
    input_state = pg.GaussianInput.coherent(INPUT_ALPHA)
    applied = channel.apply(input_state.state(0))
    single_rows.append(
        {
            "resource_squeezing": r,
            "map_frobenius_error": common.frobenius_error(channel.matrix, single_mode_target),
            "noise_trace": float(np.trace(channel.noise)),
            "noise_spectral_norm": float(np.linalg.norm(channel.noise, ord=2)),
            "noise_max_eigenvalue": float(np.linalg.eigvalsh(channel.noise).max()),
            "output_q_variance": float(applied.covariance[0, 0]),
            "output_p_variance": float(applied.covariance[1, 1]),
        }
    )

    circuit2 = pg.Circuit(2).cz(0, 1, CZ_WEIGHT)
    pattern2 = circuit2.compile(squeezing=r)
    channel2 = pg.gaussian_channel(pattern2)
    two_rows.append(
        {
            "resource_squeezing": r,
            "map_frobenius_error": common.frobenius_error(channel2.matrix, two_mode_target),
            "noise_trace": float(np.trace(channel2.noise)),
            "noise_spectral_norm": float(np.linalg.norm(channel2.noise, ord=2)),
            "noise_max_eigenvalue": float(np.linalg.eigvalsh(channel2.noise).max()),
        }
    )

# -----------------------------------------------------------------------------
# Validation
# -----------------------------------------------------------------------------

failures = [r for r in single_rows + two_rows if r["map_frobenius_error"] > MAP_TOL]
if failures:
    common.save_csv(failures, "R29_finite_squeezing_FAILURES")
    raise AssertionError("Resource squeezing altered the ideal logical map beyond tolerance; see FAILURES csv")

common.save_result(single_rows, "R29_finite_squeezing_single_mode", extra={"angle": LOGICAL_ANGLE, "squeeze": LOGICAL_SQUEEZE})
common.save_result(two_rows, "R29_finite_squeezing_cz", extra={"weight": CZ_WEIGHT})

# =============================================================================
# Final 2 x 2 publication figure
# =============================================================================

rs = [row["resource_squeezing"] for row in single_rows]
floor = 1e-16

fig, axes = plt.subplots(2, 2, figsize=(9.8, 7.4), facecolor="white")
axA, axB, axC, axD = axes.ravel()

# -----------------------------------------------------------------------------
# A. Map validation: the ideal logical operation is r-independent
# -----------------------------------------------------------------------------

axA.semilogy(rs, np.maximum([r["map_frobenius_error"] for r in single_rows], floor),
             "o-", color="red", label="Rotate+Squeeze circuit")
axA.semilogy(rs, np.maximum([r["map_frobenius_error"] for r in two_rows], floor),
             "s--", color="blue", label="CZ circuit")
axA.axhline(MAP_TOL, color="black", linestyle="--", linewidth=0.8, label="Tolerance")
axA.set_xscale("log")
style_axis(axA, "Resource squeezing $r$", "Map Frobenius error", "Logical map is $r$-independent", "A")
axA.legend(frameon=False, fontsize=7)

# -----------------------------------------------------------------------------
# B. Main result: added-noise trace vs resource squeezing
# -----------------------------------------------------------------------------

axB.loglog(rs, [row["noise_trace"] for row in single_rows], "o-", color="red", label="Rotate+Squeeze circuit")
axB.loglog(rs, [row["noise_trace"] for row in two_rows], "s-", color="blue", label="CZ circuit")
style_axis(axB, "Resource squeezing $r$", r"$\mathrm{Tr}(N)$", "Compiled-circuit noise floor", "B")
axB.legend(frameon=False, fontsize=7)

# -----------------------------------------------------------------------------
# C. Output quadrature variance approaching the noiseless limit
# -----------------------------------------------------------------------------

axC.semilogx(rs, [row["output_q_variance"] for row in single_rows], "o-", color="magenta", label="Output $q$ variance")
axC.semilogx(rs, [row["output_p_variance"] for row in single_rows], "s-", color="cyan", label="Output $p$ variance")
style_axis(axC, "Resource squeezing $r$", "Output variance", "Rotate+Squeeze circuit output", "C")
axC.legend(frameon=False, fontsize=7)

# -----------------------------------------------------------------------------
# D. Noise spectral norm and max eigenvalue: no hidden anisotropy growth
# -----------------------------------------------------------------------------

axD.loglog(rs, [row["noise_spectral_norm"] for row in single_rows], "o-", color="red", label=r"$\|N\|_2$ single")
axD.loglog(rs, [row["noise_max_eigenvalue"] for row in single_rows], "^:", color="red", alpha=0.6, label=r"$\lambda_{\max}(N)$ single")
axD.loglog(rs, [row["noise_spectral_norm"] for row in two_rows], "s-", color="blue", label=r"$\|N\|_2$ CZ")
axD.loglog(rs, [row["noise_max_eigenvalue"] for row in two_rows], "d:", color="blue", alpha=0.6, label=r"$\lambda_{\max}(N)$ CZ")
style_axis(axD, "Resource squeezing $r$", "Noise operator norm", "Noise spectral structure", "D")
axD.legend(frameon=False, fontsize=6.5)

fig.suptitle("Finite resource squeezing: MBQC compilation noise floor for fixed logical circuits",
             fontsize=10.5, fontweight="bold", y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.96], h_pad=2.4, w_pad=2.4)
common.save_figure(fig, "R29_finite_squeezing")
plt.close(fig)

common.print_summary(
    "R29 finite squeezing noise scan (main-text)",
    resource_squeezing_points=len(RESOURCE_SQUEEZING),
    max_map_error=max(r["map_frobenius_error"] for r in single_rows + two_rows),
    noise_trace_range=(min(r["noise_trace"] for r in single_rows), max(r["noise_trace"] for r in single_rows)),
)


## R30 — Loss / Attenuation: Exact Analytic Validation

R30 checks the `Loss(eta, nbar=0)` channel against the two textbook
attenuator identities for a coherent state, computed independently with
NumPy:

$$
\alpha_{\text{out}} = \sqrt{\eta}\,\alpha_{\text{in}},
\qquad
\langle n\rangle_{\text{out}} = \eta\,\langle n\rangle_{\text{in}},
$$

for transmissivity \(\eta\in(0,1]\), on **both** exact Gaussian backends
(NumPy and Piquasso).

### What is checked

For 20 values of \(\eta\): mean-vector error against
\(\mu_{\text{out}}=(2\sqrt\eta\operatorname{Re}\alpha,\,2\sqrt\eta\operatorname{Im}\alpha)\)
and mean-photon-number error against \(\eta|\alpha|^2\), on each backend.

### Success criteria

- Both backends match both analytic formulas to machine precision
  (\(\lesssim10^{-9}\)) for every \(\eta\) (panel B) — this is an exact
  linear-optics identity, not an approximation, so any measurable deviation
  would indicate a genuine implementation bug.
- The physical attenuation curve itself (panel A) shows the expected linear
  \(\eta|\alpha|^2\) decay, with both backends visually indistinguishable
  from the analytic line.


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg

set_reference_style()

ETAS = np.linspace(0.05, 1.0, 20)
ALPHA = 0.6 + 0.4j
TOL = 1e-9

rows = []
for eta in ETAS:
    eta = float(eta)
    pattern = pg.Pattern(inputs=(0,)).append(pg.Loss(0, eta, 0.0)).append(pg.Output((0,))).validate()
    expected_alpha = np.sqrt(eta) * ALPHA
    expected_mean = np.array([2 * expected_alpha.real, 2 * expected_alpha.imag])
    expected_n = eta * abs(ALPHA) ** 2

    row = {"eta": eta}
    for backend in ("gaussian", "piquasso"):
        state = pg.simulate(pattern, backend=backend, inputs={0: pg.GaussianInput.coherent(ALPHA)}, seed=0).state
        row[f"{backend}_mean_error"] = common.frobenius_error(state.mean, expected_mean)
        row[f"{backend}_photon_number_error"] = abs(state.photon_number(0) - expected_n)
        row[f"{backend}_photon_number"] = state.photon_number(0)
    row["expected_photon_number"] = expected_n
    rows.append(row)

# -----------------------------------------------------------------------------
# Validation
# -----------------------------------------------------------------------------

failures = [
    r for r in rows
    if r["gaussian_mean_error"] > TOL or r["piquasso_mean_error"] > TOL
    or r["gaussian_photon_number_error"] > TOL or r["piquasso_photon_number_error"] > TOL
]
if failures:
    common.save_csv(failures, "R30_loss_validation_FAILURES")
    raise AssertionError("Loss channel disagreed with the analytic attenuation formulas; see FAILURES csv")

common.save_result(rows, "R30_loss_validation", extra={"alpha": [ALPHA.real, ALPHA.imag]})

# =============================================================================
# Final 1 x 2 publication figure
# =============================================================================

etas = [r["eta"] for r in rows]
floor = 1e-18

fig, (axA, axB) = plt.subplots(1, 2, figsize=(9.6, 4.2), facecolor="white")

axA.plot(etas, [r["expected_photon_number"] for r in rows], "-", color="black", label=r"Analytic: $\eta|\alpha|^2$")
axA.plot(etas, [r["gaussian_photon_number"] for r in rows], "o", markersize=4, color="blue", label="NumPy Gaussian backend")
axA.plot(etas, [r["piquasso_photon_number"] for r in rows], "x", markersize=5, color="magenta", label="Piquasso backend")
style_axis(axA, "Transmissivity $\\eta$", "Output mean photon number", "Attenuation curve", "A")
axA.legend(frameon=False, fontsize=7)

axB.semilogy(etas, np.maximum([r["gaussian_mean_error"] for r in rows], floor), "o-", color="red", label="Gaussian mean error")
axB.semilogy(etas, np.maximum([r["piquasso_mean_error"] for r in rows], floor), "s--", color="blue", label="Piquasso mean error")
axB.semilogy(etas, np.maximum([r["gaussian_photon_number_error"] for r in rows], floor), "^:", color="magenta", label=r"Gaussian $\langle n\rangle$ error")
axB.semilogy(etas, np.maximum([r["piquasso_photon_number_error"] for r in rows], floor), "d-.", color="cyan", label=r"Piquasso $\langle n\rangle$ error")
axB.axhline(TOL, color="black", linestyle="--", linewidth=0.8, label="Tolerance")
style_axis(axB, "Transmissivity $\\eta$", "Error vs. analytic formula", "Exact-identity validation", "B")
axB.legend(frameon=False, fontsize=6.5)

fig.suptitle(r"Loss channel: $\alpha_{\rm out}=\sqrt{\eta}\alpha_{\rm in}$, $\langle n\rangle_{\rm out}=\eta\langle n\rangle_{\rm in}$",
             fontsize=10, fontweight="bold", y=1.02)
fig.tight_layout(rect=[0, 0, 1, 0.92], w_pad=2.2)
common.save_figure(fig, "R30_loss_validation")
plt.close(fig)

common.print_summary(
    "R30 loss / attenuation validation",
    eta_points=len(rows),
    max_mean_error=max(max(r["gaussian_mean_error"], r["piquasso_mean_error"]) for r in rows),
    max_photon_error=max(max(r["gaussian_photon_number_error"], r["piquasso_photon_number_error"]) for r in rows),
)


## R31 — Thermal Attenuation: Exact Analytic Validation

R31 generalizes R30 to a **noisy** attenuator: transmissivity \(\eta\) with a
thermal environment of mean occupation \(\bar n\) (\(\bar n=0\) reduces to
R30). The independent NumPy target for the thermal-loss channel (statistical
covariance, vacuum \(V=I\)) is

$$
\mu_{\text{out}} = \sqrt{\eta}\,\mu_{\text{in}},
\qquad
V_{\text{out}} = \eta\,V_{\text{in}} + (1-\eta)(2\bar n+1)\,I,
$$

i.e. attenuation shrinks the input covariance by \(\eta\) while injecting
isotropic thermal noise scaled by the *excess* (non-vacuum) environment
occupation \((1-\eta)(2\bar n+1)\).

### What is checked

For every combination of \(\eta\in\{0.2,0.4,0.6,0.8,1.0\}\) and
\(\bar n\in\{0,0.3,0.8,1.5,3.0\}\) (25 cases): mean and covariance error
against the formula above, on both exact Gaussian backends.

### Success criteria

- Both backends match the analytic thermal-channel formula to machine
  precision (\(\lesssim10^{-9}\)) for **every** \((\eta,\bar n)\) pair (panel
  B: error heatmap over the full grid — no isolated bad corner).
- The physical output-variance curves (panel A) show the expected behavior:
  \(\bar n=0\) monotonically shrinks toward vacuum as \(\eta\to0\), while
  larger \(\bar n\) raises the noise floor and can make the output variance
  *increase* as \(\eta\) decreases (net environment heating dominating over
  attenuation) — both physical regimes are visible directly in the curve
  family.


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg

set_reference_style()

ETAS = [0.2, 0.4, 0.6, 0.8, 1.0]
NBARS = [0.0, 0.3, 0.8, 1.5, 3.0]
ALPHA = 0.5 - 0.3j
TOL = 1e-9


def analytic_thermal_channel(eta: float, nbar: float, mean_in: np.ndarray, cov_in: np.ndarray):
    # mu_out = sqrt(eta) mu_in
    # V_out  = eta V_in + (1-eta)(2 nbar + 1) I
    mean_out = np.sqrt(eta) * mean_in
    cov_out = eta * cov_in + (1 - eta) * (2 * nbar + 1) * np.eye(2)
    return mean_out, cov_out


input_state = pg.GaussianInput.coherent(ALPHA)
mean_in = np.array(input_state.mean)
cov_in = np.array(input_state.covariance)

rows = []
for eta in ETAS:
    for nbar in NBARS:
        pattern = pg.Pattern(inputs=(0,)).append(pg.Loss(0, eta, nbar)).append(pg.Output((0,))).validate()
        expected_mean, expected_cov = analytic_thermal_channel(eta, nbar, mean_in, cov_in)

        row = {"eta": eta, "nbar": nbar}
        for backend in ("gaussian", "piquasso"):
            state = pg.simulate(pattern, backend=backend, inputs={0: input_state}, seed=0).state
            row[f"{backend}_mean_error"] = common.frobenius_error(state.mean, expected_mean)
            row[f"{backend}_covariance_error"] = common.frobenius_error(state.covariance, expected_cov)
        rows.append(row)

# -----------------------------------------------------------------------------
# Validation
# -----------------------------------------------------------------------------

failures = [
    r for r in rows
    if r["gaussian_mean_error"] > TOL or r["piquasso_mean_error"] > TOL
    or r["gaussian_covariance_error"] > TOL or r["piquasso_covariance_error"] > TOL
]
if failures:
    common.save_csv(failures, "R31_thermal_noise_FAILURES")
    raise AssertionError("Thermal loss channel disagreed with the analytic formula; see FAILURES csv")

common.save_result(rows, "R31_thermal_noise", extra={"alpha": [ALPHA.real, ALPHA.imag]})

# =============================================================================
# Final 1 x 2 publication figure
# =============================================================================

fig, (axA, axB) = plt.subplots(1, 2, figsize=(10.4, 4.4), facecolor="white")

# -----------------------------------------------------------------------------
# A. Physical result: output variance vs eta, per thermal occupation
# -----------------------------------------------------------------------------

for nbar in NBARS:
    subset = [r for r in rows if r["nbar"] == nbar]
    variances = []
    for r in subset:
        _, cov = analytic_thermal_channel(r["eta"], r["nbar"], mean_in, cov_in)
        variances.append(cov[0, 0])
    axA.plot([r["eta"] for r in subset], variances, "o-", label=f"$\\bar n$={nbar}")
style_axis(axA, "Transmissivity $\\eta$", "Output $q$ variance", "Thermal attenuation channel", "A")
axA.legend(frameon=False, fontsize=7, title="Thermal occupation")

# -----------------------------------------------------------------------------
# B. Validation heatmap over the full (eta, nbar) grid
# -----------------------------------------------------------------------------

grid = np.array([[max(next(r["gaussian_covariance_error"] for r in rows if r["eta"] == e and r["nbar"] == n), 1e-18)
                   for e in ETAS] for n in NBARS])
im = axB.imshow(np.log10(grid), cmap="viridis", aspect="auto", origin="lower")
axB.set_xticks(range(len(ETAS)))
axB.set_xticklabels(ETAS)
axB.set_yticks(range(len(NBARS)))
axB.set_yticklabels(NBARS)
style_axis(axB, "$\\eta$", "$\\bar n$", "Covariance error ($\\log_{10}$, Gaussian backend)", "B")
fig.colorbar(im, ax=axB, fraction=0.046)

fig.suptitle("Thermal loss channel: physical curve and exact-identity validation",
             fontsize=10, fontweight="bold", y=1.02)
fig.tight_layout(rect=[0, 0, 1, 0.92], w_pad=2.4)
common.save_figure(fig, "R31_thermal_noise")
plt.close(fig)

common.print_summary(
    "R31 thermal noise validation",
    cases=len(rows),
    max_mean_error=max(max(r["gaussian_mean_error"], r["piquasso_mean_error"]) for r in rows),
    max_covariance_error=max(max(r["gaussian_covariance_error"], r["piquasso_covariance_error"]) for r in rows),
)


## R32 — Detector Inefficiency / Noisy Homodyne

R32 checks the documented added-variance formula for a realistic,
imperfect homodyne detector: finite efficiency \(\eta_{\text{det}}<1\) and
additive electronic noise both broaden the measured outcome distribution
beyond the ideal quadrature variance. Two backends are used for two
different purposes: the exact Gaussian backend gives a statistically
powerful check of the formula itself; the experimental mixed-Fock backend
gives a much smaller, explicitly-labeled finite-cutoff cross-check (its
noisy path is documented to cost seconds *per shot*, so its sample size is
necessarily small).

### The added-variance formula

For a homodyne measurement with detector efficiency \(\eta_{\text{det}}\) and
electronic noise \(\nu_{\text{elec}}\) (`docs/theory.md`, "Destructive
Gaussian measurements"), the sampled outcome variance is

$$
\operatorname{Var}(x_{\text{measured}}) = \operatorname{Var}(x_{\text{ideal}})
+ \frac{1-\eta_{\text{det}}}{\eta_{\text{det}}} + \nu_{\text{elec}}.
$$

For a coherent-state input, \(\operatorname{Var}(x_{\text{ideal}})=1\)
(\(\hbar=2\)), so the predicted variance is
\(1+(1-\eta_{\text{det}})/\eta_{\text{det}}+\nu_{\text{elec}}\), computed
independently here (not by calling into the measurement backend).

### What is checked

- **Exact backend, statistically powered** (\(M=6000\) shots): sampled
  outcome variance vs. the formula above, for
  \(\eta_{\text{det}}\in\{1.0,0.9,0.7,0.5\}\times\nu_{\text{elec}}\in\{0,0.2,0.6\}\)
  (12 cases). The tolerance is a **6-sigma sampling band**
  (\(6\times\text{Var}\sqrt{2/(M-1)}\)) — a statistically justified bound for
  a variance estimate from \(M\) draws, not an arbitrary number.
- **Mixed-Fock cross-check** (finite cutoff 10, small shot counts because the
  noisy path integrates a dense density matrix per shot): the same formula,
  compared with a correspondingly looser, explicitly reported tolerance.

### Success criteria

- All 12 exact-backend cases fall within the 6-sigma sampling band (panel A:
  sampled vs. analytic variance; panel B: absolute error vs. the band).
- Mixed-Fock cases that complete land close to the same formula (panel C);
  any cutoff-guard failures are counted, not hidden.


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg

set_reference_style()

SHOTS = 6000
SEED = 20260913
ALPHA = 0.4 + 0.2j
EFFICIENCIES = [1.0, 0.9, 0.7, 0.5]
NOISES = [0.0, 0.2, 0.6]
MIXED_FOCK_CUTOFF = 10
# The mixed-Fock noisy-homodyne path convolves a dense density matrix with a
# Gaussian electronics kernel via scipy.integrate.quad_vec inside a brentq
# root-find, PER SHOT, whenever noise!=0: measured at ~8s/shot at cutoff=10
# (and >13s/shot at cutoff=20) on the reference machine. Shot counts below
# are deliberately small for the noisy cases and are a coarse cross-check,
# not a high-precision statistical estimate; the exact Gaussian-backend sweep
# above is the statistically powered result.
MIXED_FOCK_SHOTS_NOISELESS = 200
MIXED_FOCK_SHOTS_NOISY = 6


def independent_variance(prior_variance: float, efficiency: float, noise: float) -> float:
    """Documented formula: nu=(1-eta)/eta + noise, outcomes calibrated to the incident quadrature."""
    return prior_variance + (1 - efficiency) / efficiency + noise


rows = []
for efficiency in EFFICIENCIES:
    for noise in NOISES:
        pattern = pg.Pattern(inputs=(0,)).append(
            pg.Measure(0, pg.Homodyne(angle=0.0, efficiency=efficiency, noise=noise), key="m")
        )
        shots = pg.run_shots(
            pattern, SHOTS, backend="gaussian", inputs={0: pg.GaussianInput.coherent(ALPHA)}, seed=SEED
        )
        samples = shots.values("m")
        empirical_variance = float(np.var(samples, ddof=1))
        predicted_variance = independent_variance(1.0, efficiency, noise)  # coherent input: prior q variance = 1
        # Sampling error of a variance estimate from N draws scales like
        # variance*sqrt(2/(N-1)); use a 6-sigma band for a robust, non-flaky check.
        sampling_error = predicted_variance * np.sqrt(2 / (SHOTS - 1))
        rows.append(
            {
                "efficiency": efficiency,
                "noise": noise,
                "empirical_variance": empirical_variance,
                "predicted_variance": predicted_variance,
                "absolute_error": abs(empirical_variance - predicted_variance),
                "sampling_error_6sigma": 6 * sampling_error,
                "within_tolerance": abs(empirical_variance - predicted_variance) <= 6 * sampling_error,
            }
        )

failures = [r for r in rows if not r["within_tolerance"]]
if failures:
    common.save_csv(failures, "R32_noisy_measurement_FAILURES")
    raise AssertionError(f"{len(failures)} noisy-homodyne cases exceeded the statistical tolerance; see FAILURES csv")

# -----------------------------------------------------------------------------
# Mixed-Fock cross-check (experimental, finite-cutoff approximation).
# -----------------------------------------------------------------------------

mixed_rows = []
for efficiency in (1.0, 0.8):
    for noise in (0.0, 0.3):
        pattern = pg.Pattern(inputs=(0,)).append(
            pg.Measure(0, pg.Homodyne(angle=0.0, efficiency=efficiency, noise=noise), key="m")
        )
        shots = MIXED_FOCK_SHOTS_NOISELESS if noise == 0.0 else MIXED_FOCK_SHOTS_NOISY
        failure = common.safe_cutoff_run(
            lambda efficiency=efficiency, noise=noise, shots=shots: pg.run_shots(
                pattern, shots, backend="piquasso-mixed-fock", cutoff=MIXED_FOCK_CUTOFF,
                inputs={0: pg.GaussianInput.coherent(ALPHA)}, seed=SEED,
            ),
            label=f"mixed-fock eff={efficiency} noise={noise}",
        )
        if not failure["ok"]:
            mixed_rows.append({"efficiency": efficiency, "noise": noise, "ok": False, "error": failure["error"]})
            continue
        samples = failure["value"].values("m")
        empirical_variance = float(np.var(samples, ddof=1))
        predicted_variance = independent_variance(1.0, efficiency, noise)
        mixed_rows.append(
            {
                "efficiency": efficiency, "noise": noise, "ok": True,
                "empirical_variance": empirical_variance, "predicted_variance": predicted_variance,
                "absolute_error": abs(empirical_variance - predicted_variance),
                "cutoff": MIXED_FOCK_CUTOFF, "shots": shots,
            }
        )

common.save_result(
    rows, "R32_noisy_measurement",
    extra={"shots": SHOTS, "alpha": [ALPHA.real, ALPHA.imag], "mixed_fock": mixed_rows},
)
common.save_csv(mixed_rows, "R32_noisy_measurement_mixed_fock")

# =============================================================================
# Final 1 x 3 publication figure
# =============================================================================

fig, (axA, axB, axC) = plt.subplots(1, 3, figsize=(13.4, 4.2), facecolor="white")

# -----------------------------------------------------------------------------
# A. Sampled vs. analytic outcome variance (main physical result)
# -----------------------------------------------------------------------------

for noise in NOISES:
    subset = [r for r in rows if r["noise"] == noise]
    axA.plot([r["efficiency"] for r in subset], [r["predicted_variance"] for r in subset], "-", color="black")
    axA.plot([r["efficiency"] for r in subset], [r["empirical_variance"] for r in subset], "o", label=f"noise={noise}")
style_axis(axA, "Detector efficiency", "Outcome variance", "Sampled vs. formula (lines=analytic)", "A")
axA.legend(frameon=False, fontsize=7)

# -----------------------------------------------------------------------------
# B. Statistical validation: error vs. the 6-sigma sampling band
# -----------------------------------------------------------------------------

x = np.arange(len(rows))
axB.bar(x, [r["absolute_error"] for r in rows], color="blue", label="Absolute error")
axB.plot(x, [r["sampling_error_6sigma"] for r in rows], "--", color="black", label=r"6$\sigma$ tolerance")
axB.set_xticks(x)
axB.set_xticklabels([f"$\\eta$={r['efficiency']}\nn={r['noise']}" for r in rows], fontsize=5.5, rotation=0)
style_axis(axB, "", "Variance error", "Exact-backend statistical validation", "B")
axB.legend(frameon=False, fontsize=7)

# -----------------------------------------------------------------------------
# C. Mixed-Fock cross-check (finite cutoff, small shot count -- reported)
# -----------------------------------------------------------------------------

ok_mixed = [r for r in mixed_rows if r["ok"]]
labels = [f"$\\eta$={r['efficiency']}\nn={r['noise']}" for r in mixed_rows]
xm = np.arange(len(mixed_rows))
axC.bar(xm, [r["predicted_variance"] if r["ok"] else 0 for r in mixed_rows], width=0.35,
        color="black", alpha=0.4, label="Analytic")
axC.bar(xm + 0.35, [r["empirical_variance"] if r["ok"] else 0 for r in mixed_rows], width=0.35,
        color="cyan", label="Mixed-Fock sampled")
for i, r in enumerate(mixed_rows):
    if not r["ok"]:
        axC.text(i + 0.175, 0.02, "guard", ha="center", fontsize=6, rotation=90, color="red")
axC.set_xticks(xm + 0.175)
axC.set_xticklabels(labels, fontsize=6)
style_axis(axC, "", "Outcome variance", f"Mixed-Fock cross-check (c={MIXED_FOCK_CUTOFF})", "C")
axC.legend(frameon=False, fontsize=7)

fig.suptitle("Noisy homodyne: documented added-variance formula", fontsize=10.5,
             fontweight="bold", y=1.02)
fig.tight_layout(rect=[0, 0, 1, 0.92], w_pad=2.4)
common.save_figure(fig, "R32_noisy_measurement")
plt.close(fig)

common.print_summary(
    "R32 detector inefficiency / noisy homodyne",
    cases=len(rows),
    max_absolute_error=max(r["absolute_error"] for r in rows),
    mixed_fock_ok=sum(1 for r in mixed_rows if r["ok"]),
)


## R33 — Fock Cutoff Convergence Across Non-Gaussian Workloads

Every non-Gaussian experiment in this suite (R19–R28) truncates an
infinite-dimensional Fock space at a finite cutoff. R33 checks that this
truncation actually **converges** as the cutoff grows, for four
representative workloads (cat state, photon subtraction, cubic phase, GKP
resource) spanning the different non-Gaussian mechanisms used elsewhere in
the suite.

### What "converged" means here

`cutoff_convergence` runs the same pattern at a strictly increasing sequence
of cutoffs and reports, at each step, the retained norm, boundary population
(probability mass sitting at the truncation edge — a direct truncation-error
indicator), and fidelity/trace distance **to the previous, coarser cutoff**.
This experiment additionally computes fidelity/trace distance to the
**highest tested cutoff**, computed independently here (not part of the study
object). The highest cutoff is explicitly a *reference point*, not an
infinite-cutoff ground truth — convergence must be judged from the *trend*
across the whole sequence, not merely from the last comparison being small.

### What is checked

For each workload, at 5 increasing cutoffs: retained norm, boundary
population, fidelity/trace distance to the previous cutoff, and fidelity to
the highest tested cutoff.

### Success criteria

- Infidelity to the highest cutoff **decreases monotonically** (up to small
  numerical noise) as the cutoff grows, for every workload (top row) — a
  genuine convergence trend, not a single lucky comparison.
- Fidelity to the *previous* cutoff approaches 1 (bottom row) — successive
  refinements agree increasingly well with each other, the standard signal
  that a truncated calculation has stabilized.
- Low-energy workloads (cat, photon subtraction, cubic phase) are expected to
  converge quickly even at the smallest tested cutoff; the GKP workload,
  built from a genuinely wide comb structure, is expected to show a visibly
  slower, more informative convergence trend across the full cutoff range
  24–96.


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg

set_reference_style()

WORKLOADS = {
    "cat_state": dict(
        pattern=pg.Pattern(inputs=()).append(pg.Prepare(0, state=pg.CatResource(1.5, parity=1))).append(pg.Output((0,))),
        cutoffs=[16, 24, 32, 48, 64],
        node=0,
        kwargs={},
    ),
    "photon_subtraction": dict(
        pattern=pg.non_gaussian.photon_subtraction(0.3),
        cutoffs=[8, 12, 16, 24, 32],
        node="in",
        kwargs={"inputs": {"in": pg.FockInput.number(2)}, "measurement_outcomes": {"count": 1}},
    ),
    "cubic_phase": dict(
        pattern=pg.Pattern(inputs=(0,)).append(pg.CubicPhase(0, 0.15)).append(pg.Output((0,))),
        cutoffs=[16, 24, 32, 48, 64],
        node=0,
        kwargs={"inputs": {0: pg.GaussianInput.coherent(0.3 + 0.0j)}},
    ),
    "gkp_resource": dict(
        pattern=pg.Pattern(inputs=()).append(pg.Prepare(0, state=pg.GKPResource(0, peak_width=0.4, envelope=0.4))).append(pg.Output((0,))),
        cutoffs=[24, 32, 48, 64, 96],
        node=0,
        kwargs={},
    ),
}

all_rows = {}
for name, spec in WORKLOADS.items():
    study = pg.cutoff_convergence(spec["pattern"], spec["cutoffs"], **spec["kwargs"])
    final_state = study.results[-1].state
    rows = []
    for row, result in zip(study.rows, study.results, strict=True):
        fidelity_to_highest = pg.fidelity(result.state, final_state) if row["cutoff"] != spec["cutoffs"][-1] else 1.0
        trace_distance_to_highest = pg.trace_distance(result.state, final_state) if row["cutoff"] != spec["cutoffs"][-1] else 0.0
        rows.append(
            {
                "workload": name,
                "cutoff": row["cutoff"],
                "dimension": common.fock_dimension(len(result.state.nodes), row["cutoff"]),
                "seconds": row["seconds"],
                "norm": row["norm"],
                "minimum_retained_norm": row["minimum_retained_norm"],
                "maximum_boundary_population": row["maximum_boundary_population"],
                "photon_number": row["photon_numbers"].get(spec["node"]),
                "fidelity_to_previous": row["fidelity_to_previous"],
                "trace_distance_to_previous": row["trace_distance_to_previous"],
                "fidelity_to_highest_cutoff": fidelity_to_highest,
                "trace_distance_to_highest_cutoff": trace_distance_to_highest,
            }
        )
    all_rows[name] = rows
    print(f"  {name}: " + ", ".join(f"c={r['cutoff']}:F={r['fidelity_to_highest_cutoff']:.6f}" for r in rows), flush=True)

flat_rows = [row for rows in all_rows.values() for row in rows]
common.save_result(flat_rows, "R33_fock_cutoff", extra={"workloads": list(WORKLOADS)})

# =============================================================================
# Final 2 x 4 publication figure: one column per workload
# =============================================================================

names = list(WORKLOADS)
fig, axes = plt.subplots(2, len(names), figsize=(4.0 * len(names), 6.6), facecolor="white")

for col, name in enumerate(names):
    rows = all_rows[name]
    letter_top = "ABCD"[col]
    letter_bot = "EFGH"[col]

    infidelities = [max(1e-16, 1 - r["fidelity_to_highest_cutoff"]) for r in rows[:-1]]
    axes[0, col].semilogy([r["cutoff"] for r in rows[:-1]], infidelities, "o-", color="red")
    style_axis(axes[0, col], "Fock cutoff",
               "1 - fidelity to highest cutoff" if col == 0 else "",
               name, letter_top)

    comparable = [r for r in rows if r["fidelity_to_previous"] is not None]
    axes[1, col].plot([r["cutoff"] for r in comparable], [r["fidelity_to_previous"] for r in comparable],
                       "o-", color="blue")
    axes[1, col].axhline(1.0, color="black", linestyle=":", linewidth=0.8)
    style_axis(axes[1, col], "Fock cutoff",
               "Fidelity to previous cutoff" if col == 0 else "",
               "", letter_bot)

fig.suptitle("Fock cutoff convergence (top: vs. highest tested cutoff; bottom: successive refinements)",
             fontsize=10.5, fontweight="bold", y=0.995)
fig.tight_layout(rect=[0, 0, 1, 0.95], h_pad=2.4, w_pad=2.0)
common.save_figure(fig, "R33_fock_cutoff")
plt.close(fig)

common.print_summary(
    "R33 Fock cutoff convergence",
    workloads=list(WORKLOADS),
    min_final_fidelity=min(rows[-2]["fidelity_to_highest_cutoff"] for rows in all_rows.values() if len(rows) > 1),
)


## R34 — Cubic Injection Convergence: 2D Scan Over Cutoff and Resource Squeezing

R23 swept resource squeezing at a few fixed cutoffs; R34 completes the
picture with a full two-dimensional grid over **both** Fock cutoff and
resource squeezing, so the two independent sources of error in finite-
resource cubic MBQC injection — finite Fock truncation and finite ancilla
squeezing — can be told apart.

### Two independent error sources, one fidelity number

Fidelity to the same independent NumPy/SciPy `CubicPhase` reference as
R22/R23 is computed at every \((\text{cutoff},r)\) grid point, with the
ancilla outcome fixed at \(m=0\) (isolating the resource envelope, R23). A
cell can fail to reach high fidelity for two structurally different reasons:
the cutoff may be too small to hold the injected state (a truncation
problem, fixable by raising the cutoff), or the resource squeezing may be too
small (an intrinsic finite-energy approximation error, fixable only by more
squeezing) — the 2D grid separates these visually in a way a 1D sweep at
fixed cutoff (R23) cannot.

### What is checked

Fidelity (and the Fock backend's own retained-norm safety guard outcome) at
every combination of cutoff \(\in\{24,32,48,64,96\}\) and resource squeezing
\(r\in\{0.1,\dots,2.0\}\) (30 cells), gamma fixed at 0.15.

### Success criteria

- The fidelity heatmap (panel A) should show fidelity increasing along the
  \(r\) axis at fixed cutoff (resource squeezing helps, as in R23) and be
  roughly flat along the cutoff axis once the cutoff is large enough for the
  given \(r\) (truncation is no longer the bottleneck) — a genuine
  **separation** of the two error mechanisms, not a single blended trend.
- Any cells where the Fock backend's norm guard triggers are shown explicitly
  (panel B), not silently absent from the heatmap.


In [ ]:
from __future__ import annotations

import numpy as np
from scipy.linalg import expm

import photographiq as pg

set_reference_style()

GAMMA = 0.15
ALPHA = 0.2 + 0.0j
CUTOFFS = [24, 32, 48, 64, 96]
SQUEEZING_VALUES = [0.1, 0.3, 0.6, 1.0, 1.5, 2.0]
FIXED_OUTCOME = 0.0


def independent_cubic_unitary(gamma, cutoff):
    a = np.diag(np.sqrt(np.arange(1, cutoff)), k=1)
    q = a + a.T
    return expm(1j * gamma * (q @ q @ q) / 6)


def truncated_input_vector(cutoff):
    pattern = pg.Pattern(inputs=(0,)).append(pg.Output((0,))).validate()
    state = pg.simulate(
        pattern, backend="piquasso-fock", cutoff=cutoff, inputs={0: pg.GaussianInput.coherent(ALPHA)}, seed=0
    ).state
    amplitudes = dict(zip(state.basis, state.state_vector, strict=True))
    return np.array([amplitudes.get((n,), 0j) for n in range(cutoff)])


rows = []
for cutoff in CUTOFFS:
    before = truncated_input_vector(cutoff)
    ideal_after = independent_cubic_unitary(GAMMA, cutoff) @ before
    ideal_after /= np.linalg.norm(ideal_after)

    for r in SQUEEZING_VALUES:
        pattern = pg.non_gaussian.cubic_injection(GAMMA, squeezing=r)
        failure = common.safe_cutoff_run(
            lambda: pg.simulate(
                pattern, backend="piquasso-fock", cutoff=cutoff,
                inputs={"in": pg.GaussianInput.coherent(ALPHA)},
                measurement_outcomes={"m": FIXED_OUTCOME}, seed=0,
            ),
            label=f"cutoff={cutoff} r={r}",
        )
        if not failure["ok"]:
            rows.append({"cutoff": cutoff, "resource_squeezing": r, "ok": False, "fidelity": None, "error": failure["error"]})
            continue
        result = failure["value"]
        amplitudes = dict(zip(result.state.basis, result.state.state_vector, strict=True))
        actual = np.array([amplitudes.get((n,), 0j) for n in range(cutoff)])
        fidelity = float(np.clip(abs(np.vdot(ideal_after, actual)) ** 2, 0, 1))
        rows.append(
            {
                "cutoff": cutoff, "resource_squeezing": r, "ok": True, "fidelity": fidelity,
                "log10_infidelity": common.safe_log10(1 - fidelity),
                "retained_norm": result.state.retained_norms[-1] if result.state.retained_norms else None,
            }
        )

common.save_result(rows, "R34_cubic_convergence", extra={"gamma": GAMMA, "alpha": [ALPHA.real, ALPHA.imag]})

ok_rows = [r for r in rows if r["ok"]]
fidelity_grid = np.full((len(SQUEEZING_VALUES), len(CUTOFFS)), np.nan)
ok_grid = np.zeros((len(SQUEEZING_VALUES), len(CUTOFFS)))
for r in rows:
    i = SQUEEZING_VALUES.index(r["resource_squeezing"])
    j = CUTOFFS.index(r["cutoff"])
    ok_grid[i, j] = 1.0 if r["ok"] else 0.0
    if r["ok"]:
        fidelity_grid[i, j] = r["fidelity"]

# =============================================================================
# Final 1 x 2 publication figure
# =============================================================================

fig, (axA, axB) = plt.subplots(1, 2, figsize=(11.4, 4.6), facecolor="white")

mesh = axA.imshow(fidelity_grid, aspect="auto", origin="lower", cmap="viridis", vmin=0, vmax=1)
axA.set_xticks(range(len(CUTOFFS)))
axA.set_xticklabels(CUTOFFS)
axA.set_yticks(range(len(SQUEEZING_VALUES)))
axA.set_yticklabels(SQUEEZING_VALUES)
style_axis(axA, "Fock cutoff", "Resource squeezing $r$",
           f"Fidelity to ideal CubicPhase($\\gamma$={GAMMA})", "A")
fig.colorbar(mesh, ax=axA, label="Fidelity")

mesh2 = axB.imshow(ok_grid, aspect="auto", origin="lower", cmap="RdYlGn", vmin=0, vmax=1)
axB.set_xticks(range(len(CUTOFFS)))
axB.set_xticklabels(CUTOFFS)
axB.set_yticks(range(len(SQUEEZING_VALUES)))
axB.set_yticklabels(SQUEEZING_VALUES)
style_axis(axB, "Fock cutoff", "Resource squeezing $r$",
           f"Norm-guard outcome ({len(ok_rows)}/{len(rows)} ok)", "B")
fig.colorbar(mesh2, ax=axB, label="1=ok, 0=guard-triggered")

fig.suptitle("Cubic MBQC injection: separating truncation error from finite-squeezing error",
             fontsize=10.5, fontweight="bold", y=1.01)
fig.tight_layout(rect=[0, 0, 1, 0.93], w_pad=2.6)
common.save_figure(fig, "R34_cubic_convergence")
plt.close(fig)

common.print_summary(
    "R34 cubic injection convergence (2D scan)",
    total_cells=len(rows),
    ok_cells=len(ok_rows),
    max_fidelity=max((r["fidelity"] for r in ok_rows), default=None),
    min_fidelity=min((r["fidelity"] for r in ok_rows), default=None),
)


## R35 — Fock-Space Dimension and Memory Model (Fig. 8 supporting)

R35 is analytical, not empirical: it evaluates the exact total-photon Fock
dimension formula the Fock backend itself uses
(`PiquassoFockBackend.dimension`), cross-checked against an independently
written combinatorial formula, over a full modes × cutoff grid, and converts
dimension into concrete memory costs.

### The dimension formula

For \(m\) modes truncated at total photon number \(<c\) (cutoff \(c\)), the
number of basis states is the standard "stars and bars" count

$$
D(m,c) = \binom{m+c-1}{m},
$$

cross-checked here against `math.comb` computed independently
(`common.fock_dimension`) rather than trusting the backend's own formula in
isolation. A pure state vector costs \(16D\) bytes (complex128); a dense
density matrix costs \(16D^2\) bytes — a rough dense **upper-bound** estimate,
not the actual (typically sparser/structured) memory footprint of any
specific backend implementation (`docs/performance.md`).

### What is checked

Dimension agreement between the two independent formulas at every
\((m,c)\) grid point (an exact integer equality, not a numerical tolerance);
resulting pure-vector and dense-density-matrix byte estimates; and how many
grid points exceed the backend's own default `max_dimension` allocation
guard.

### Success criteria

- The two dimension formulas agree exactly everywhere (a discrete identity;
  any single disagreement would indicate a real implementation divergence).
- Pure-vector memory grows extremely fast in cutoff, especially at higher
  mode counts (panel B) — this is *why* the Fock backend needs a
  `max_dimension` guard at all; the guard's default threshold is shown
  directly against the growth curves.
- The dimension heatmap (panel A) makes the combinatorial (not merely
  exponential-in-one-variable) growth visible across the full grid.


In [ ]:
from __future__ import annotations

import inspect

import numpy as np

from photographiq.backends.fock import PiquassoFockBackend

set_reference_style()

MODES = list(range(1, 21))
CUTOFFS = list(range(2, 41))


def default_max_dimension() -> int:
    return inspect.signature(PiquassoFockBackend.__init__).parameters["max_dimension"].default


max_dimension = default_max_dimension()

backend = PiquassoFockBackend(cutoff=CUTOFFS[0])
rows = []
for modes in MODES:
    for cutoff in CUTOFFS:
        backend.cutoff = cutoff
        dimension_native = backend.dimension(modes)
        dimension_independent = common.fock_dimension(modes, cutoff)
        if dimension_native != dimension_independent:
            raise AssertionError(f"Dimension formulas disagree at modes={modes}, cutoff={cutoff}")
        rows.append(
            {
                "modes": modes,
                "cutoff": cutoff,
                "dimension": dimension_native,
                "pure_vector_bytes": common.vector_bytes(dimension_native),
                "dense_density_matrix_bytes_rough_estimate": common.density_bytes(dimension_native),
                "exceeds_default_max_dimension": dimension_native > max_dimension,
            }
        )

common.save_result(rows, "R35_fock_dimension_scaling", extra={"default_max_dimension": max_dimension})

# =============================================================================
# Final 1 x 2 publication figure
# =============================================================================

dims = np.array([[common.fock_dimension(m, c) for c in CUTOFFS] for m in MODES])

fig, (axA, axB) = plt.subplots(1, 2, figsize=(11.0, 4.4), facecolor="white")

mesh = axA.imshow(
    np.log10(dims), aspect="auto", origin="lower", cmap="viridis",
    extent=[CUTOFFS[0], CUTOFFS[-1], MODES[0], MODES[-1]],
)
# Contour marking where dimension crosses the default max_dimension guard.
axA.contour(CUTOFFS, MODES, dims, levels=[max_dimension], colors="red", linewidths=1.2)
style_axis(axA, "Cutoff", "Modes", r"$\log_{10}$(Fock dimension)", "A")
fig.colorbar(mesh, ax=axA)

for modes in (1, 2, 4, 8, 16, 20):
    axB.semilogy(CUTOFFS, [common.vector_bytes(common.fock_dimension(modes, c)) for c in CUTOFFS], "o-",
                 label=f"{modes} modes", markersize=3)
axB.axhline(common.vector_bytes(max_dimension), color="red", linestyle="--", label="default max_dimension")
style_axis(axB, "Cutoff", "Pure state-vector bytes", "Memory scaling", "B")
axB.legend(frameon=False, fontsize=6.5)

fig.suptitle(r"Fock-space dimension and memory scaling ($D=\binom{m+c-1}{m}$)",
             fontsize=10.5, fontweight="bold", y=1.01)
fig.tight_layout(rect=[0, 0, 1, 0.93], w_pad=2.4)
common.save_figure(fig, "R35_fock_dimension_scaling")
plt.close(fig)

common.print_summary(
    "R35 Fock-space dimension and memory model",
    grid_points=len(rows),
    default_max_dimension=max_dimension,
    max_dimension_in_grid=max(r["dimension"] for r in rows),
    n_exceeding_default_guard=sum(r["exceeds_default_max_dimension"] for r in rows),
)


## R36 — Gaussian Backend Runtime Scaling (Fig. 8)

R36 benchmarks preparing and entangling an \(N\)-mode line resource through
both exact Gaussian backends, with warm-up runs and repeated timing
(`common.benchmark`: median, IQR, std from `time.perf_counter`). These are
**single-machine wall-clock observations**, not a comparative performance
claim across hardware or a proof of asymptotic complexity
(`docs/performance.md`, `docs/validation.md`).

### What a log-log fit means here

A least-squares fit of \(\log(\text{median runtime})\) vs. \(\log(N)\) gives
an *empirical* exponent \(p\) in runtime\(\;\approx a N^{p}\) over the
specific sampled range \(N\in\{10,20,50,100,200\}\) on this machine — reported
as an observation, never as a proven complexity bound.

### What is checked

Median runtime, standard deviation and IQR of `simulate(...)` for both the
NumPy Gaussian backend and the Piquasso Gaussian backend, at 5 mode counts.

### Success criteria

- Runtime grows with mode count for both backends (panel A), with the fitted
  log-log exponent reported per backend rather than asserted against a
  theoretical value — this experiment measures, it does not claim, a
  complexity class.
- The measurement is reproducible in trend (small IQR relative to the median)
  even though absolute timings are specific to this machine.


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg

set_reference_style()

MODE_COUNTS = [10, 20, 50, 100, 200]


def build_pattern(n_modes: int) -> pg.Pattern:
    graph = pg.CVGraph.line(n_modes, squeezing=1.0, inputs=(0,))
    return pg.Pattern(graph)


def fitted_exponent(n, y):
    slope, _ = np.polyfit(np.log(n), np.log(y), 1)
    return float(slope)


rows = []
for n_modes in MODE_COUNTS:
    pattern = build_pattern(n_modes)
    for backend in ("gaussian", "piquasso"):
        warmup, repeats = (2, 9) if n_modes <= 50 else (1, 5)
        timing = common.benchmark(
            lambda pattern=pattern, backend=backend: pg.simulate(pattern, backend=backend, seed=0),
            warmup=warmup, repeats=repeats,
        )
        rows.append(
            {
                "modes": n_modes,
                "commands": len(pattern.commands),
                "backend": backend,
                "median_seconds": timing["median_seconds"],
                "std_seconds": timing["std_seconds"],
                "iqr_seconds": timing["iqr_seconds"],
                "min_seconds": timing["min_seconds"],
                "max_seconds": timing["max_seconds"],
                "repeats": timing["repeats"],
                "warmup": timing["warmup"],
            }
        )
        print(f"  modes={n_modes} backend={backend}: median={timing['median_seconds']:.5f}s", flush=True)

common.save_result(rows, "R36_gaussian_runtime", extra={"environment": metadata.collect()})

exponents = {}
for backend in ("gaussian", "piquasso"):
    subset = [r for r in rows if r["backend"] == backend]
    exponents[backend] = fitted_exponent([r["modes"] for r in subset], [r["median_seconds"] for r in subset])

# =============================================================================
# Final figure
# =============================================================================

fig, ax = plt.subplots(figsize=(6.4, 4.4), facecolor="white")
for backend, color in (("gaussian", "red"), ("piquasso", "blue")):
    subset = [r for r in rows if r["backend"] == backend]
    ax.errorbar(
        [r["modes"] for r in subset], [r["median_seconds"] for r in subset],
        yerr=[r["iqr_seconds"] / 2 for r in subset], fmt="o-", label=f"{backend} ($p$={exponents[backend]:.2f})",
        color=color, capsize=3,
    )
ax.set_xscale("log")
ax.set_yscale("log")
style_axis(ax, "Number of modes (line resource)", "Median runtime (s)",
           "Gaussian backend runtime scaling (this machine only)", "A")
ax.legend(frameon=False, fontsize=8)
fig.tight_layout()
common.save_figure(fig, "R36_gaussian_runtime")
plt.close(fig)

common.print_summary("R36 Gaussian runtime scaling", mode_counts=MODE_COUNTS, fitted_exponents=exponents)


## R37 — Compiler Runtime Scaling

R37 isolates `Circuit.compile` itself (not execution) from R10's resource
counting: it measures wall-clock compile time, with warm-up and repeated
timing, for a growing chain of alternating rotation/CZ logical gates on two
modes. R10 already established (empirically) that resource-node/measurement
counts scale linearly in gate count; R37 asks whether the *compiler's own
runtime* follows the same trend or adds extra overhead.

### What is checked

Median compile time (`common.benchmark`) alongside resulting resource-node,
command and measurement counts, for gate counts 1–100.

### Success criteria

- Compile time grows with the number of gates (panel A), with an empirical
  log-log exponent reported (not asserted against a theoretical value — this
  is a runtime measurement, per R10's discussion of what a power-law fit
  can and cannot claim here).
- Resource-node and command counts (panel B) reproduce R10's linear-in-gate-
  count trend for this specific alternating rotation/CZ circuit, confirming
  consistency between the two independent scaling experiments.


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg

set_reference_style()

GATE_COUNTS = [1, 2, 5, 10, 20, 50, 100]
SQUEEZING = 1.0


def build_circuit(n_gates: int) -> pg.Circuit:
    circuit = pg.Circuit(2)
    for i in range(n_gates):
        if i % 2 == 0:
            circuit.rotate(i % 2, 0.1 + 0.01 * i)
        else:
            circuit.cz(0, 1, 0.2 + 0.01 * i)
    return circuit


def fitted_exponent(n, y):
    slope, _ = np.polyfit(np.log(n), np.log(y), 1)
    return float(slope)


rows = []
for n_gates in GATE_COUNTS:
    circuit = build_circuit(n_gates)
    warmup, repeats = (2, 9) if n_gates <= 20 else (1, 5)
    timing = common.benchmark(
        lambda circuit=circuit: circuit.compile(squeezing=SQUEEZING, return_trace=True),
        warmup=warmup, repeats=repeats,
    )
    pattern, trace = timing["result"]
    measurements = sum(1 for c in pattern.commands if isinstance(c, pg.Measure))
    rows.append(
        {
            "n_gates": n_gates,
            "median_compile_seconds": timing["median_seconds"],
            "std_compile_seconds": timing["std_seconds"],
            "iqr_compile_seconds": timing["iqr_seconds"],
            "repeats": timing["repeats"],
            "warmup": timing["warmup"],
            "resource_nodes": len(pattern.graph.nodes) - circuit.modes,
            "commands": len(pattern.commands),
            "measurements": measurements,
        }
    )
    print(f"  n_gates={n_gates}: median={timing['median_seconds']:.5f}s, nodes={rows[-1]['resource_nodes']}", flush=True)

common.save_result(rows, "R37_compiler_runtime", extra={"environment": metadata.collect()})

time_exponent = fitted_exponent([r["n_gates"] for r in rows], [r["median_compile_seconds"] for r in rows])
node_exponent = fitted_exponent([r["n_gates"] for r in rows], [r["resource_nodes"] for r in rows])

# =============================================================================
# Final 1 x 2 publication figure
# =============================================================================

fig, (axA, axB) = plt.subplots(1, 2, figsize=(9.4, 4.0), facecolor="white")

axA.errorbar(
    [r["n_gates"] for r in rows], [r["median_compile_seconds"] for r in rows],
    yerr=[r["iqr_compile_seconds"] / 2 for r in rows], fmt="o-", color="red", capsize=3,
    label=f"$p$={time_exponent:.2f}",
)
axA.set_xscale("log")
axA.set_yscale("log")
style_axis(axA, "Number of logical gates", "Median compile time (s)", "Compile-time scaling", "A")
axA.legend(frameon=False, fontsize=8)

axB.plot([r["n_gates"] for r in rows], [r["resource_nodes"] for r in rows], "o-", color="blue",
         label=f"Resource nodes ($p$={node_exponent:.2f})")
axB.plot([r["n_gates"] for r in rows], [r["commands"] for r in rows], "s-", color="magenta", label="Commands")
axB.set_xscale("log")
axB.set_yscale("log")
style_axis(axB, "Number of logical gates", "Count", "Resource-count consistency with R10", "B")
axB.legend(frameon=False, fontsize=7)

fig.suptitle("Compiler runtime scaling (this machine only)", fontsize=10.5, fontweight="bold", y=1.01)
fig.tight_layout(rect=[0, 0, 1, 0.92], w_pad=2.4)
common.save_figure(fig, "R37_compiler_runtime")
plt.close(fig)

common.print_summary("R37 compiler runtime scaling", gate_counts=GATE_COUNTS,
                      fitted_time_exponent=time_exponent, fitted_node_exponent=node_exponent)


## R38 — PhotoGraphiQ Overhead vs. Raw Piquasso (Fig. 8)

R38 quantifies the abstraction cost of PhotoGraphiQ's graph/pattern/MBQC
layer over the underlying Piquasso numerical engine. Raw Piquasso executes a
matched \(N\)-mode Gaussian workload (per-mode squeeze+rotate, plus a CZ
chain) as **one** program with **one** `GaussianSimulator` call — the most
efficient native-engine usage; PhotoGraphiQ's `piquasso` backend instead
executes one native Program/Simulator call *per gate*
(`backends/piquasso.py`). The overhead ratio

$$
\text{ratio}(N) = \frac{T_{\text{PhotoGraphiQ}}(N)}{T_{\text{raw Piquasso}}(N)}
$$

quantifies exactly this architectural cost, on this machine — not a general
claim about Piquasso's or PhotoGraphiQ's performance elsewhere
(`MANUSCRIPT_MAP.md`).

### What is checked

Median runtime of both execution paths (warm-up + repeated timing) at mode
counts \(\{5,10,20,40\}\), and the resulting overhead ratio.

### Success criteria

- Both paths' runtimes and their ratio are reported as direct machine
  observations (panels A, B) — never presented as a portable benchmark
  result.
- The overhead ratio's trend with \(N\) shows whether the per-gate execution
  model becomes relatively more or less costly as the workload grows, which
  is the architecturally meaningful question here (not the absolute ratio at
  any one \(N\)).


In [ ]:
from __future__ import annotations

import numpy as np
import piquasso as pq

import photographiq as pg

set_reference_style()

MODE_COUNTS = [5, 10, 20, 40]
HBAR = 2.0


def raw_piquasso_run(n_modes: int):
    with pq.Program() as program:
        pq.Q(*range(n_modes)) | pq.Vacuum()
        for i in range(n_modes):
            pq.Q(i) | pq.Squeezing(r=0.1 + 0.01 * i)
            pq.Q(i) | pq.Phaseshifter(phi=0.2 + 0.01 * i)
        for i in range(n_modes - 1):
            pq.Q(i, i + 1) | pq.ControlledZ(s=0.3)
    return pq.GaussianSimulator(d=n_modes, config=pq.Config(hbar=HBAR)).execute(program).state


def build_pattern(n_modes: int) -> pg.Pattern:
    pattern = pg.Pattern(inputs=tuple(range(n_modes)))
    for i in range(n_modes):
        pattern.append(pg.Squeeze(i, 0.1 + 0.01 * i))
        pattern.append(pg.Rotate(i, 0.2 + 0.01 * i))
    for i in range(n_modes - 1):
        pattern.append(pg.Entangle(i, i + 1, 0.3))
    pattern.append(pg.Output(tuple(range(n_modes))))
    return pattern.validate()


rows = []
for n_modes in MODE_COUNTS:
    pattern = build_pattern(n_modes)
    warmup, repeats = (2, 9) if n_modes <= 10 else (1, 5)

    raw_timing = common.benchmark(lambda n=n_modes: raw_piquasso_run(n), warmup=warmup, repeats=repeats)
    pg_timing = common.benchmark(
        lambda pattern=pattern: pg.simulate(pattern, backend="piquasso", seed=0), warmup=warmup, repeats=repeats
    )

    rows.append(
        {
            "modes": n_modes,
            "gates": 3 * n_modes - 1,
            "raw_piquasso_median_seconds": raw_timing["median_seconds"],
            "raw_piquasso_iqr_seconds": raw_timing["iqr_seconds"],
            "photographiq_median_seconds": pg_timing["median_seconds"],
            "photographiq_iqr_seconds": pg_timing["iqr_seconds"],
            "overhead_ratio": pg_timing["median_seconds"] / raw_timing["median_seconds"],
        }
    )
    print(f"  modes={n_modes}: raw={raw_timing['median_seconds']:.5f}s pg={pg_timing['median_seconds']:.5f}s ratio={rows[-1]['overhead_ratio']:.2f}", flush=True)

common.save_result(rows, "R38_piquasso_overhead", extra={"environment": metadata.collect()})

# =============================================================================
# Final 1 x 2 publication figure
# =============================================================================

modes = [r["modes"] for r in rows]

fig, (axA, axB) = plt.subplots(1, 2, figsize=(9.6, 4.0), facecolor="white")

axA.loglog(modes, [r["raw_piquasso_median_seconds"] for r in rows], "o-", color="red", label="Raw Piquasso (1 program)")
axA.loglog(modes, [r["photographiq_median_seconds"] for r in rows], "s-", color="blue", label="PhotoGraphiQ (piquasso backend)")
style_axis(axA, "Modes", "Median runtime (s)", "Runtime by execution path", "A")
axA.legend(frameon=False, fontsize=7)

axB.plot(modes, [r["overhead_ratio"] for r in rows], "o-", color="magenta")
style_axis(axB, "Modes", r"Overhead ratio $T_{\rm PG}/T_{\rm PQ}$", "Abstraction-layer cost", "B")

fig.suptitle("PhotoGraphiQ overhead over raw Piquasso (this machine only)",
             fontsize=10.5, fontweight="bold", y=1.01)
fig.tight_layout(rect=[0, 0, 1, 0.92], w_pad=2.4)
common.save_figure(fig, "R38_piquasso_overhead")
plt.close(fig)

common.print_summary("R38 PhotoGraphiQ overhead vs. raw Piquasso", mode_counts=MODE_COUNTS)


## R39 — Serialization Round-Trip Invariant

R39 is a software-correctness experiment: it checks that PhotoGraphiQ's
versioned JSON pattern format is a **faithful, lossless** encoding of a
pattern, including its labels, symbolic parameter/outcome expressions, and
execution behavior — not merely its gross structure.

### The round-trip invariant

For encode/decode functions \(E=\)`Pattern.to_json`,
\(D=\)`Pattern.from_json`, the correctness property to check is the standard
round-trip identity

$$
D(E(P)) = P,
$$

interpreted at three levels of strictness, since Python equality alone would
not catch a semantically-losslessness bug hidden behind object identity:

1. **Structural equality**: inputs, outputs, commands, and declared
   parameters compare equal directly.
2. **Dependency equality**: the causal dependency DAG (R12) has the same
   edge set before and after the round trip — labels and expressions
   (`Outcome`, `Parameter`) must be restored, not merely their string
   representations.
3. **Behavioral equality**: executing the original and restored patterns with
   the *same* seed and inputs gives bit-identical classical outcomes and
   \(\|\mu_{\text{orig}}-\mu_{\text{restored}}\|<10^{-12}\),
   \(\|V_{\text{orig}}-V_{\text{restored}}\|<10^{-12}\) — the strongest test,
   since it exercises the restored `Outcome`/`Parameter` expressions inside
   an actual simulation, not just their JSON representation.

### Test pattern

A three-node, labelled (`"alice"`, `"bob"`, `"carol"`) adaptive pattern with
an outcome-dependent measurement angle on `"bob"` (the same genuine-adaptivity
construction as R11) and a two-outcome-dependent correction on `"carol"` —
chosen specifically to exercise every serialization-sensitive feature
(labels, `Parameter`, multi-term `Outcome` expressions) in one pattern.

### Success criteria

- All three levels of equality hold exactly (panel: compliance grid) — a
  discrete, exact software check with no numerical tolerance except the
  \(10^{-12}\) floor used only for the simulated floating-point state
  comparison.


In [ ]:
from __future__ import annotations

import numpy as np

import photographiq as pg

set_reference_style()


def build_pattern():
    adjacency = [[0, 1, 0], [1, 0, 1], [0, 1, 0]]
    graph = pg.CVGraph.from_adjacency(adjacency, labels=("alice", "bob", "carol"), squeezing=1.0, inputs=("alice",))
    pattern = pg.Pattern(graph)
    pattern.measure("alice", pg.Homodyne.q(), key="m0")
    pattern.displace("bob", q=-pg.Outcome("m0"))
    pattern.measure("bob", pg.Homodyne(angle=0.1 * pg.Outcome("m0") + pg.Parameter("theta")), key="m1")
    pattern.displace("carol", q=-pg.Outcome("m1"), p=pg.Outcome("m0") + pg.Outcome("m1"))
    pattern.append(pg.Output(("carol",)))
    return pattern.validate()


pattern = build_pattern()
text = pattern.to_json()
restored = pg.Pattern.from_json(text)

checks = {
    "inputs_preserved": restored.inputs == pattern.inputs,
    "outputs_preserved": restored.outputs == pattern.outputs,
    "commands_structurally_equal": restored.commands == pattern.commands,
    "parameters_preserved": restored.parameters == pattern.parameters,
    "dependency_graph_edges_preserved": sorted(restored.dependencies().edges()) == sorted(pattern.dependencies().edges()),
}

original_result = pg.simulate(
    pattern, backend="gaussian", inputs={"alice": pg.GaussianInput.coherent(0.3 + 0.1j)},
    parameters={"theta": 0.4}, seed=7,
)
restored_result = pg.simulate(
    restored, backend="gaussian", inputs={"alice": pg.GaussianInput.coherent(0.3 + 0.1j)},
    parameters={"theta": 0.4}, seed=7,
)
checks["identical_seeded_outcomes"] = original_result.outcomes == restored_result.outcomes
mean_error = common.frobenius_error(original_result.state.mean, restored_result.state.mean)
covariance_error = common.frobenius_error(original_result.state.covariance, restored_result.state.covariance)
checks["identical_seeded_output_mean"] = mean_error < 1e-12
checks["identical_seeded_output_covariance"] = covariance_error < 1e-12

failures = [name for name, ok in checks.items() if not ok]
if failures:
    common.save_json({"failures": failures}, "R39_serialization_FAILURES")
    raise AssertionError(f"Serialization round-trip checks failed: {failures}")

common.save_json(checks, "R39_serialization")
common.write_metadata("R39_serialization")
common.save_raw(text, "R39_serialization_pattern.json")

# =============================================================================
# Final figure: round-trip compliance grid (software-correctness result)
# =============================================================================

fig, ax = plt.subplots(figsize=(6.6, 4.0), facecolor="white")
labels = list(checks)
y = np.arange(len(labels))[::-1]
for yi, label in zip(y, labels):
    ax.barh(yi, 1.0, color="blue" if checks[label] else "red", edgecolor="white", height=0.7)
    ax.text(1.02, yi, "PASS" if checks[label] else "FAIL", va="center", fontsize=8)
ax.set_yticks(y)
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlim(0, 1)
ax.set_xticks([])
style_axis(ax, "", "", f"D(E(P))=P round-trip checks (mean err={mean_error:.1e}, cov err={covariance_error:.1e})", "A")
for spine in ("top", "right", "bottom"):
    ax.spines[spine].set_visible(False)

fig.suptitle("R39: serialization round-trip invariant", fontsize=10.5, fontweight="bold", y=1.0)
fig.tight_layout(rect=[0, 0, 1, 0.93])
common.save_figure(fig, "R39_serialization")
plt.close(fig)

common.print_summary("R39 serialization round-trip", **checks)


## R40 — Seed Reproducibility and Statistical Consistency of Shots

R40 checks two properties every seeded stochastic simulator must have to be
scientifically trustworthy: **exact** reproducibility for a fixed seed, and
**statistical indistinguishability** (not exact equality) of the sampled
distribution across different seeds.

### What is checked

For a single-mode coherent-state homodyne measurement (\(M=2000\) shots
each): (1) two runs with seed 42 give bit-identical raw outcome arrays;
(2) a run with seed 43 gives a **different** raw sequence but is not
statistically distinguishable from seed 42's distribution, via a two-sample
Kolmogorov–Smirnov test,

$$
D_{\mathrm{KS}} = \sup_x \big|\hat F_{42}(x)-\hat F_{43}(x)\big|,
$$

accepted at \(p>0.01\); (3) both seeds' sample means agree with the
theoretical coherent-state \(q\)-mean \(2\operatorname{Re}(\alpha)\) within
statistical tolerance.

### Success criteria

- Same-seed runs are bit-identical; different-seed runs are not (panel B:
  KS statistic and both mean errors, both well inside their tolerances).
- The KS test does not reject the null hypothesis that both seeds draw from
  the same underlying distribution (\(p>0.01\)) — different seeds are
  different random draws of the *same physics*, not different physics.
- Both empirical distributions visually overlap and center on the
  theoretical mean (panel A).


In [ ]:
from __future__ import annotations

import numpy as np
from scipy import stats

import photographiq as pg

set_reference_style()

SHOTS = 2000
ALPHA = 0.3 + 0.2j


def build_pattern():
    return pg.Pattern(inputs=(0,)).append(pg.Measure(0, pg.Homodyne.q(), key="m"))


pattern = build_pattern()
inputs = {0: pg.GaussianInput.coherent(ALPHA)}

run_a = pg.run_shots(pattern, SHOTS, backend="gaussian", inputs=inputs, seed=42)
run_b = pg.run_shots(pattern, SHOTS, backend="gaussian", inputs=inputs, seed=42)
run_c = pg.run_shots(pattern, SHOTS, backend="gaussian", inputs=inputs, seed=43)

samples_a, samples_b, samples_c = run_a.values("m"), run_b.values("m"), run_c.values("m")

identical_same_seed = bool(np.array_equal(samples_a, samples_b))
identical_cross_seed = bool(np.array_equal(samples_a, samples_c))

ks_same_distribution = stats.ks_2samp(samples_a, samples_c)
ks_self_consistency = stats.ks_2samp(samples_a, samples_b)  # trivially identical arrays -> statistic 0

mean_a, mean_c = float(np.mean(samples_a)), float(np.mean(samples_c))
theoretical_mean = 2 * ALPHA.real  # q-quadrature mean of a coherent state

summary = {
    "shots": SHOTS,
    "identical_outcomes_same_seed": identical_same_seed,
    "identical_outcomes_different_seeds": identical_cross_seed,
    "ks_statistic_seed42_vs_seed43": float(ks_same_distribution.statistic),
    "ks_pvalue_seed42_vs_seed43": float(ks_same_distribution.pvalue),
    "ks_statistic_seed42_vs_itself": float(ks_self_consistency.statistic),
    "sample_mean_seed42": mean_a,
    "sample_mean_seed43": mean_c,
    "theoretical_mean": theoretical_mean,
    "mean_error_seed42": abs(mean_a - theoretical_mean),
    "mean_error_seed43": abs(mean_c - theoretical_mean),
}

checks = {
    "same_seed_is_bit_identical": identical_same_seed,
    "different_seeds_are_not_identical": not identical_cross_seed,
    "different_seeds_pass_ks_same_distribution": ks_same_distribution.pvalue > 0.01,
    "sample_means_close_to_theory": summary["mean_error_seed42"] < 0.1 and summary["mean_error_seed43"] < 0.1,
}
failures = [name for name, ok in checks.items() if not ok]
if failures:
    common.save_json({"summary": summary, "failures": failures}, "R40_seeded_shots_FAILURES")
    raise AssertionError(f"Seed reproducibility checks failed: {failures}")

common.save_json({**summary, "checks": checks}, "R40_seeded_shots")
common.write_metadata("R40_seeded_shots")

# =============================================================================
# Final 1 x 2 publication figure
# =============================================================================

fig, (axA, axB) = plt.subplots(1, 2, figsize=(10.0, 4.2), facecolor="white")

axA.hist(samples_a, bins=40, alpha=0.6, label="seed=42", color="red", density=True)
axA.hist(samples_c, bins=40, alpha=0.6, label="seed=43", color="blue", density=True)
axA.axvline(theoretical_mean, color="black", linestyle="--", label="Theoretical mean")
style_axis(axA, "Homodyne outcome $q$", "Density", f"Seeded shot distributions (n={SHOTS} each)", "A")
axA.legend(frameon=False, fontsize=7)

metrics = ["KS statistic\n(42 vs 43)", "Mean error\n(seed 42)", "Mean error\n(seed 43)"]
values = [summary["ks_statistic_seed42_vs_seed43"], summary["mean_error_seed42"], summary["mean_error_seed43"]]
tolerances = [None, 0.1, 0.1]
axB.bar(metrics, values, color=["magenta", "red", "blue"])
for i, tol in enumerate(tolerances):
    if tol is not None:
        axB.hlines(tol, i - 0.4, i + 0.4, color="black", linestyle="--")
axB.text(0, values[0], f"$p$={summary['ks_pvalue_seed42_vs_seed43']:.3f}", ha="center", va="bottom", fontsize=7)
style_axis(axB, "", "Value", "Statistical consistency checks", "B")

fig.suptitle("Seed reproducibility and cross-seed statistical consistency", fontsize=10.5,
             fontweight="bold", y=1.02)
fig.tight_layout(rect=[0, 0, 1, 0.92], w_pad=2.4)
common.save_figure(fig, "R40_seeded_shots")
plt.close(fig)

common.print_summary("R40 seed reproducibility / shots", **checks)


## R41 — JAX Autodiff vs. Central Finite Differences (Experimental)

R41 checks the differentiable Fock generator behind `photographiq.autodiff`
for **internal consistency**: does `jax.grad` of `expectation(...)` agree
with a central finite-difference estimate of the *same* function? This is
not a comparison to an external analytic formula — it checks that two
different ways of differentiating the same numerical pipeline (automatic
differentiation through the Fock simulation vs. finite differencing its
output) agree, per the package's own documented validation approach
(`docs/validation/feature-matrix.md`).

### Central finite differences

For a scalar function \(f(t)\) (an expectation value as a function of one
gate parameter), the central-difference estimate is

$$
f'(t_0) \approx \frac{f(t_0+h)-f(t_0-h)}{2h} = f'(t_0) + O(h^2)
\;-\;\text{floating-point cancellation for }h\text{ too small}.
$$

There is a genuine trade-off: truncation error shrinks as \(h\to0\), but
floating-point cancellation grows — so the finite-difference estimate is
expected to match `jax.grad` best at some intermediate \(h\), not
monotonically improve as \(h\to0\). The validation therefore requires the
**best** tested step size to match, not every step size.

### Gates tested

Six gate families, each with one differentiable `Parameter`: rotation,
displacement, squeezing, cubic phase, Kerr, and loss — spanning Gaussian,
cubic non-Gaussian, and non-unitary (loss) generators, at Fock cutoff 12.

### Success criteria

- For every gate, at least one tested step size gives relative error
  \(<10^{-5}\) between `jax.grad` and the finite-difference estimate (panel
  A) — internal consistency of the autodiff pipeline.
- The autodiff gradient value itself and the best-matching finite-difference
  value agree closely across all six gate families (panel B), confirming
  panel A's error trend reflects real numerical agreement, not a
  coincidental cancellation.


In [ ]:
from __future__ import annotations

set_reference_style()

if not common.HAS_JAX:
    print("SKIPPED: jax is not installed; see requirements_notes.md")
else:
    import jax
    jax.config.update("jax_enable_x64", True)
    import jax.numpy as jnp
    import numpy as np

    import photographiq as pg
    from photographiq.autodiff import expectation

    CUTOFF = 12
    ALPHA = 0.3 + 0.2j
    T0 = 0.4
    STEP_SIZES = [1e-2, 1e-3, 1e-4, 1e-5]
    TOL = 1e-5

    GATES = {
        "rotation": (lambda t: pg.Rotate(0, t), "q"),
        "displacement": (lambda t: pg.Displace(0, q=t), "q"),
        "squeezing": (lambda t: pg.Squeeze(0, t), "photon_number"),
        "cubic_phase": (lambda t: pg.CubicPhase(0, t), "p"),
        "kerr": (lambda t: pg.Kerr(0, t), "q"),
        "loss": (lambda t: pg.Loss(0, t, 0.0), "photon_number"),
    }

    inputs = {0: pg.GaussianInput.coherent(ALPHA)}
    rows = []

    for name, (gate_builder, observable) in GATES.items():
        gate = gate_builder(pg.Parameter("t"))
        pattern = pg.Pattern(inputs=(0,)).append(gate).append(pg.Output((0,)))
        # Keep the loss transmissivity in (0,1]: evaluate around t0 scaled down.
        t0 = T0 if name != "loss" else 0.5

        def f(t, pattern=pattern, observable=observable):
            return expectation(pattern, {"t": t}, cutoff=CUTOFF, inputs=inputs, observable=observable)

        grad_value = float(jax.grad(f)(jnp.asarray(t0)))
        for h in STEP_SIZES:
            fd_value = float((f(t0 + h) - f(t0 - h)) / (2 * h))
            relative_error = abs(grad_value - fd_value) / max(abs(grad_value), 1e-12)
            rows.append(
                {
                    "gate": name,
                    "observable": observable,
                    "t0": t0,
                    "step_size": h,
                    "jax_gradient": grad_value,
                    "finite_difference_gradient": fd_value,
                    "absolute_error": abs(grad_value - fd_value),
                    "relative_error": relative_error,
                }
            )
        print(f"  {name}: jax_grad={grad_value:.8f}", flush=True)

    # The finite-difference approximation should agree with autodiff to a few
    # times h^2 (central-difference truncation error) for a well-behaved h;
    # too-small h instead suffers floating-point cancellation. Require the
    # BEST of the tested step sizes to be accurate, not every one of them.
    best_by_gate = {}
    best_row_by_gate = {}
    for row in rows:
        if row["relative_error"] < best_by_gate.get(row["gate"], float("inf")):
            best_by_gate[row["gate"]] = row["relative_error"]
            best_row_by_gate[row["gate"]] = row
    failures = [gate for gate, err in best_by_gate.items() if err > TOL]
    if failures:
        common.save_csv([r for r in rows if r["gate"] in failures], "R41_jax_gradients_FAILURES")
        raise AssertionError(f"No tested step size matched autodiff for gates: {failures}; see FAILURES csv")

    common.save_result(rows, "R41_jax_gradients", extra={"cutoff": CUTOFF, "alpha": [ALPHA.real, ALPHA.imag]})

    # =========================================================================
    # Final 1 x 2 publication figure
    # =========================================================================

    fig, (axA, axB) = plt.subplots(1, 2, figsize=(10.4, 4.4), facecolor="white")

    colors = {"rotation": "red", "displacement": "blue", "squeezing": "magenta",
              "cubic_phase": "cyan", "kerr": "orange", "loss": "green"}
    for name in GATES:
        subset = [r for r in rows if r["gate"] == name]
        axA.loglog([r["step_size"] for r in subset], [max(1e-16, r["relative_error"]) for r in subset],
                    "o-", color=colors[name], label=name)
    axA.axhline(TOL, color="black", linestyle="--", linewidth=0.8, label="Tolerance")
    style_axis(axA, "Finite-difference step size $h$", "Relative error vs. JAX autodiff",
               "Autodiff/finite-difference consistency", "A")
    axA.legend(frameon=False, fontsize=6.5)

    names = list(GATES)
    jax_vals = [best_row_by_gate[n]["jax_gradient"] for n in names]
    fd_vals = [best_row_by_gate[n]["finite_difference_gradient"] for n in names]
    axB.scatter(jax_vals, fd_vals, c=[colors[n] for n in names], s=60, edgecolors="black", linewidths=0.5)
    for n, x, y in zip(names, jax_vals, fd_vals, strict=True):
        axB.annotate(n, (x, y), xytext=(4, 4), textcoords="offset points", fontsize=6.5)
    lims = [min(jax_vals + fd_vals) * 1.15, max(jax_vals + fd_vals) * 1.15 + 1e-9]
    axB.plot(lims, lims, "--", color="black", linewidth=0.8, label="Equality")
    style_axis(axB, "JAX gradient", "Best finite-difference gradient",
               "Gradient values agree, not just their trend", "B")
    axB.legend(frameon=False, fontsize=7)

    fig.suptitle("JAX autodiff vs. central finite differences (experimental)",
                 fontsize=10.5, fontweight="bold", y=1.01)
    fig.tight_layout(rect=[0, 0, 1, 0.93], w_pad=2.6)
    common.save_figure(fig, "R41_jax_gradients")
    plt.close(fig)

    common.print_summary("R41 JAX deterministic gradient validation", gates=list(GATES), best_relative_errors=best_by_gate)


## R42 — Stochastic Gradient Estimators: Unbiasedness Check (Experimental)

R42 checks the two generic Monte Carlo gradient-estimator primitives in
`photographiq.autodiff` against exact analytic gradients on toy problems
chosen specifically so a closed form exists — this is explicitly
experimental functionality (`docs/validation/feature-matrix.md`), and
agreement here is necessarily **statistical** (bias vs. standard error over
repeated batches), never exact-precision.

### Two estimator families

1. **Score-function / REINFORCE** (`score_function_surrogate`): for
   \(X\sim\mathrm{Bernoulli}(\sigma(\theta))\) and a fixed payoff
   \(f(X)\in\{f_0,f_1\}\), the exact gradient of \(\mathbb E[f(X)]\) is

   $$
   \frac{d}{d\theta}\mathbb E[f(X)] = (f_1-f_0)\,\sigma(\theta)\big(1-\sigma(\theta)\big),
   $$

   compared against the surrogate-loss gradient
   \(\nabla_\theta\,\mathbb E[f(X)\log p_\theta(X)]\) estimated from
   \(N=20000\) samples, repeated over 10 independent batches.

2. **Pathwise / reparameterization** (`gaussian_sample`): for
   \(Z\sim\mathcal N(0,1)\), \(X=\theta+\sigma Z\), and
   \(f(X)=X^2\), the exact gradient of \(\mathbb E[f(X)]\) is
   \(2\theta\) (since \(\mathbb E[Z]=0\)), compared against
   differentiating the reparameterized sample through `jax.grad`.

### What "correct" means for a Monte Carlo estimator

An unbiased estimator's sample mean over repeated batches should be within a
few standard errors of the true value — **not** exactly equal to it. The
check here is \(|\text{bias}|\le5\times\text{SE}+10^{-6}\), a standard
statistical consistency criterion, explicitly different in kind from the
machine-precision tolerances used for exact algebraic identities elsewhere in
this suite (R1–R32).

### Success criteria

- Both estimators' bias falls within the 5-sigma statistical band (panel A:
  estimate ± 5 SE bracketing the exact gradient; panel B: bias measured in
  units of its own standard error, both well under 5).


In [ ]:
from __future__ import annotations

set_reference_style()

if not common.HAS_JAX:
    print("SKIPPED: jax is not installed; see requirements_notes.md")
else:
    import jax
    jax.config.update("jax_enable_x64", True)
    import jax.numpy as jnp
    import numpy as np

    from photographiq.autodiff import gaussian_sample, score_function_surrogate

    N_SAMPLES = 20000
    N_REPEATS = 10
    THETA0 = 0.3
    F_ONE, F_ZERO = 2.0, -1.0  # values attached to the two Bernoulli branches

    def score_function_case():
        def analytic_gradient(theta):
            p = jax.nn.sigmoid(theta)
            return (F_ONE - F_ZERO) * p * (1 - p)

        exact_gradient = float(analytic_gradient(THETA0))

        estimates = []
        for repeat in range(N_REPEATS):
            rng = np.random.default_rng(1000 + repeat)
            p0 = float(jax.nn.sigmoid(THETA0))
            outcomes = rng.random(N_SAMPLES) < p0  # samples from Bernoulli(p(theta0))
            values = jnp.where(jnp.asarray(outcomes), F_ONE, F_ZERO)

            def loss(theta, outcomes=outcomes, values=values):
                p = jax.nn.sigmoid(theta)
                log_probs = jnp.where(jnp.asarray(outcomes), jnp.log(p), jnp.log1p(-p))
                return score_function_surrogate(values, log_probs)

            estimates.append(float(jax.grad(loss)(jnp.asarray(THETA0))))

        estimates = np.array(estimates)
        return {
            "estimator": "score_function_surrogate",
            "exact_gradient": exact_gradient,
            "mean_estimate": float(estimates.mean()),
            "bias": float(estimates.mean() - exact_gradient),
            "standard_error": float(estimates.std(ddof=1) / np.sqrt(N_REPEATS)),
            "n_samples_per_repeat": N_SAMPLES,
            "n_repeats": N_REPEATS,
        }

    def pathwise_case():
        sigma = 0.7

        def analytic_gradient(theta):
            return 2.0 * theta  # d/dtheta E_z[(theta+sigma*z)^2] = 2*theta since E[z]=0

        exact_gradient = float(analytic_gradient(THETA0))

        estimates = []
        for repeat in range(N_REPEATS):
            rng = np.random.default_rng(2000 + repeat)
            z_batch = rng.standard_normal((N_SAMPLES, 1))

            def loss(theta, z_batch=z_batch):
                mean = jnp.array([theta])
                covariance = jnp.array([[sigma**2]])
                samples = jax.vmap(lambda z: gaussian_sample(mean, covariance, z))(jnp.asarray(z_batch))
                return jnp.mean(samples[:, 0] ** 2)

            estimates.append(float(jax.grad(loss)(THETA0)))

        estimates = np.array(estimates)
        return {
            "estimator": "gaussian_sample (pathwise)",
            "exact_gradient": exact_gradient,
            "mean_estimate": float(estimates.mean()),
            "bias": float(estimates.mean() - exact_gradient),
            "standard_error": float(estimates.std(ddof=1) / np.sqrt(N_REPEATS)),
            "n_samples_per_repeat": N_SAMPLES,
            "n_repeats": N_REPEATS,
        }

    results = [score_function_case(), pathwise_case()]

    # A statistically consistent unbiased estimator should have |bias| within
    # a few standard errors of zero; this is not an exact-precision check.
    failures = [r for r in results if abs(r["bias"]) > 5 * r["standard_error"] + 1e-6]
    if failures:
        common.save_json(failures, "R42_stochastic_gradients_FAILURES")
        raise AssertionError("An estimator's bias exceeded its statistical tolerance; see FAILURES json")

    common.save_json({"results": results}, "R42_stochastic_gradients")
    common.write_metadata("R42_stochastic_gradients")

    # =========================================================================
    # Final 1 x 2 publication figure
    # =========================================================================

    fig, (axA, axB) = plt.subplots(1, 2, figsize=(9.4, 4.2), facecolor="white")

    names = [r["estimator"] for r in results]
    axA.errorbar(
        range(len(results)), [r["mean_estimate"] for r in results],
        yerr=[5 * r["standard_error"] for r in results], fmt="o", capsize=4, color="red", label="Estimate $\\pm$5 SE",
    )
    axA.plot(range(len(results)), [r["exact_gradient"] for r in results], "x", markersize=10, color="blue",
             label="Exact gradient")
    axA.set_xticks(range(len(results)))
    axA.set_xticklabels(names, rotation=15, ha="right", fontsize=8)
    style_axis(axA, "", "Gradient estimate", "Stochastic gradient estimators", "A")
    axA.legend(frameon=False, fontsize=7)

    bias_in_sigma = [r["bias"] / r["standard_error"] for r in results]
    axB.bar(range(len(results)), bias_in_sigma, color="magenta")
    axB.axhline(5, color="black", linestyle="--", linewidth=0.8, label=r"$\pm5\sigma$ tolerance")
    axB.axhline(-5, color="black", linestyle="--", linewidth=0.8)
    axB.set_xticks(range(len(results)))
    axB.set_xticklabels(names, rotation=15, ha="right", fontsize=8)
    style_axis(axB, "", "Bias / standard error", "Unbiasedness in sampling units", "B")
    axB.legend(frameon=False, fontsize=7)

    fig.suptitle("Stochastic gradient estimators: unbiasedness (experimental)", fontsize=10.5,
                 fontweight="bold", y=1.02)
    fig.tight_layout(rect=[0, 0, 1, 0.92], w_pad=2.6)
    common.save_figure(fig, "R42_stochastic_gradients")
    plt.close(fig)

    common.print_summary("R42 conditional / estimator gradients", results=results)


## Aggregate manuscript tables (`generate_tables.py`)

In [45]:
import importlib
import generate_tables
importlib.reload(generate_tables)
generate_tables.main()


Generating manuscript summary tables...
  wrote compiler_accuracy.csv / compiler_accuracy.md (5 rows)
  wrote backend_agreement.csv / backend_agreement.md (2 rows)
  wrote piquasso_validation.csv / piquasso_validation.md (12 rows)
  wrote graphix_structure.csv / graphix_structure.md (4 rows)


  wrote non_gaussian_validation.csv / non_gaussian_validation.md (4 rows)
  wrote convergence_summary.csv / convergence_summary.md (5 rows)
  wrote performance_summary.csv / performance_summary.md (15 rows)
  wrote feature_status.csv / feature_status.md (12 rows)
Done. See D:\Research\QML\results_1\cvqc\thesis_part_2\softwares\PhotoGraphiQ\paper_experiments\tables
